# LSM-ECG thesis — config 11 vs continuous liquids with memory

Each MIT-BIH record is streamed through the liquid once, start to end; the readout takes config 11's 32 windows around each beat from that run. The question: does a liquid run on the continuous recording (with memory) classify SVEB better than config 11's per-beat liquid? The plan (Claude doc "LSM-ECG plan: config 11 vs continuous liquids with memory") has the rows, the rule and the checks.

| row | liquid | internal connections | memory | input |
|---|---|---|---|---|
| B1 | paper's 7×7×7 (config 11, three fixes) | on | none | per beat |
| B2 | no liquid | — | — | per beat |
| P1 | paper's | on | none | whole record |
| P2 | paper's | off | none | whole record |
| P3 | paper's | on | adaptation, Salaj spread (60%, 1 ms–6 s) | whole record |
| P3f | paper's | on | P3's thresholds frozen at their DS1 averages | whole record |
| P4 | paper's | off | adaptation, Salaj spread | whole record |
| P5 | paper's | on | adaptation, one 2 s clock | whole record |
| M0 | Maass 15×3×3 | on | dynamic synapses | per beat |
| M1 | Maass | on | dynamic synapses | whole record |
| M3 | Maass | on | dynamic synapses + adaptation | whole record |
| M3f | Maass | on | M3's thresholds frozen | whole record |

**MODE:** `SMOKE` (tiny records, every step, writes to `WORK_DIR_smoke`), `BENCHMARK` (every check, timings, projected cost; B1 and B2 are run in full here and kept), `FULL` (everything). Run SMOKE, then BENCHMARK, read `benchmark/benchmark.md`, then FULL. A restarted session continues where it stopped.

**Results:** `WORK_DIR/summary.md` (headline table), `rows.json` (one row per fit), `runs/<row>_s<seed>__svm/` (DS2 predictions), `states/` (32-window spike counts per beat), `liquids/`, `check.json`, `benchmark/`.

## 0. Constants (edit here)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CONSTANTS — the only cell you normally need to edit                 ║
# ╚══════════════════════════════════════════════════════════════════════╝
# Config 11 vs continuous liquids with memory. Results go to WORK_DIR (SMOKE: WORK_DIR + "_smoke").

# ---- Machine -----------------------------------------------------------
MACHINE = "local"                  # "kaggle" or "local" (your own Linux machine; set its paths below)

MACHINES = {
    "kaggle": {
        "INPUT_DIR":   "/kaggle/input/datasets/tienthinh22/mitdb-1-0-0/mit-bih-arrhythmia-database-1.0.0",
        "WORK_DIR":    "/kaggle/working",
        "SCRATCH_DIR": "/kaggle/working/scratch",
        "RESUME_DIRS": ["/kaggle/input/datasets/tienthinh22/mitdb-common"],
        "SESSION_BUDGET_HOURS": 11.3,
        "DISK_LIMIT_GB": 19.0,
    },
    "local": {
        "INPUT_DIR":   "~/lsm_ecg/mitdb-1.0.0/mit-bih-arrhythmia-database-1.0.0",
        "WORK_DIR":    "~/lsm_ecg/output_stream",   # a new folder; SMOKE writes to output_stream_smoke
        "SCRATCH_DIR": None,
        "RESUME_DIRS": [],
        "SESSION_BUDGET_HOURS": 240.0,
        "DISK_LIMIT_GB": 250.0,
    },
}
_M = MACHINES[MACHINE]
INPUT_DIR            = _M["INPUT_DIR"]
WORK_DIR             = _M["WORK_DIR"]
SCRATCH_DIR          = _M["SCRATCH_DIR"] or WORK_DIR.rstrip("/") + "/scratch"
RESUME_DIRS          = _M["RESUME_DIRS"]
AUTO_RESUME_ROOT     = "/kaggle/input"
SESSION_BUDGET_HOURS = _M["SESSION_BUDGET_HOURS"]
DISK_LIMIT_GB        = _M["DISK_LIMIT_GB"]

# ---- This notebook ------------------------------------------------------
# Config 11 against liquids that stream each whole MIT-BIH record (the plan: "LSM-ECG plan: config 11 vs continuous
# liquids with memory"). Twelve rows: B1 (config 11 with the three fixes, per beat), B2 (no liquid), P1-P5 and P3f
# (the paper's liquid, streaming), M0 (Maass's liquid per beat), M1, M3 and M3f (Maass's liquid, streaming).
MODE                 = "SMOKE"     # "SMOKE"     : every step end to end on records cut to SMOKE_SECONDS (fake or real
                                   #               data; the numbers mean nothing). Writes to WORK_DIR + "_smoke".
                                   # "BENCHMARK" : real data; every check (B1 and B2 on DS2 and the 12 CV splits
                                   #               included: real results, kept), the timings on BENCHMARK_SECONDS of
                                   #               every record, the projected FULL time and cost, the switches
                                   # "FULL"      : every row, its readouts and the summary
ROWS                 = ("B1", "B2", "P1", "P2", "P3", "P3f", "P4", "P5", "M0", "M1", "M3", "M3f")
STREAM_SEEDS         = (2026, 69, 420)   # liquid seeds (the replication's); SMOKE uses the first two
EXACT_CV             = True        # exact CV on the 12 DS1 splits for B1, B2, P1-P5 and P3f (SVEB F1 test)
STACK                = "auto"      # liquids share one run: True / False / "auto" (the benchmark's choice)
CHUNKS               = "auto"      # Maass's records in pieces with a warm-up: True / False / "auto"; the paper's rows
                                   # always run unchunked
CHUNK_PIECES         = 5           # pieces per record (6 min each)
CHUNK_WARMUP_S       = 60.0        # each piece starts this much early, from rest; the warm-up's windows are not kept
                                   # Stacking (both liquids): allowed if the stacked liquid's numbers equal each
                                   # liquid's own and a short run agrees to rounding. Maass's chunks: used only if
                                   # they give the unbroken run's spikes exactly.
DROP_SECONDS         = 30.0        # keep vs drop: every row also refit and tested without the beats in each
                                   # record's first 30 s (the liquid starts from rest; CPU only, DS2 only)
SEGMENT_STEPS        = 8192        # input steps per simulator call (state carried across); memory only, not results
CV_PENDING_FITS      = 45          # CV states wait on disk for their fits (about 0.56 GB each); the GPU pauses beyond
ADAPT_SALAJ          = dict(beta=1.7, b0=0.03, tau_ms=(1.0, 6000.0), fraction=0.6)   # Salaj et al. 2021,
                                   # duplication/reversal: 60% adaptive, tau_a uniform 1 ms - 6 s, 1.7 / 30 per Hz
ADAPT_ONE_CLOCK_MS   = 2000.0      # P5: every adaptive neuron at tau_a = 2 s, same strength
MAASS_INPUT_FRACTION = 0.3         # Maass 2002: input to 30% of the neurons
FIT_THREADS          = 3           # readout fits side by side, one core each; each needs about 2 GB of RAM
PRESCALE             = True        # scale each training matrix once (faster, more RAM); same results
REPLICATION_DIR      = "~/lsm_ecg/output"   # the replication's output folder (runs/ inside), only read: the
                                   # machinery check (config 11 seed 2026, DS2, the authors' reset)
T1_ROWS              = "~/lsm_ecg/output_t1_rrcheck/t1_rrcheck/rows.json"   # task 1's rows.json, only read: the B1 /
                                   # B2 confusion-matrix checks (skipped, with a note, if it is not there)
SMOKE_SECONDS        = 120         # SMOKE: records cut to their first 2 min, beats outside dropped
BENCHMARK_SECONDS    = 300         # BENCHMARK: 5 min of every record for the timings and equality checks
GPU_PRICE_PER_H      = 0.32        # for the projected cost
TASK                 = "stream"    # (log prefix of the shared task code)
VERIFY               = False       # (the task notebooks' CV check is replaced by this notebook's checks)
if MODE not in ("SMOKE", "BENCHMARK", "FULL"):
    raise ValueError(f"MODE = {MODE!r}")
if MODE == "SMOKE":                # SMOKE never mixes with real results
    WORK_DIR = (WORK_DIR.rstrip("/") + "/smoke") if MACHINE == "kaggle" else WORK_DIR.rstrip("/") + "_smoke"
    SCRATCH_DIR = _M["SCRATCH_DIR"] + "_smoke" if _M["SCRATCH_DIR"] else WORK_DIR.rstrip("/") + "/scratch"

# ---- Session control ---------------------------------------------------
SELF_TEST            = False
SMOKE_TEST           = MODE == "SMOKE"   # set by MODE above
BENCHMARK            = False
ANALYSIS_RESERVE_MIN = 25
RETRY_FAILED         = True
OLD_ANALYSIS         = False       # True = also rebuild the replication's tables and figures

# ---- Software ----------------------------------------------------------
PIN_TORCH_240  = False             # runs on the installed PyTorch
AUTHORS_REPO   = "https://github.com/Richard97Code/LSM-ECG-Classification.git"
AUTHORS_COMMIT = "02e45c4f3232886aed16a40d77de1f851ef4fcdd"

# ---- Experiment scope --------------------------------------------------
SEEDS = list(STREAM_SEEDS)              # read by the reused cells

OLD_GROUPS = ["checks", "brian2", "main_final", "ablation_final", "rr_final", "baseline_final", "gain_final",
              "main_cv", "gain_cv", "ablation_cv", "rr_cv", "baseline_cv", "nn_tune", "retune", "kstale_final",
              "kstale_cv", "huang_final", "huang_cv", "meanrr_final", "meanrr_cv", "elmrr_final", "elmrr_cv",
              "rrliqmean_final", "rrliqmean_cv", "meanrr_nn", "hybrid", "working_diag", "working_final", "working_cv"]
GROUP_ORDER = OLD_GROUPS                     # read by the reused cells only; this notebook runs no replication group
GROUPS_ENABLED = {g: False for g in GROUP_ORDER}
SHARE = "BOTH"; SHARE_GROUPS = {"A": [], "B": []}
MERGE_DIRS = []

# replication settings that its cells still read (unchanged values)
GAINS = [1, 2, 3, 5, 8, 13, 21, 34]; DENSE_LAMBDAS = [1.5, 2.5]; RETUNE_TRIALS = 30
RETUNE_GRID = {"gain": [1, 2, 3, 5, 8, 13], "lambda_val": [0.8, 1.5, 2.5], "num_levels": [50, 100, 200, 400]}
NN_EPOCHS = 15; BRIAN2_BEATS = 10; NN_TUNE_TRIALS = 30; NN_PARAM_CAP = 35000; NN_TUNE_MAX_EPOCHS = 30
NN_TUNE_CHECKPOINTS = [5, 10, 15, 20, 25, 30]
HUANG_MATRICES = 15; HUANG_DIM = 50; HUANG_C = 1.0; HUANG_GAMMAS = [1.3, 1 / (2 * 1.3 ** 2)]
HUANG_THRESHOLD = 0.8; HUANG_SVM_CACHE_MB = 500; HYBRID_RULES = ["replace", "add"]
WORKING_LAMBDA = 2.0; WORKING_GAINS = [1, 3, 10, 30, 100, 300, 1000]; WORKING_DIAG_BEATS = 300
WORKING_T_PERTURB = 150; WORKING_MAX_FIRING = 0.2
SAVE_STATES = []

# ---- Performance -------------------------------------------------------
SIM_BATCH_CV    = 4096
SIM_BATCH_FINAL = 2000
NEW_SIM_BATCH   = 2000     # beats per GPU batch for the liquid simulator (results do not depend on it; bigger = faster)
PIPELINE        = True
FIT_WORKERS     = "auto"
CORE_SEARCH     = False
RAM_RESERVE_GB  = 4.0
READOUT_JOBS    = 4


## 1. Environment

In [ ]:
NOTEBOOK_BUILD = "8e7fca84a966"   # fingerprint of this build's code cells (do not edit)
# Environment: installs, imports, versions, device. Smoke-test overrides.
import os, sys, subprocess, time, json, platform, shutil, glob, hashlib, traceback, math, re, gc, random
import importlib, importlib.util, datetime, itertools, warnings, contextlib, io, heapq, collections
warnings.filterwarnings("ignore")
SESSION_START = time.time()
ON_KAGGLE = os.path.isdir("/kaggle/working") and "KAGGLE_KERNEL_RUN_TYPE" in os.environ
NOTEBOOK_BUILD = globals().get("NOTEBOOK_BUILD", "dev")      # fingerprint of the code cells, set by the build
if SHARE is None:
    SHARE = "BOTH"
if SHARE != "BOTH" and SHARE not in SHARE_GROUPS:
    raise ValueError(f"SHARE must be 'BOTH', 'A' or 'B', not {SHARE!r}")
MERGE_MODE = bool(MERGE_DIRS)
if MERGE_MODE and SHARE != "BOTH":
    print(f"merge mode: SHARE = {SHARE!r} ignored (the merged result is checked against the whole plan)")
    SHARE = "BOTH"
if MACHINE == "kaggle" and not ON_KAGGLE:
    print("NOTE: MACHINE = 'kaggle' but this does not look like a Kaggle session (paths may not exist).")
INPUT_DIR, WORK_DIR, SCRATCH_DIR = (os.path.abspath(os.path.expanduser(p)) for p in (INPUT_DIR, WORK_DIR, SCRATCH_DIR))
if isinstance(RESUME_DIRS, (list, tuple)):
    RESUME_DIRS = [os.path.abspath(os.path.expanduser(p)) for p in RESUME_DIRS]
os.makedirs(os.path.join(WORK_DIR, "logs"), exist_ok=True)
import threading
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # deterministic cuBLAS
SESSION_TAG = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
LOG_PATH = os.path.join(WORK_DIR, "logs", f"session_{SESSION_TAG}.log")

class _Tee:
    """Everything the notebook prints (all cells, all threads, warnings and tracebacks) also goes to LOG_PATH,
    flushed line by line, so `tail -f WORK_DIR/logs/latest.log` shows it live from another terminal."""
    def __init__(self, stream, path):
        self._s, self._f, self._lock = stream, open(path, "a", buffering=1), threading.Lock()
    def write(self, text):
        with self._lock:
            try:
                self._f.write(text); self._f.flush()
            except Exception:
                pass
        return self._s.write(text)
    def flush(self):
        self._s.flush()
    def __getattr__(self, name):
        return getattr(self._s, name)

if getattr(sys.stdout, "_lsm_tee", False):           # notebook re-run in the same kernel: switch to the new log
    for _t in (sys.stdout, sys.stderr):
        _t._f = open(LOG_PATH, "a", buffering=1)
else:
    sys.stdout, sys.stderr = _Tee(sys.stdout, LOG_PATH), _Tee(sys.stderr, LOG_PATH)
    sys.stdout._lsm_tee = sys.stderr._lsm_tee = True
try:                                                    # a crash's traceback goes into the log file too
    def _exc_to_log(shell, etype, value, tb, tb_offset=None):
        with open(LOG_PATH, "a") as f:
            f.write("\n" + "".join(traceback.format_exception(etype, value, tb)))
        shell.showtraceback((etype, value, tb), tb_offset=tb_offset)
    get_ipython().set_custom_exc((Exception,), _exc_to_log)
except NameError:
    pass
_latest = os.path.join(WORK_DIR, "logs", "latest.log")
try:
    if os.path.islink(_latest) or os.path.exists(_latest):
        os.remove(_latest)
    os.symlink(os.path.basename(LOG_PATH), _latest)
except OSError:
    pass
print(f"session {SESSION_TAG}: everything printed is also written to {LOG_PATH} (and {_latest})")

N_CPU = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
if PIPELINE:
    # one fit per core: the maths libraries must not start their own thread pools inside each fit
    for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        os.environ[_v] = "1"

import importlib.metadata as _md

def _ver(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return None

def _pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-3000:]

# Kaggle's core packages must not change version mid-session (a NumPy swap after it is loaded breaks imports),
# so every install is constrained to the versions already present; pip then picks compatible package versions.
CORE = ("numpy", "scipy", "pandas", "matplotlib", "torch")
CORE_BEFORE = {p: _ver(p) for p in CORE}
INSTALL_REPORT = []
if not ON_KAGGLE:
    # Your own machine: you install the packages (in a virtual environment); here we only check.
    _need = {"numpy": None, "scipy": None, "pandas": None, "matplotlib": None, "torch": None, "snntorch": "0.9.4",
             "wfdb": None, "tqdm": None, "scikit-learn": "1.7.2", "joblib": None, "threadpoolctl": None}
    _miss = [p for p in _need if _ver(p) is None]
    if _miss:
        raise RuntimeError(f"missing packages {_miss}: pip install them in the virtual environment, activate it, then run the notebook")
    for p, want in _need.items():
        if want and _ver(p) != want:
            print(f"WARNING: {p} {_ver(p)} installed, {want} expected")
    INSTALL_REPORT.append({"package": "(local machine: installed by hand)", "status": "checked", "tail": ""})
    if _ver("brian2") is None:
        print("WARNING: Brian2 is not installed; the brian2 check will be marked FAILED, everything else runs")
if ON_KAGGLE:
    os.makedirs(SCRATCH_DIR, exist_ok=True)
    def _constraints(skip=()):
        path = os.path.join(SCRATCH_DIR, "pip_constraints.txt")
        with open(path, "w") as f:
            f.write("\n".join(f"{p}=={v}" for p, v in CORE_BEFORE.items() if v and p not in skip) + "\n")
        return path
    if PIN_TORCH_240 and sys.version_info[:2] <= (3, 12):
        code, out = _pip("torch==2.4.0", "-c", _constraints(skip=("torch",)))
        INSTALL_REPORT.append({"package": "torch==2.4.0", "status": "ok" if code == 0 else "FAILED", "tail": out})
    elif PIN_TORCH_240:
        INSTALL_REPORT.append({"package": "torch==2.4.0", "status": "skipped",
                               "tail": f"Python {platform.python_version()} has no torch 2.4.0; using Kaggle's torch {CORE_BEFORE['torch']}"})
    for pkg, need in [("snntorch==0.9.4", "required"), ("wfdb", "required"), ("tqdm", "required"),
                      ("scikit-learn==1.7.2", "preferred"), ("brian2", "optional")]:
        code, out = _pip(pkg, "-c", _constraints())
        INSTALL_REPORT.append({"package": pkg, "need": need, "status": "ok" if code == 0 else "FAILED", "tail": out})
    for r in INSTALL_REPORT:
        print(f"install {r['package']:<22} -> {r['status']}" + (f"  ({r['tail']})" if r["status"] == "skipped" else ""))
    bad = [r for r in INSTALL_REPORT if r["status"] == "FAILED" and r.get("need") == "required"]
    if bad:
        raise RuntimeError("Required install failed (is Internet ON in the notebook settings?):\n" + bad[0]["tail"])
    for r in INSTALL_REPORT:
        if r["status"] == "FAILED" and r.get("need") == "preferred":
            print(f"WARNING: {r['package']} could not be installed; using Kaggle's scikit-learn {_ver('scikit-learn')}")
        if r["status"] == "FAILED" and r.get("need") == "optional":
            print("WARNING: Brian2 could not be installed; the brian2 check will be marked FAILED, everything else runs")
    changed = {p: (CORE_BEFORE[p], _ver(p)) for p in CORE if _ver(p) != CORE_BEFORE[p]
               and not (p == "torch" and any(r["package"] == "torch==2.4.0" and r["status"] == "ok" for r in INSTALL_REPORT))}
    if changed:
        raise RuntimeError(f"pip changed core packages {changed}. Restart the session and run again.")

import numpy as np, pandas as pd, torch, snntorch, sklearn, scipy, wfdb, joblib
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedGroupKFold
from scipy import stats as sstats
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TORCH_PINNED = torch.__version__.split("+")[0] == "2.4.0"

def gpu_supported(cap, arch_list):
    """Can this PyTorch build run on a GPU of compute capability cap=(major, minor)? A GPU runs code built for its own
    major version and a lower or equal minor (sm_61 runs sm_60), or PTX (compute_XY) at or below its capability."""
    for a in arch_list:
        kind, _, num = a.partition("_")
        if not num.isdigit():
            continue
        maj, mnr = int(num[:-1]), int(num[-1])
        if kind == "sm" and maj == cap[0] and mnr <= cap[1]:
            return True
        if kind == "compute" and (maj, mnr) <= tuple(cap):
            return True
    return False

if DEVICE.type == "cuda":
    _cap = torch.cuda.get_device_capability(0)
    _gpu = torch.cuda.get_device_name(0)
    _archs = torch.cuda.get_arch_list()
    _hint = (f"Install a PyTorch build that includes it: on Python <= 3.12 'pip install torch==2.4.0' (the authors' version); "
             f"otherwise a CUDA 12.6 build, 'pip install \"torch<2.15\" --index-url https://download.pytorch.org/whl/cu126'. "
             f"Older GPUs (e.g. Pascal sm_6x) are not in the CUDA 12.8+ / 13.x builds.")
    if not gpu_supported(_cap, _archs):
        raise RuntimeError(f"PyTorch {torch.__version__} (CUDA {torch.version.cuda}) has no code for this GPU: {_gpu}, "
                           f"compute capability {_cap[0]}.{_cap[1]}; this build supports {_archs}. {_hint}")
    try:
        torch.ones(8, device=DEVICE).add_(1).sum().item()          # a real kernel launch, not just the device query
    except Exception as ex:
        raise RuntimeError(f"PyTorch cannot run on {_gpu} (capability {_cap[0]}.{_cap[1]}): {ex}. {_hint}")
    print(f"GPU: {_gpu}, compute capability {_cap[0]}.{_cap[1]}: supported by PyTorch {torch.__version__} (CUDA {torch.version.cuda})")
# Plain float32 everywhere: Ampere+ GPUs (e.g. RTX 3090) would otherwise run convolutions in TF32 (a T4 cannot)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)   # PyTorch's deterministic algorithms on
if PIPELINE:
    from threadpoolctl import threadpool_limits
    threadpool_limits(1)                 # same as the environment variables above, in case NumPy was already loaded
    torch.set_num_threads(max(1, N_CPU // 4) if DEVICE.type == "cpu" else 1)
import threading
READOUT_EPOCHS = 15            # authors' value (part of the replication, not a tuning knob)

def _ram_gb():
    try:
        with open("/proc/meminfo") as f:
            return round(int(f.readline().split()[1]) / 1e6, 1)
    except Exception:
        return None

ENV = {
    "python": platform.python_version(), "platform": platform.platform(),
    "torch": torch.__version__, "torch_pinned_2_4_0": TORCH_PINNED, "snntorch": snntorch.__version__,
    "sklearn": sklearn.__version__, "numpy": np.__version__, "scipy": scipy.__version__,
    "pandas": pd.__version__, "wfdb": wfdb.__version__, "device": str(DEVICE),
    "cpu_count": N_CPU, "ram_gb": _ram_gb(), "on_kaggle": ON_KAGGLE, "machine": MACHINE, "smoke_test": SMOKE_TEST,
    "pipeline": PIPELINE, "cpu_model": next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), "?")
                 if os.path.exists("/proc/cpuinfo") else platform.processor(),
}
if DEVICE.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    ENV.update({"gpu": p.name, "gpu_sm_count": p.multi_processor_count, "gpu_capability": f"{p.major}.{p.minor}",
                "gpu_mem_gb": round(p.total_memory / 1e9, 1), "cuda": torch.version.cuda})
try:
    import brian2
    ENV["brian2"] = brian2.__version__
except Exception as e:
    ENV["brian2"] = f"unavailable ({e.__class__.__name__})"

if SMOKE_TEST:
    SEEDS = SEEDS[:1]
    GAINS = [1, 21]
    DENSE_LAMBDAS = DENSE_LAMBDAS[:1]
    RETUNE_TRIALS = 2
    NN_EPOCHS = 1
    BRIAN2_BEATS = 2
    NN_TUNE_TRIALS = 2
    NN_TUNE_MAX_EPOCHS = 2
    NN_TUNE_CHECKPOINTS = [1, 2]
    HUANG_MATRICES = 2
    WORKING_DIAG_BEATS = 40
    SAVE_STATES = [s for s in SAVE_STATES if s["seed"] in SEEDS]
SMOKE_CONFIGS   = [11, 12, 13, 22]       # used only when SMOKE_TEST
SMOKE_CV_SPLITS = [(1, 1), (1, 2), (1, 3)]   # used only when SMOKE_TEST (3: the CV test needs 3 splits)
SMOKE_PER_CLASS = {0: 10 ** 9, 1: 10 ** 9, 2: 10 ** 9}   # SMOKE keeps every beat here; the next cell keeps
                                         # those in the first SMOKE_SECONDS of each record

for k, v in ENV.items():
    print(f"{k:>20}: {v}")
if PIN_TORCH_240 and not TORCH_PINNED:
    print("NOTE: torch 2.4.0 is not active; the exact-replication comparison will still run but cannot match.")


## 2. The authors' code (pinned commit)

In [ ]:
# The authors' code, cloned at a pinned commit. Their functions are used directly;
# bugs are patched only inside the "fixed" liquid modes (see the liquid cell).
REPO_DIR = os.path.join(SCRATCH_DIR, "authors_repo")
os.makedirs(SCRATCH_DIR, exist_ok=True)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "-q", AUTHORS_REPO, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", AUTHORS_COMMIT], check=True)
ENV["authors_commit"] = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                                       capture_output=True, text=True).stdout.strip()
assert ENV["authors_commit"] == AUTHORS_COMMIT, ENV["authors_commit"]
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.preprocessing import signals as A_sig
from src.preprocessing import stratification as A_strat
from src.encoding import spike_coding as A_enc
from src.utils.reproducibility import seed_everything

def _load_script(name, rel):
    spec = importlib.util.spec_from_file_location(name, os.path.join(REPO_DIR, rel))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)          # only imports + definitions; their main blocks are guarded
    return mod

A03 = _load_script("authors_cv_script", "scripts/03_run_24_configurations_cv.py")
LSM3D = A03.LiquidStateMachine3D           # identical class in scripts 02 and 03 (checked)

AUTHORS_CONFIGS = pd.read_csv(os.path.join(REPO_DIR, "configs/all_24_configurations.csv"))
ENC_SHORT = {"multithreshold": "multi", "temporal_rate": "rate", "bounded_delay_latency": "latency"}
CONFIGS = {int(r.config_id): {"config_id": int(r.config_id), "filter": r.filtering, "norm": r.normalization,
                              "encoder": ENC_SHORT[r.encoding], "readout": "svm" if r.readout_loss == "hinge" else "lr",
                              "authors_csv": r.source_result_file}
           for r in AUTHORS_CONFIGS.itertuples()}
print(f"authors' code at {ENV['authors_commit'][:10]}; {len(CONFIGS)} configurations loaded")


## 3. lsm2: the liquid code

In [ ]:
# lsm2 (the liquid code: config, liquids, simulator, statistics, measures, gain rule, calibration, inputs,
# streaming), written to SCRATCH_DIR and imported.
LSM2_FILES = {}
LSM2_FILES['lsm2/config.py'] = r'''"""Every fixed number of the redesign plan, with its source or reason.

Sources (as labelled in the plan):
  (a) Perez-Lopez et al. 2026 and the authors' code, commit 02e45c4
  (c) Maass, Natschlager & Markram 2002, Section 5 and Appendix B
  (d) Schmitt 2022, Brian2 reproduction of (c)'s Figure 2 (maass2002.py)
  (e) Legenstein & Maass 2007
Numbers marked "plan" are the plan's own choices; the reason is in the plan (section named).
Numbers marked "code" were not fixed by the plan; their reason is given here and listed in NOTES.md.
"""
import math

# ---------------------------------------------------------------- time
FS = 360.0                          # MIT-BIH sampling rate (Hz), (a)
SAMPLE_MS = 1000.0 / FS             # 2.7778 ms, one ECG sample = one step of (a)'s liquid
BEAT_SAMPLES = 577                  # (a): beats are R-0.8 s .. R+0.8 s at 360 Hz
SIM_SAMPLES = 650                   # (a): num_sim_steps = 650 (input is zero after the beat)
R_SAMPLE = 288                      # plan, Readouts: the R peak is sample 288 of the 577-sample window
WIN, NWIN = 20, 32                  # (a): spike counts in 32 windows of 20 samples (film readout)
SNAPSHOT_SAMPLES = (288, 342, 396)  # plan, Readouts: R, R + 150 ms, R + 300 ms
STATE_TAU_MS = 30.0                 # (c): liquid states are spikes filtered with a 30 ms exponential
N_SUB_REFERENCE = 28                # plan, Time grid: 28 sub-steps per sample (0.0992 ms) is the reference grid

# ---------------------------------------------------------------- X: (a)'s liquid, both bugs fixed
X = dict(
    betaE=0.95, betaI=0.80,         # (a) leak per 2.778 ms step (about 54 / 12 ms)
    threshold=1.0,                  # (a) Uthr
    rho=0.9,                        # (a) target fill of the paper's input rule
    excit_ratio=0.8,                # (a) and (c)
    C={"EE": 0.3, "EI": 0.2, "IE": 0.4, "II": 0.1},   # (a) and (c)
    lam=0.8,                        # (a)
    mu_rsv={"EE": 0.020, "EI": 0.025, "IE": 0.050, "II": 0.040},   # (a) type means
    P_in=0.1,                       # (a) code line 691
    dims=(7, 7, 7),                 # (a) 343 neurons
)
I_TARGET = X["rho"] * (1 - X["betaE"]) * X["threshold"]     # 0.045: (a)'s rule, mean input per step
TYPES = ("EE", "EI", "IE", "II")    # pre-post

# ---------------------------------------------------------------- Maass (c), in the paper's units (threshold 1)
MAASS = dict(
    tau_m_ms=30.0,                  # (c) membrane time constant
    v_thresh_mV=15.0, v_reset_mV=13.5, I_b_nA=13.5, R_Mohm=1.0,   # (c); background 13.5 mV of 15
    refrac_ms={"E": 3.0, "I": 2.0},  # (c)
    tau_s_ms={"E": 3.0, "I": 6.0},   # (c) synaptic current decay by presynaptic type
    tau_in_ms=3.0,                   # plan: input synapses are excitatory, so 3 ms for every target
    delay_ms={"EE": 1.5, "EI": 0.8, "IE": 0.8, "II": 0.8},   # (c)
    A_nA={"EE": 30.0, "EI": 60.0, "IE": -19.0, "II": -19.0},  # (c)
    U={"EE": 0.5, "EI": 0.05, "IE": 0.25, "II": 0.32},          # (c)
    D_s={"EE": 1.1, "EI": 0.125, "IE": 0.7, "II": 0.144},       # (c)
    F_s={"EE": 0.05, "EI": 1.2, "IE": 0.02, "II": 0.06},        # (c)
    udf_spread=0.5,                  # (c): SD 50% of the mean
    lam=2.0,                         # (c)
    in_nA={"E": 18.0, "I": 9.0},     # (c) input amplitudes
    in_fraction=0.3,                 # (c) input injected into 30% of neurons
    dims=(15, 3, 3),                 # (c) 135-neuron column
    input_rate_hz=20.0,              # (d) (and (c)'s Figure 4 task)
)
TAP = MAASS["I_b_nA"] * MAASS["R_Mohm"] / MAASS["v_thresh_mV"]    # 0.9 of threshold
RESET_TAP = MAASS["v_reset_mV"] / MAASS["v_thresh_mV"]            # 0.9

def _push(A_nA, tau_s_ms, U=1.0):
    """Voltage push (in thresholds) of one current pulse A*exp(-t/tau_s) through R into a 30 ms membrane:
    A * U * tau_s * R / (tau_m * V_thresh), the tap's convention (plan, factors 9 and 10)."""
    return A_nA * 1e-9 * U * tau_s_ms * 1e-3 * MAASS["R_Mohm"] * 1e6 / (MAASS["tau_m_ms"] * 1e-3 * MAASS["v_thresh_mV"] * 1e-3)

_pre = {"EE": "E", "EI": "E", "IE": "I", "II": "I"}
MAASS_PUSH = {t: abs(_push(MAASS["A_nA"][t], MAASS["tau_s_ms"][_pre[t]], MAASS["U"][t])) for t in TYPES}
# EE 0.1, EI 0.02, IE 0.0633, II 0.0811 (plan, factor 3: ratios 1 / 0.2 / 0.63 / 0.81; factor 10: EE 0.1)
MAASS_RATIOS = {t: MAASS_PUSH[t] / MAASS_PUSH["EE"] for t in TYPES}
MAASS_IN_PUSH = {k: _push(MAASS["in_nA"][k], MAASS["tau_in_ms"]) for k in ("E", "I")}   # 0.12 / 0.06 (factor 9)
MAASS_EE_PUSH = MAASS_PUSH["EE"]   # 0.1 (factor 10)

# ---------------------------------------------------------------- adaptive thresholds (Bellec et al. 2018, LSNN)
# Bellec, Salaj, Subramoney, Legenstein & Maass 2018, "Long short-term memory and learning-to-learn in networks of
# spiking neurons", Section 5 parameters: threshold B_j = b0 + beta b_j, b_j <- rho b_j + (1 - rho) z_j with
# rho = exp(-dt / tau_a) and z_j = 1/dt (ms) at a spike; beta 1.6, b0 0.03, tau_a uniform on 1..1000 ms, 40% of the
# neurons adaptive. Translation to Maass's neurons (rest 0.9, threshold 1): the threshold rise is taken relative to the
# rest-to-threshold gap, theta_j = 1 + (1 - 0.9) beta b_j / b0, since that gap decides firing in both models.
ADAPT = dict(beta=1.6, b0=0.03, tau_ms=(1.0, 1000.0), fraction=0.4)

# ---------------------------------------------------------------- the 11 factors (plan table)
FACTOR_NAMES = {
    1: "reach lambda", 2: "code-only normalisations", 3: "E/I balance", 4: "leak", 5: "operating point",
    6: "input wiring", 7: "fine timing", 8: "dynamic synapses", 9: "input size", 10: "weight scale",
    11: "size and shape",
}
ALL_FACTORS = frozenset(range(1, 12))
LEAK_MS_Y = 30.0                    # factor 4: (c)'s 30 ms for E and I

# ---------------------------------------------------------------- gain grid (plan, Choosing the recurrent gain; decision 5)
G_LOW = (0.1, 0.3)
G_COARSE = (1, 3, 10, 30, 100, 300, 1000)
G_EXTEND_STEP, G_EXTEND_MAX = 3.0, 3
G_FINE_STEP, G_FINE_K = 1.4, 3      # g* x 1.4^k, k = -3..3 (1.4^3 ~ 2.7 splits a 3x step into three)

# ---------------------------------------------------------------- level thresholds (plan, Correctness criteria)
TOL_1A = 1e-9                       # 1a, 1b open loop (float64)
TOL_1C = 1e-4                       # 1c float32 vs float64
LEVEL2_SE = 4.5                     # level 2 (decision 24)
BAND_HZ = (2.0, 40.0)               # level 3 firing band (decision 7)
HIGH_HZ, HIGH_FRAC = 100.0, 0.05    # at most 5% of neurons above 100 Hz (decision 7)
N_CAL_BEATS = 500                   # plan: 500 beats for level 3 and the rank scores (decision 8)
GEN_BEATS, GEN_COPIES = 4, 125      # decision 8: 4 beats x 125 jittered copies
JITTER_MS = 4.0                     # decision 8: (e)'s 4 ms jitter for its 30-channel patterns
STATE_READ_SAMPLE = 396             # decision 8: states read at R + 300 ms
START_TEST_SAMPLE = 576             # level 3: starting-state rule judged at sample 576
MEMORY_T_MS = (0, 25, 50, 100, 200, 400)   # level 4 lags
MEMORY_PASS_T_MS = 100              # level 4 pass lag (plan: well past 30-54 ms membranes and 3-6 ms currents)
MEMORY_MIN_GAIN = 0.05              # level 4 (decision 9)
MEMORY_WARMUP = 100                 # level 4: 278 ms warm-up
SURROGATE_TOL = 0.10                # level 4: surrogate spikes per sample within 10% of ECG's
E1D_N_TRY_MAX = 28                  # 1d: n = 1, 2, 3, ... up to the reference grid
E1E_T_CRIT = 4.2                    # 1e: t at 8 df, two-sided 0.3%
E1E_FRAC = 0.95                     # 1e: 95% of time points
E1E_RUNS = 5                        # 1e: 5 runs per side

SEEDS_SEARCH = (1, 2, 3)            # code: liquid seeds for the ladder search (the plan fixes 3 seeds, not their values)
SEEDS_CONFIRM = (101, 202, 303)     # plan, confirmation
SEEDS_RETRY = (404, 505, 606)       # plan, confirmation retry
SEEDS_TEST_REDESIGN = (707, 808, 909)   # plan, Maass check redesign
SEEDS_CLASSIFY = (2026, 69, 420)    # plan, ground rules

# ---------------------------------------------------------------- code choices the plan does not fix (see NOTES.md)
CODE = dict(
    twin_beats=100,          # twin-run cross-check per g and seed: a reported cross-check, 100 beats give a stable mean ratio
    twin_early=(289, 306),   # samples of the first 50 ms after the extra spike
    twin_late=(559, 576),    # samples of the last 50 ms of the beat
    memory_sequences=500,    # level 4: same count as level 3's beats; 250 train / 250 test sequences
    ridge_alphas=tuple(10.0 ** k for k in range(-3, 5)),   # level 4 inner-CV penalty grid
    ridge_inner_folds=5,     # level 4 inner CV (grouped by sequence)
    calib_tol=0.01,          # factor 5: bisection stops at 1% of the target rate (well inside seed-to-seed spread)
    calib_iters=40,          # factor 5: bisection cap
    search_split_seed=0,     # DS1 patients halved by a fixed random permutation of the 22 records
    sim_batch=500,           # beats per GPU batch (fixed on every machine: plan, determinism)
    x_update="equations",    # X's reset follows (a)'s Eqs. 17-19 (the plan). "authors_code" (the snnTorch neuron of
                             # (a)'s code) exists only so the tests can show the difference; the notebook never sets it
)
'''
LSM2_FILES['lsm2/liquid.py'] = r'''"""Building liquids: every ladder candidate (X plus any set of the 11 factors), Maass's two versions for the
Maass check (script and paper), and X converted from the authors' own class (for 1f and E1's paper liquid).

A Liquid holds the realised network and every time constant already converted to its grid (n sub-steps per
input step). Input weights are stored as a base matrix; the input size (paper's rule, factor-5 calibration or
factor 9) is a separate scalar `in_scale`, and the recurrent gain g a scalar given to the simulator.
"""
from dataclasses import dataclass, field
import math
import numpy as np

from . import config as C

TYPE_INDEX = {t: i for i, t in enumerate(C.TYPES)}     # EE 0, EI 1, IE 2, II 3


def round_half_up(x):
    return int(math.floor(x + 0.5))


def steps_of(ms, h_ms):
    """A time in ms as a whole number of sub-steps: rounded to the nearest, at least 1 (plan, Time grid)."""
    return max(1, int(round(ms / h_ms)))


@dataclass
class Liquid:
    # structure
    N: int
    dims: tuple
    is_exc: np.ndarray          # (N,) bool
    pos: np.ndarray             # (N, 3)
    pre: np.ndarray             # (S,) int
    post: np.ndarray            # (S,) int
    typ: np.ndarray             # (S,) int 0..3 (EE, EI, IE, II)
    w: np.ndarray               # (S,) signed base weight: a push in thresholds ("ours"), or J amplitude ("brian2")
    U: np.ndarray               # (S,) utilisation (dynamic synapses), else ones
    D_ms: np.ndarray            # (S,)
    F_ms: np.ndarray            # (S,)
    Ubar: np.ndarray            # (4,) nominal type means of U
    Win: np.ndarray             # (C, N) base input weights, >= 0
    in_neurons: np.ndarray      # (N,) bool: neurons with any input synapse
    # time grid and neuron model
    n_sub: int                  # sub-steps per input step
    h_ms: float                 # sub-step length
    beta: np.ndarray            # (N,) leak per sub-step
    tap: np.ndarray             # (N,) tap per sub-step ("ours"); 0 without factor 5
    reset_mode: str             # "subtract" or "value"
    reset_value: float
    start_lo: float             # starting voltages drawn uniformly in [start_lo, start_hi]; equal = fixed
    start_hi: float
    rest: float                 # resting level (0, or 0.9 with the tap): lower end of level 3's random starts
    gap: np.ndarray             # (N,) int minimum gap between spikes, in sub-steps
    clamp: bool                 # voltage clamped during the blocked sub-steps
    strict: bool                # threshold test u > 1 (Brian2 script) instead of u >= 1 (ours, (a)'s H)
    schedule: str               # "ours" or "brian2"
    syn_model: str              # "static", "markram" (two-state, (c)'s form) or "tsodyks3" ((d)'s three-state)
    delay: np.ndarray           # (4,) int sub-steps by type
    cur_a: object               # None (push at once) or (2,) decay per sub-step for E- and I-presynaptic currents
    in_a: object                # None or (N,) decay per sub-step of each neuron's input current
    # Brian2-schedule constants (script version only)
    J_b: float = 0.0
    kappa: object = None        # (N,) exact weight of the decaying input current in the membrane update
    syn_tau_ms: object = None   # (4,) current decay per type (tsodyks3)
    # bookkeeping
    factors: frozenset = frozenset()
    snn_quirk: bool = False     # (a)'s code (snnTorch Leaky, reset_delay=False): the threshold test subtracts 1 more
                                # when the stored voltage is above 1; u > 1 strict. Default: (a)'s Eqs. 17-19
    g_maass: float = None       # g at Maass's own scale (factor 10)
    in_scale_fixed: float = None   # input size fixed by the design (factor 9, Maass versions)
    in_split: tuple = (1.0, 1.0)   # base input means onto E / I neurons (2:1 with factors 5 or 9)
    adapt_c: object = None         # (N,) threshold rise per unit of b (0 = not adaptive); None = no adaptation
    adapt_tau_ms: object = None    # (N,) adaptation time constants (ms)
    thr_base: object = None        # (N,) fixed per-neuron threshold (the frozen-threshold controls); None = 1
    reset_to_thr: bool = False     # subtract reset removes the current threshold (Salaj et al. 2021), not 1
    meta: dict = field(default_factory=dict)

    @property
    def S(self):
        return len(self.pre)

    def ee_mean_push(self, g=1.0):
        """Mean isolated EE push per presynaptic spike, in thresholds (the plan's absolute g scale)."""
        m = self.typ == 0
        if not m.any():
            return 0.0
        iso = np.abs(self.w[m]) * (self.U[m] / self.Ubar[0] if self.syn_model == "markram" else 1.0)
        return float(g * iso.mean())


# ---------------------------------------------------------------- shared pieces
def _grid(dims):
    return np.stack(np.meshgrid(*[np.arange(d) for d in dims], indexing="ij"), -1).reshape(-1, 3).astype(np.float64)


def _type_matrix(is_exc):
    pe, qe = is_exc[:, None], is_exc[None, :]
    T = np.where(pe & qe, 0, np.where(pe & ~qe, 1, np.where(~pe & qe, 2, 3)))
    return T


def _draw_udf(rng_n, rng_u, T, mean_of_type, cap, bad_rule):
    """U, D or F for every (pre, post) pair: Normal(mean, 0.5 mean); draws at or below 0 replaced by
    uniform(0, 2 mean) ("redraw", plan decision 17) or by the mean ("mean", (d)); U capped at 1 if cap."""
    mean = np.vectorize(lambda t: mean_of_type[C.TYPES[t]])(T).astype(np.float64)
    z = rng_n.standard_normal(T.shape)
    v = mean + C.MAASS["udf_spread"] * mean * z
    repl = rng_u.random(T.shape) * 2 * mean
    if bad_rule == "redraw":
        v = np.where(v <= 0, repl, v)
    else:      # (d): U < 0 -> mean; D, F <= 0 -> mean
        v = np.where(v <= 0, mean, v)
    if cap:
        v = np.minimum(v, 1.0)
    return v


# ---------------------------------------------------------------- ladder candidates
def candidate_settings(factors):
    f = frozenset(factors)
    return {
        "dims": C.MAASS["dims"] if 11 in f else C.X["dims"],
        "lam": C.MAASS["lam"] if 1 in f else C.X["lam"],
        "mu": ({t: C.X["mu_rsv"]["EE"] * C.MAASS_RATIOS[t] for t in C.TYPES} if 3 in f else dict(C.X["mu_rsv"])),
        "normalise": 2 not in f,
        "tau_E_ms": C.LEAK_MS_Y if 4 in f else -C.SAMPLE_MS / math.log(C.X["betaE"]),
        "tau_I_ms": C.LEAK_MS_Y if 4 in f else -C.SAMPLE_MS / math.log(C.X["betaI"]),
        "tap": 5 in f,
        "input30": 6 in f,
        "fine": 7 in f,
        "dynamic": 8 in f,
        "maass_input": 9 in f,
        "maass_scale": 10 in f,
    }


def build_candidate(factors, seed, n_sub, n_channels, n_sub_input_step_ms=C.SAMPLE_MS, in_fraction=None):
    """A ladder candidate: X ((a), bugs fixed) with the given factors switched on.
    n_sub: sub-steps per sample (1 without factor 7 unless a grid check asks otherwise).
    in_fraction (factor 6 only): share of neurons that receive the input; None = (c)'s 30%. 1.0 = every neuron, as
    in (d)'s script. The input weights come from the same draw either way, so the 30% neurons keep their weights."""
    f = frozenset(factors)
    s = candidate_settings(f)
    N = int(np.prod(s["dims"]))
    h = n_sub_input_step_ms / n_sub
    # independent streams, so candidates with the same seed share their random numbers where they can
    r_types, r_conn, r_w = (np.random.default_rng([seed, k]) for k in (1, 2, 3))
    r_un, r_uu, r_dn, r_du, r_fn, r_fu = (np.random.default_rng([seed, k]) for k in (4, 5, 6, 7, 8, 9))
    r_inm, r_inw, r_inn = (np.random.default_rng([seed, k]) for k in (10, 11, 12))
    n_exc = int(N * C.X["excit_ratio"])
    is_exc = np.zeros(N, bool)
    is_exc[r_types.permutation(N)[:n_exc]] = True
    pos = _grid(s["dims"])
    T = _type_matrix(is_exc)
    Cm = np.vectorize(lambda t: C.X["C"][C.TYPES[t]])(T)
    d = np.sqrt(((pos[:, None, :] - pos[None, :, :]) ** 2).sum(-1))
    p = np.clip(Cm * np.exp(-(d / s["lam"]) ** 2), 0, 1)
    conn = r_conn.random((N, N)) < p
    np.fill_diagonal(conn, False)                            # no self-connections ((a); kept in Y)
    mu = np.vectorize(lambda t: s["mu"][C.TYPES[t]])(T)
    Wmag = r_w.standard_exponential((N, N)) * mu             # Gamma(shape 1, scale mu): spread = mean
    if s["normalise"]:
        fan_in = np.maximum(conn.sum(0, keepdims=True), 1)   # (a) lines 168-169: divide by the receiver's count
        Wmag = Wmag / fan_in
    W = np.where(is_exc[:, None], Wmag, -Wmag)
    pre, post = np.nonzero(conn)
    order = np.lexsort((post, pre, T[pre, post]))           # sorted by type, then pre, then post
    pre, post = pre[order], post[order]
    typ = T[pre, post]
    if s["dynamic"]:
        Ufull = _draw_udf(r_un, r_uu, T, C.MAASS["U"], True, "redraw")
        Dfull = _draw_udf(r_dn, r_du, T, {k: v * 1000 for k, v in C.MAASS["D_s"].items()}, False, "redraw")
        Ffull = _draw_udf(r_fn, r_fu, T, {k: v * 1000 for k, v in C.MAASS["F_s"].items()}, False, "redraw")
        U, D, F = Ufull[pre, post], Dfull[pre, post], Ffull[pre, post]
    else:
        U, D, F = np.ones(len(pre)), np.full(len(pre), np.inf), np.full(len(pre), np.inf)
    Ubar = np.array([C.MAASS["U"][t] for t in C.TYPES])
    # input wiring and base weights
    if s["input30"]:
        k = round_half_up((C.MAASS["in_fraction"] if in_fraction is None else in_fraction) * N)
        in_n = np.zeros(N, bool)
        in_n[r_inn.permutation(N)[:k]] = True
        mask = np.repeat(in_n[None, :], n_channels, 0)
    else:
        mask = r_inm.random((n_channels, N)) < C.X["P_in"]
        in_n = mask.any(0)
    split = (1.0, 0.5) if (s["tap"] or s["maass_input"]) else (1.0, 1.0)
    m_in = np.where(is_exc, split[0], split[1])[None, :]
    Win = r_inw.standard_exponential((n_channels, N)) * m_in * mask
    if s["normalise"]:                                       # (a) lines 71-75: each column's mean active weight = m
        cnt = mask.sum(0, keepdims=True)
        col = Win.sum(0, keepdims=True) / (cnt + 1e-8)
        Win = Win * (m_in / (col + 1e-8))
    # time constants on the grid
    tau = np.where(is_exc, s["tau_E_ms"], s["tau_I_ms"])
    beta = np.exp(-h / tau)
    tap = C.TAP * (1 - beta) if s["tap"] else np.zeros(N)
    if s["fine"]:
        gap = np.where(is_exc, steps_of(C.MAASS["refrac_ms"]["E"], h), steps_of(C.MAASS["refrac_ms"]["I"], h))
        delay = np.array([steps_of(C.MAASS["delay_ms"][t], h) for t in C.TYPES])
        cur_a = np.exp(-h / np.array([C.MAASS["tau_s_ms"]["E"], C.MAASS["tau_s_ms"]["I"]]))
        in_a = np.full(N, math.exp(-h / C.MAASS["tau_in_ms"]))
        clamp = True
    else:
        # X: the 2.78 ms gap and the one-sample delay, in ms, converted to the grid in use
        gap = np.full(N, steps_of(C.SAMPLE_MS, h))
        delay = np.full(4, steps_of(C.SAMPLE_MS, h))
        cur_a, in_a, clamp = None, None, False
    liq = Liquid(
        N=N, dims=tuple(s["dims"]), is_exc=is_exc, pos=pos, pre=pre, post=post, typ=typ, w=W[pre, post].astype(np.float64),
        U=U, D_ms=D, F_ms=F, Ubar=Ubar, Win=Win.astype(np.float64), in_neurons=in_n,
        n_sub=n_sub, h_ms=h, beta=beta, tap=tap,
        reset_mode="value" if s["tap"] else "subtract", reset_value=C.RESET_TAP if s["tap"] else 0.0,
        start_lo=C.RESET_TAP if s["tap"] else 0.0, start_hi=1.0 if s["tap"] else 0.0,
        rest=C.TAP if s["tap"] else 0.0,
        gap=gap.astype(np.int64), clamp=clamp, strict=False, schedule="ours",
        syn_model="markram" if s["dynamic"] else "static", delay=delay.astype(np.int64), cur_a=cur_a, in_a=in_a,
        factors=f, in_split=split,
        meta={"kind": "candidate", "seed": int(seed), "n_channels": int(n_channels), "settings": s},
    )
    if liq.reset_mode == "subtract" and C.CODE["x_update"] == "authors_code":
        liq.snn_quirk, liq.strict = True, True
    m = typ == 0
    liq.g_maass = C.MAASS_EE_PUSH / float(np.abs(liq.w[m]).mean()) if m.any() else None
    if s["maass_input"]:
        liq.in_scale_fixed = C.MAASS_IN_PUSH["E"]           # base means 1 / 0.5 -> 0.12 / 0.06 per input spike
    return liq


def add_adaptation(liq, seed, beta=None, b0=None, tau_ms=None, fraction=None):
    """Adaptive thresholds (config.ADAPT, Bellec et al. 2018 Section 5) on a random `fraction` of the neurons, each
    with its own tau_a drawn uniformly from tau_ms. Own random streams, so the rest of the liquid is unchanged."""
    a = C.ADAPT
    beta = a["beta"] if beta is None else beta
    b0 = a["b0"] if b0 is None else b0
    lo, hi = a["tau_ms"] if tau_ms is None else tau_ms
    frac = a["fraction"] if fraction is None else fraction
    N = liq.N
    k = round_half_up(frac * N)
    chosen = np.zeros(N, bool)
    chosen[np.random.default_rng([seed, 13]).permutation(N)[:k]] = True
    tau = np.full(N, np.inf)
    tau[chosen] = np.random.default_rng([seed, 14]).uniform(lo, hi, k)
    gap = 1.0 - (liq.rest if liq.rest > 0 else 0.0)
    liq.adapt_c = np.where(chosen, gap * beta / b0, 0.0)
    liq.adapt_tau_ms = tau
    liq.meta["adaptation"] = {"beta": beta, "b0": b0, "tau_ms": [lo, hi], "fraction": frac, "n_adaptive": int(k), "gap": gap}
    return liq


def p_in_effective(liq):
    """Connection probability the paper's rule uses: P_in = 0.1, or 1 for the input neurons with factor 6."""
    return 1.0 if 6 in liq.factors else C.X["P_in"]


def paper_rule_scale(ain, liq):
    """(a)'s rule: mu_in = I_target / (A_in P_in), always with X's excitatory leak (plan, Input size)."""
    return C.I_TARGET / (ain * p_in_effective(liq))


# ---------------------------------------------------------------- Maass's liquid, two versions (Maass check)
def build_maass_paper(seed, n_sub=None, n_channels=1):
    """Our reading of (c), the version Y copies: Y with all 11 factors and one Poisson input channel."""
    n_sub = n_sub or C.N_SUB_REFERENCE
    liq = build_candidate(C.ALL_FACTORS, seed, n_sub, n_channels)
    liq.meta["kind"] = "maass_paper"
    return liq


def build_maass_script(seed, n_sub=None, step_ms=C.SAMPLE_MS, d_replacement="fixed"):
    """The liquid of (d)'s Brian2 script, in threshold units, with its six settings (plan, level 1e):
    input onto I neurons decays over 6 ms, bad U, D, F draws replaced by the mean, U not capped, no input delay,
    an arriving spike acts after that step's state update (Brian2 schedule), each step's synaptic current held
    constant (exact neuron update with step-constant synaptic current). Also from (d): input to all 135 neurons
    at a fixed 18 / 9 nA, self-connections allowed, E neurons are the first 108 indices with shuffled positions,
    three-state synapses (used resources recover only once the current has ended)."""
    n_sub = n_sub or C.N_SUB_REFERENCE
    N = int(np.prod(C.MAASS["dims"]))
    h = step_ms / n_sub
    rng = np.random.default_rng([seed, 1000])
    idx = rng.permutation(N)
    pos = np.stack([idx % 3, (idx // 3) % 3, idx // 9], 1).astype(np.float64)
    n_exc = int(0.8 * N)
    is_exc = np.zeros(N, bool); is_exc[:n_exc] = True
    T = _type_matrix(is_exc)
    Cm = np.vectorize(lambda t: C.X["C"][C.TYPES[t]])(T)
    d2 = ((pos[:, None, :] - pos[None, :, :]) ** 2).sum(-1)
    p = Cm * np.exp(-d2 / C.MAASS["lam"] ** 2)
    conn = np.random.default_rng([seed, 1001]).random((N, N)) < p          # self-connections allowed ((d))
    pre, post = np.nonzero(conn)
    order = np.lexsort((post, pre, T[pre, post]))
    pre, post = pre[order], post[order]
    typ = T[pre, post]
    J = C.MAASS["R_Mohm"] / C.MAASS["v_thresh_mV"]                         # nA -> thresholds (1 MOhm / 15 mV)
    A = np.array([C.MAASS["A_nA"][C.TYPES[t]] for t in typ])
    r = np.random.default_rng([seed, 1002])
    w = np.sign(A) * r.standard_exponential(len(pre)) * np.abs(A) * J       # Gamma(1, |A|) with sign ((d))
    def udf(name, scale, k):
        mean = np.array([C.MAASS[name][C.TYPES[t]] for t in typ]) * scale
        v = mean + 0.5 * mean * np.random.default_rng([seed, k]).standard_normal(len(pre))
        if d_replacement == "as_written":       # (d) as written: only the first two synapses of each group
            for ti in range(4):
                first = np.nonzero(typ == ti)[0][:2]
                v[first] = mean[first]
            return v
        return np.where(v < 0, mean, v) if name == "U" else np.where(v <= 0, mean, v)
    U, D, F = udf("U", 1.0, 1003), udf("D_s", 1000.0, 1004), udf("F_s", 1000.0, 1005)
    Ubar = np.array([C.MAASS["U"][t] for t in C.TYPES])
    Win = (np.where(is_exc, C.MAASS["in_nA"]["E"], C.MAASS["in_nA"]["I"]) * J)[None, :]   # J jump per input spike
    tau_m = C.MAASS["tau_m_ms"]
    beta = np.full(N, math.exp(-h / tau_m))
    in_tau = np.where(is_exc, 3.0, 6.0)                                     # (d): tau_stimulus 3 ms (E), 6 ms (I)
    a_in = np.exp(-h / in_tau)
    kappa = in_tau / (in_tau - tau_m) * (np.exp(-h / in_tau) - np.exp(-h / tau_m))   # Brian2's exact solution
    gap = np.where(is_exc, int((C.MAASS["refrac_ms"]["E"] + 1e-3 * h) / h), int((C.MAASS["refrac_ms"]["I"] + 1e-3 * h) / h))
    delay = np.array([int(round(C.MAASS["delay_ms"][t] / h)) for t in C.TYPES])   # Brian2 rounds delays to the grid
    syn_tau = np.array([C.MAASS["tau_s_ms"]["E"], C.MAASS["tau_s_ms"]["E"], C.MAASS["tau_s_ms"]["I"], C.MAASS["tau_s_ms"]["I"]])
    liq = Liquid(
        N=N, dims=C.MAASS["dims"], is_exc=is_exc, pos=pos, pre=pre, post=post, typ=typ, w=w, U=U, D_ms=D, F_ms=F,
        Ubar=Ubar, Win=Win, in_neurons=np.ones(N, bool), n_sub=n_sub, h_ms=h, beta=beta, tap=np.zeros(N),
        reset_mode="value", reset_value=C.RESET_TAP, start_lo=C.RESET_TAP, start_hi=1.0, rest=C.TAP,
        gap=gap.astype(np.int64), clamp=True, strict=True, schedule="brian2", syn_model="tsodyks3",
        delay=delay.astype(np.int64), cur_a=None, in_a=a_in, J_b=C.TAP, kappa=kappa, syn_tau_ms=syn_tau,
        factors=frozenset(), in_scale_fixed=1.0, in_split=(1.0, 0.5),
        meta={"kind": "maass_script", "seed": int(seed), "n_channels": 1, "d_replacement": d_replacement},
    )
    liq.g_maass = 1.0
    return liq


# ---------------------------------------------------------------- X from the authors' class
def from_authors_model(model, n_sub=1, snn_quirk=None):
    """X exactly as (a)'s class built it (weights already include its normalisations and its g_rec):
    used for the 1f regression and for E1's paper liquid at g = 1. Input weights are returned at their size
    (in_scale 1)."""
    import torch
    W = model.rsv_weights.detach().double().cpu().numpy()
    Win = model.input_rsv_weights.detach().double().cpu().numpy()
    is_exc = model.is_excit.detach().cpu().numpy().astype(bool)
    N = W.shape[0]
    T = _type_matrix(is_exc)
    pre, post = np.nonzero(W)
    order = np.lexsort((post, pre, T[pre, post]))
    pre, post = pre[order], post[order]
    h = C.SAMPLE_MS / n_sub
    tau = np.where(is_exc, -C.SAMPLE_MS / math.log(C.X["betaE"]), -C.SAMPLE_MS / math.log(C.X["betaI"]))
    liq = Liquid(
        N=N, dims=tuple(model.reservoir_dim), is_exc=is_exc, pos=_grid(model.reservoir_dim), pre=pre, post=post,
        typ=T[pre, post], w=W[pre, post], U=np.ones(len(pre)), D_ms=np.full(len(pre), np.inf),
        F_ms=np.full(len(pre), np.inf), Ubar=np.array([C.MAASS["U"][t] for t in C.TYPES]), Win=Win,
        in_neurons=(Win != 0).any(0), n_sub=n_sub, h_ms=h, beta=np.exp(-h / tau), tap=np.zeros(N),
        reset_mode="subtract", reset_value=0.0, start_lo=0.0, start_hi=0.0, rest=0.0,
        gap=np.full(N, steps_of(C.SAMPLE_MS, h)), clamp=False, strict=False, schedule="ours", syn_model="static",
        delay=np.full(4, steps_of(C.SAMPLE_MS, h)), cur_a=None, in_a=None, factors=frozenset(),
        in_scale_fixed=1.0, meta={"kind": "authors_X", "n_channels": int(Win.shape[0])},
    )
    if snn_quirk or (snn_quirk is None and C.CODE["x_update"] == "authors_code"):
        liq.snn_quirk, liq.strict = True, True
    return liq
'''
LSM2_FILES['lsm2/sim.py'] = r'''"""The simulator: one batched update for X, Y and every candidate between them (each factor is a setting of the
Liquid), plus the Brian2 schedule used only for (d)'s own liquid (level 1e and the Maass check's script version).

Units: thresholds (the voltage threshold is 1). Time: input steps (one ECG sample, or 0.1 ms for 1e), each split
into n sub-steps. Our order within a sub-step (plan, Simulation details):
  1. synapse states recover/fade over the sub-step; spikes whose delay ends now arrive (dynamic synapses
     update u, push, R in (d)'s order)
  2. currents: without fine timing a push arrives all at once; with it, it feeds a current that decays over
     3 ms (from E and from inputs) or 6 ms (from I), scaled so its total equals the push
  3. every neuron integrates: u <- beta u + input + recurrent + tap (not during a clamped refractory period)
  4. threshold test (u >= 1, outside the minimum gap), then reset (subtract 1, or set to 0.9 with the tap)
Input spikes of an input step act in its first sub-step ((a)'s Eq. 16: x[t] W_in in step t); this step's spikes
reach their targets from the next sub-step on (delay >= 1).
"""
import math
import numpy as np
import torch

from . import config as C

# Fast path (same arithmetic, fewer Python round trips): the "ours" schedule run one input step at a time on fixed
# buffers, and on a CUDA card each input step replayed as one captured CUDA graph. The notebook compares it with the
# plain loop on the card before using it and switches it off if they differ.
FAST = True
USE_CUDA_GRAPHS = True


def _t(a, device, dtype):
    return torch.as_tensor(np.asarray(a), device=device, dtype=dtype)


class Simulator:
    def __init__(self, liq, device="cpu", dtype=torch.float64, g=1.0, in_scale=None, recurrence=True):
        self.liq, self.device, self.dtype = liq, torch.device(device), dtype
        self.g = float(g) if recurrence else 0.0
        if in_scale is None:
            if liq.in_scale_fixed is None:
                raise ValueError("this liquid needs an input size (paper rule or calibration)")
            in_scale = liq.in_scale_fixed
        self.in_scale = float(in_scale)
        d, t = self.device, dtype
        N = liq.N
        self.N = N
        self.beta = _t(liq.beta, d, t)
        self.tap = _t(liq.tap, d, t)
        self.gap = _t(liq.gap, d, torch.int64)
        self.Win = _t(liq.Win * self.in_scale, d, t)
        self.z_decay = math.exp(-liq.h_ms / C.STATE_TAU_MS)
        # adaptive thresholds: theta = 1 + c b; b <- rho b + (1 - rho) spike / h  (b in spikes per ms)
        self.adapt = liq.adapt_c is not None and bool(np.any(np.asarray(liq.adapt_c) > 0))
        if self.adapt:
            self.a_c = _t(liq.adapt_c, d, t)
            rho = np.exp(-liq.h_ms / np.asarray(liq.adapt_tau_ms, float))     # 1 for non-adaptive neurons (tau inf)
            self.a_rho = _t(rho, d, t)
            self.a_in = _t((1.0 - rho) / liq.h_ms, d, t)
        # fixed per-neuron thresholds (frozen controls): theta = thr_base (+ c b with adaptation); None = 1
        self.thr_base = None if liq.thr_base is None else _t(liq.thr_base, d, t)
        self.var_thr = self.adapt or self.thr_base is not None
        if liq.reset_to_thr and liq.snn_quirk:
            raise ValueError("the threshold reset needs the Eqs. 17-19 neuron (snn_quirk off)")
        self.L = int(max(liq.delay.max(), 1)) + 1           # spike-history ring length
        S = liq.S
        self.S = S
        self.static = liq.syn_model == "static"
        if liq.schedule == "ours":
            if self.static:
                # one dense matrix per (delay, presynaptic class): pushes of the spikes that left `delay` sub-steps ago
                self.groups = {}
                for ti in range(4):
                    m = liq.typ == ti
                    if not m.any():
                        continue
                    key = (int(liq.delay[ti]), 0 if ti < 2 else 1)
                    W = self.groups.get(key)
                    if W is None:
                        W = torch.zeros(N, N, device=d, dtype=t)
                    W[_t(liq.pre[m], d, torch.int64), _t(liq.post[m], d, torch.int64)] += _t(liq.w[m] * self.g, d, t)
                    self.groups[key] = W
            else:
                self._prep_sparse()
                Ubar_s = liq.Ubar[liq.typ]
                self.wg = _t(liq.w * self.g / Ubar_s, d, t)        # push = (w / U-bar) u R
                self.U = _t(liq.U, d, t)
                self.eD = _t(np.exp(-liq.h_ms / liq.D_ms), d, t)
                self.eF = _t(np.exp(-liq.h_ms / liq.F_ms), d, t)
            if liq.cur_a is not None:
                self.aE, self.aI = float(liq.cur_a[0]), float(liq.cur_a[1])
            self.in_a = None if liq.in_a is None else _t(liq.in_a, d, t)
        else:   # Brian2 schedule, (d)'s three-state synapses, currents in voltage-equivalent units
            self._prep_sparse()
            self.wg = _t(liq.w * self.g, d, t)
            self.U = _t(liq.U, d, t)
            tau = liq.syn_tau_ms[liq.typ]
            D = liq.D_ms
            h = liq.h_ms
            self.ey = _t(np.exp(-h / tau), d, t)
            self.eD = _t(np.exp(-h / D), d, t)
            self.cxy = _t(tau * (np.exp(-h / tau) - np.exp(-h / D)) / (D - tau), d, t)
            self.eF = _t(np.exp(-h / liq.F_ms), d, t)
            self.kappa = _t(liq.kappa, d, t)
            self.in_a = _t(liq.in_a, d, t)
            self.J_b = float(liq.J_b)

    def _prep_sparse(self):
        liq, d, t = self.liq, self.device, self.dtype
        self.pre_idx = _t(liq.pre, d, torch.int64)
        self.syn_delay = liq.delay[liq.typ]
        self.delay_groups = [(int(dl), _t(np.nonzero(self.syn_delay == dl)[0], d, torch.int64))
                             for dl in np.unique(self.syn_delay)]
        self.post_idx = _t(liq.post, d, torch.int64)
        self.nE = int((liq.typ < 2).sum())                  # synapses are sorted by type: E-presynaptic first
        M = torch.zeros(liq.S, liq.N, device=d, dtype=t)
        if liq.S:
            M[torch.arange(liq.S, device=d), _t(liq.post, d, torch.int64)] = 1.0
        self.M_E, self.M_I, self.M = M[:self.nE], M[self.nE:], M

    # ------------------------------------------------------------------ helpers
    def _arrivals(self, hist, k, B):
        """(B, S) 0/1: which synapses receive a spike in sub-step k."""
        a = torch.zeros(B, self.S, device=self.device, dtype=self.dtype)
        for dl, idx in self.delay_groups:
            if k - dl < 0:
                continue
            src = hist[(k - dl) % self.L]
            a[:, idx] = src[:, self.pre_idx[idx]]
        return a

    def start_state(self, u0):
        return torch.as_tensor(u0, device=self.device, dtype=self.dtype).clone()

    def _threshold(self, ab):
        """The current threshold of every neuron (only when it is not the fixed 1)."""
        if self.adapt:
            return (1.0 if self.thr_base is None else self.thr_base) + self.a_c * ab
        return self.thr_base

    # ------------------------------------------------------------------ main loop
    @torch.no_grad()
    def run(self, x, u0, T, rec=None, open_loop=None, trace=False, own_spikes=None, fast=None, state=None, t0=0):
        """x: (B, T_in, C) input spikes per input step; u0: (B, N) starting voltages; T: input steps to simulate.
        rec: {"windows": (win, nwin), "states_at": [t, ...], "states_every": (t0, t1), "neuron_counts": (t0, t1),
              "raster": bool}; times are input steps, a state is read at the end of its step.
        open_loop: (T*n, B, N) recorded spikes used as the presynaptic spikes (the liquid's own spikes are still
        computed and returned). own_spikes: (T*n, B, N) recorded spikes that also replace each neuron's own
        threshold decisions (level 1c: pure arithmetic drift, no flips); out["flips"] counts where the threshold
        test would have decided differently. trace: return u and currents every sub-step and synapse states
        every input step. fast: None = the module's FAST; False = this plain loop (the reference).
        Streaming (fast path only): state = the "state" a previous call returned (rec "keep_state"), to continue from
        it (u0 is then ignored); t0 = the input steps already simulated before this call (keeps the spike-history
        ring in phase). rec "step_counts": spikes per neuron in every input step, (B, T, N) uint8; rec "ab_weight":
        (B, T) weights, returns "ab_sum" = sum over steps of weight x adaptation variable b (at each step's end)."""
        liq, dev, dt = self.liq, self.device, self.dtype
        rec = rec or {}
        x = torch.as_tensor(x, device=dev)
        streaming = state is not None or t0 or rec.get("step_counts") or rec.get("keep_state") or rec.get("ab_weight") is not None
        if (FAST if fast is None else fast) and liq.schedule == "ours" and open_loop is None and own_spikes is None and not trace \
                and not rec.get("raster"):
            return self._run_fast(x, u0, T, rec, graphs=USE_CUDA_GRAPHS and dev.type == "cuda", state=state, t0=t0)
        if streaming:
            raise ValueError("streaming (state, t0, step_counts, keep_state, ab_weight) needs the fast path")
        B, T_in, Cin = x.shape
        N, n = self.N, liq.n_sub
        u = self.start_state(u0)
        ref = torch.zeros(B, N, device=dev, dtype=torch.int64)
        hist = torch.zeros(self.L, B, N, device=dev, dtype=dt)
        z = torch.zeros(B, N, device=dev, dtype=dt)
        ab = torch.zeros(B, N, device=dev, dtype=dt) if self.adapt else None
        ours = liq.schedule == "ours"
        fine = ours and liq.cur_a is not None
        IE = torch.zeros(B, N, device=dev, dtype=dt) if fine else None
        II = torch.zeros(B, N, device=dev, dtype=dt) if fine else None
        Iin = torch.zeros(B, N, device=dev, dtype=dt) if (self.in_a is not None) else None
        if not self.static or not ours:
            R = torch.ones(B, self.S, device=dev, dtype=dt)       # resources ready (x in (d))
            q = torch.zeros(B, self.S, device=dev, dtype=dt)      # utilisation u
            y = torch.zeros(B, self.S, device=dev, dtype=dt) if not ours else None
        out = {}
        win = rec.get("windows")
        if win:
            wl, nw = win
            counts = torch.zeros(B, nw, N, device=dev, dtype=torch.int32)
        st_at = sorted(set(rec.get("states_at", [])))
        st_at_set = {t: i for i, t in enumerate(st_at)}
        if st_at:
            states = torch.zeros(B, len(st_at), N, device=dev, dtype=torch.float32)
        se = rec.get("states_every")
        if se:
            states_every = torch.zeros(B, se[1] - se[0], N, device=dev, dtype=torch.float32)
        nc = rec.get("neuron_counts")
        if nc:
            ncount = torch.zeros(B, N, device=dev, dtype=torch.int64)
        raster = torch.zeros(T * n, B, N, device=dev, dtype=torch.bool) if rec.get("raster") else None
        tr = {"u": [], "I1": [], "I2": [], "I3": [], "syn": []} if trace else None
        if open_loop is not None:
            open_loop = torch.as_tensor(open_loop, device=dev)
        if own_spikes is not None:
            own_spikes = torch.as_tensor(own_spikes, device=dev).bool()
            flips = torch.zeros((), device=dev, dtype=torch.int64)
        zero_in = torch.zeros(B, N, device=dev, dtype=dt)
        k = 0
        for t in range(T):
            step_spk = torch.zeros(B, N, device=dev, dtype=dt) if (win or nc) else None
            if t < T_in:
                in_push = x[:, t, :].to(dt) @ self.Win
            else:
                in_push = zero_in
            for j in range(n):
                blocked = ref > 0
                if ours:
                    # 1. synapses
                    if self.static:
                        pE = pI = None
                        for (dl, cls), W in self.groups.items():
                            if k - dl < 0:
                                continue
                            p = hist[(k - dl) % self.L] @ W
                            if cls == 0:
                                pE = p if pE is None else pE + p
                            else:
                                pI = p if pI is None else pI + p
                        pE = zero_in if pE is None else pE
                        pI = zero_in if pI is None else pI
                    else:
                        R = 1.0 - (1.0 - R) * self.eD
                        q = q * self.eF
                        a = self._arrivals(hist, k, B)
                        q = q + a * self.U * (1.0 - q)
                        push = a * self.wg * q * R
                        R = R - a * q * R
                        pE = push[:, :self.nE] @ self.M_E if self.nE else zero_in
                        pI = push[:, self.nE:] @ self.M_I if self.S > self.nE else zero_in
                    ip = in_push if j == 0 else zero_in
                    # 2. currents
                    if fine:
                        IE = self.aE * IE + (1 - self.aE) * pE
                        II = self.aI * II + (1 - self.aI) * pI
                        Iin = self.in_a * Iin + (1 - self.in_a) * ip
                        total = IE + II + Iin
                    else:
                        total = pE + pI + ip
                    # 3. integrate
                    u_new = self.beta * u + total + self.tap
                    if liq.clamp:
                        u_new = torch.where(blocked, u, u_new)
                    # 4. threshold and reset
                    thr = None
                    if liq.snn_quirk:       # snnTorch: the spike test sees u_new - [u_prev > 1]
                        crossed = (u_new - (u > 1.0).to(dt)) > 1.0
                    elif self.var_thr:
                        thr = self._threshold(ab)
                        crossed = (u_new > thr) if liq.strict else (u_new >= thr)
                    else:
                        crossed = (u_new > 1.0) if liq.strict else (u_new >= 1.0)
                    spk = crossed & ~blocked
                    if own_spikes is not None:
                        flips += (spk != own_spikes[k]).sum()
                        spk = own_spikes[k]
                    if liq.reset_mode == "subtract":
                        u = u_new - (spk.to(dt) * thr if (liq.reset_to_thr and thr is not None) else spk.to(dt))
                    else:
                        u = torch.where(spk, torch.full_like(u_new, liq.reset_value), u_new)
                    if self.adapt:
                        ab = self.a_rho * ab + self.a_in * spk.to(dt)
                else:
                    # Brian2 schedule of (d)'s network (its sorted objects): synapse state update (exact), the
                    # summed current from the updated y, exact neuron update with that current held constant over
                    # the step and the input current decaying exactly; threshold; arrivals (on_pre); reset.
                    q = q * self.eF
                    R, y = R * self.eD + (1 - self.eD) + y * self.cxy, y * self.ey
                    # scatter-add per target, as Brian2's summed variable does (a runaway synapse of (d) as written
                    # then only affects its own target, instead of spreading NaN through a matrix product)
                    Jsyn = torch.zeros_like(zero_in).index_add_(1, self.post_idx, y * self.wg) if self.S else zero_in
                    u_new = self.beta * u + (1 - self.beta) * (self.J_b + Jsyn) + self.kappa * Iin
                    u_new = torch.where(blocked, u, u_new)
                    Iin = Iin * self.in_a
                    spk = (u_new > 1.0) & ~blocked
                    if own_spikes is not None:
                        flips += (spk != own_spikes[k]).sum()
                        spk = own_spikes[k]
                    if j == 0 and t < T_in:
                        Iin = Iin + in_push
                    hist[k % self.L] = (open_loop[k] if open_loop is not None else spk).to(dt)
                    a = self._arrivals(hist, k, B)
                    q = q + a * self.U * (1.0 - q)
                    y = y + a * q * R
                    R = R - a * q * R
                    u = torch.where(spk, torch.full_like(u_new, liq.reset_value), u_new)
                ref = torch.where(spk, self.gap - 1, torch.clamp(ref - 1, min=0))
                if ours:
                    hist[k % self.L] = (open_loop[k] if open_loop is not None else spk).to(dt)
                spf = spk.to(dt)
                z = z * self.z_decay + spf
                if step_spk is not None:
                    step_spk += spf
                if raster is not None:
                    raster[k] = spk
                if trace:
                    tr["u"].append(u.detach().cpu().numpy().copy())
                    if ours:
                        tr["I1"].append((IE if fine else pE).cpu().numpy().copy())
                        tr["I2"].append((II if fine else pI).cpu().numpy().copy())
                        tr["I3"].append((Iin if fine else ip).cpu().numpy().copy())
                    else:
                        tr["I1"].append(Jsyn.cpu().numpy().copy()); tr["I3"].append(Iin.cpu().numpy().copy())
                k += 1
            # end of input step t
            if win and t < wl * nw:
                counts[:, t // wl] += step_spk.to(torch.int32)
            if nc and nc[0] <= t < nc[1]:
                ncount += step_spk.to(torch.int64)
            if t in st_at_set:
                states[:, st_at_set[t]] = z.float()
            if se and se[0] <= t < se[1]:
                states_every[:, t - se[0]] = z.float()
            if trace and (not self.static or not ours):
                syn = {"R": R.cpu().numpy().copy(), "u": q.cpu().numpy().copy()}
                if not ours:
                    syn["y"] = y.cpu().numpy().copy()
                tr["syn"].append(syn)
        if win:
            out["windows"] = counts
        if st_at:
            out["states_at"] = states
            out["states_at_times"] = st_at
        if se:
            out["states_every"] = states_every
        if nc:
            out["neuron_counts"] = ncount
        if raster is not None:
            out["raster"] = raster
        if trace:
            out["trace"] = tr
        out["u_end"] = u
        if own_spikes is not None:
            out["flips"] = int(flips)
        return out


    # ------------------------------------------------------------------ fast path ("ours" schedule only)
    def _run_fast(self, x, u0, T, rec, graphs, state=None, t0=0):
        """The same arithmetic as run()'s "ours" branch, in the same order, on fixed buffers: one input step is one
        call of `step` (or one replay of its CUDA graph). The spike-history ring holds m * n sub-steps (m = ceil(L / n))
        so the slots an input step reads depend only on t mod m; slots not yet written hold zeros, which is what the
        plain loop's skipped early arrivals add."""
        liq, dev, dt = self.liq, self.device, self.dtype
        B, T_in, Cin = x.shape
        N, n = self.N, liq.n_sub
        m = -(-self.L // n)
        Lr = m * n
        fine = liq.cur_a is not None
        if t0 % m:
            raise ValueError(f"t0 = {t0} must be a multiple of {m} (the spike-history ring's period in input steps)")
        u_start = torch.zeros(B, N, device=dev, dtype=dt) if state is not None else self.start_state(u0)
        st = {"u": u_start, "ref": torch.zeros(B, N, device=dev, dtype=torch.int64),
              "hist": torch.zeros(Lr, B, N, device=dev, dtype=dt), "z": torch.zeros(B, N, device=dev, dtype=dt),
              "step": torch.zeros(B, N, device=dev, dtype=dt), "inp": torch.zeros(B, N, device=dev, dtype=dt)}
        if self.adapt:
            st["ab"] = torch.zeros(B, N, device=dev, dtype=dt)
        if fine:
            st["IE"] = torch.zeros(B, N, device=dev, dtype=dt)
            st["II"] = torch.zeros(B, N, device=dev, dtype=dt)
        if self.in_a is not None:
            st["Iin"] = torch.zeros(B, N, device=dev, dtype=dt)
        if not self.static:
            st["R"] = torch.ones(B, self.S, device=dev, dtype=dt)
            st["q"] = torch.zeros(B, self.S, device=dev, dtype=dt)
        if state is not None:                                  # continue a stream where the previous call stopped
            if set(state) != set(st) - {"step", "inp"}:
                raise ValueError(f"state keys {sorted(state)} do not match this liquid's {sorted(set(st) - {'step', 'inp'})}")
            for k, v in state.items():
                st[k].copy_(v)
        zero_in = torch.zeros(B, N, device=dev, dtype=dt)

        if not self.static:
            # arrivals as gathers (capture-safe, no index_put): each delay group's presynaptic columns, then the
            # synapses back in their own order (synapses are sorted by type, so this is usually already in order)
            gpre = [(dl, self.pre_idx[idx]) for dl, idx in self.delay_groups]
            order = torch.cat([idx for _, idx in self.delay_groups])
            inv = None if torch.equal(order, torch.arange(self.S, device=dev)) else torch.argsort(order)

        def arrivals(hist, kk):
            a = torch.cat([hist[(kk - dl) % Lr].index_select(1, pre) for dl, pre in gpre], 1)
            return a if inv is None else a.index_select(1, inv)

        def step(phase):
            u, ref, hist, z = st["u"], st["ref"], st["hist"], st["z"]
            IE, II, Iin = st.get("IE"), st.get("II"), st.get("Iin")
            R, q = st.get("R"), st.get("q")
            ab = st.get("ab")
            step_spk = st["step"]
            step_spk.zero_()
            in_push = st["inp"]
            for j in range(n):
                kk = phase * n + j
                blocked = ref > 0
                if self.static:
                    pE = pI = None
                    for (dl, cls), W in self.groups.items():
                        p = hist[(kk - dl) % Lr] @ W
                        if cls == 0:
                            pE = p if pE is None else pE + p
                        else:
                            pI = p if pI is None else pI + p
                    pE = zero_in if pE is None else pE
                    pI = zero_in if pI is None else pI
                else:
                    R = 1.0 - (1.0 - R) * self.eD
                    q = q * self.eF
                    a = arrivals(hist, kk)
                    q = q + a * self.U * (1.0 - q)
                    push = a * self.wg * q * R
                    R = R - a * q * R
                    pE = push[:, :self.nE] @ self.M_E if self.nE else zero_in
                    pI = push[:, self.nE:] @ self.M_I if self.S > self.nE else zero_in
                ip = in_push if j == 0 else zero_in
                if fine:
                    IE = self.aE * IE + (1 - self.aE) * pE
                    II = self.aI * II + (1 - self.aI) * pI
                    Iin = self.in_a * Iin + (1 - self.in_a) * ip
                    total = IE + II + Iin
                else:
                    total = pE + pI + ip
                u_new = self.beta * u + total + self.tap
                if liq.clamp:
                    u_new = torch.where(blocked, u, u_new)
                thr = None
                if liq.snn_quirk:
                    crossed = (u_new - (u > 1.0).to(dt)) > 1.0
                elif self.var_thr:
                    thr = self._threshold(ab)
                    crossed = (u_new > thr) if liq.strict else (u_new >= thr)
                else:
                    crossed = (u_new > 1.0) if liq.strict else (u_new >= 1.0)
                spk = crossed & ~blocked
                if liq.reset_mode == "subtract":
                    u = u_new - (spk.to(dt) * thr if (liq.reset_to_thr and thr is not None) else spk.to(dt))
                else:
                    u = torch.where(spk, torch.full_like(u_new, liq.reset_value), u_new)
                if self.adapt:
                    ab = self.a_rho * ab + self.a_in * spk.to(dt)
                ref = torch.where(spk, self.gap - 1, torch.clamp(ref - 1, min=0))
                hist[kk % Lr] = spk.to(dt)
                spf = spk.to(dt)
                z = z * self.z_decay + spf
                step_spk += spf
            st["u"].copy_(u); st["ref"].copy_(ref); st["z"].copy_(z)
            if fine:
                st["IE"].copy_(IE); st["II"].copy_(II)
            if Iin is not None:
                st["Iin"].copy_(Iin)
            if R is not None:
                st["R"].copy_(R); st["q"].copy_(q)
            if ab is not None:
                st["ab"].copy_(ab)

        run_step = {ph: (lambda ph=ph: step(ph)) for ph in range(m)}
        if graphs:
            backup = {k: v.clone() for k, v in st.items()}
            side = torch.cuda.Stream(device=dev)
            side.wait_stream(torch.cuda.current_stream(dev))
            with torch.cuda.stream(side):
                for ph in range(m):
                    step(ph)                                   # warm-up (cuBLAS handles, allocator)
            torch.cuda.current_stream(dev).wait_stream(side)
            pool = None
            for ph in range(m):
                g = torch.cuda.CUDAGraph()
                with torch.cuda.graph(g, pool=pool):
                    step(ph)
                pool = g.pool()
                run_step[ph] = g.replay
            for k, v in st.items():                            # undo the warm-up's changes to the state
                v.copy_(backup[k])
            del backup

        out = {}
        win = rec.get("windows")
        if win:
            wl, nw = win
            counts = torch.zeros(B, nw, N, device=dev, dtype=torch.int32)
        st_at = sorted(set(rec.get("states_at", [])))
        st_at_set = {t: i for i, t in enumerate(st_at)}
        if st_at:
            states = torch.zeros(B, len(st_at), N, device=dev, dtype=torch.float32)
        se = rec.get("states_every")
        if se:
            states_every = torch.zeros(B, se[1] - se[0], N, device=dev, dtype=torch.float32)
        nc = rec.get("neuron_counts")
        if nc:
            ncount = torch.zeros(B, N, device=dev, dtype=torch.int64)
        raster = torch.zeros(T * n, B, N, device=dev, dtype=torch.bool) if rec.get("raster") else None
        if raster is not None:
            raise ValueError("the fast path does not record rasters: use fast=False")
        sc = torch.zeros(B, T, N, device=dev, dtype=torch.uint8) if rec.get("step_counts") else None
        abw = rec.get("ab_weight")
        if abw is not None:
            if not self.adapt:
                raise ValueError("ab_weight needs an adaptive liquid")
            abw = torch.as_tensor(abw, device=dev, dtype=torch.float64)
            ab_sum = torch.zeros(B, N, device=dev, dtype=torch.float64)
        for t in range(T):
            if t < T_in:
                st["inp"].copy_(x[:, t, :].to(dt) @ self.Win)
            else:
                st["inp"].zero_()
            run_step[(t0 + t) % m]()
            step_spk, z = st["step"], st["z"]
            if sc is not None:
                sc[:, t] = step_spk.to(torch.uint8)
            if abw is not None:
                ab_sum += abw[:, t, None] * st["ab"]
            if win and t < wl * nw:
                counts[:, t // wl] += step_spk.to(torch.int32)
            if nc and nc[0] <= t < nc[1]:
                ncount += step_spk.to(torch.int64)
            if t in st_at_set:
                states[:, st_at_set[t]] = z.float()
            if se and se[0] <= t < se[1]:
                states_every[:, t - se[0]] = z.float()
        if win:
            out["windows"] = counts
        if st_at:
            out["states_at"] = states
            out["states_at_times"] = st_at
        if se:
            out["states_every"] = states_every
        if nc:
            out["neuron_counts"] = ncount
        if sc is not None:
            out["step_counts"] = sc
        if abw is not None:
            out["ab_sum"] = ab_sum
        if rec.get("keep_state"):
            out["state"] = {k: v.clone() for k, v in st.items() if k not in ("step", "inp")}
        out["u_end"] = st["u"].clone()
        return out


# ---------------------------------------------------------------- starting states (CPU, from (seed, beat index))
def start_voltages(liq, seed, beat_ids, stream, random_start=None):
    """Starting voltages for each beat, drawn on the CPU from (seed, stream, beat index) so they do not depend
    on batching or the card. random_start=None: the liquid's own rule (0 in X; uniform 0.9..1 with the tap).
    random_start=True: uniform between the resting level and the threshold (level 3's two starting states)."""
    lo, hi = (liq.rest, 1.0) if random_start else (liq.start_lo, liq.start_hi)
    out = np.empty((len(beat_ids), liq.N))
    for i, b in enumerate(beat_ids):
        if hi > lo:
            out[i] = np.random.default_rng([int(seed), int(stream), int(b)]).uniform(lo, hi, liq.N)
        else:
            out[i] = lo
    return out
'''
LSM2_FILES['lsm2/stats.py'] = r'''"""Statistics and decision rules (plan, Statistics and decision rules): the paired Nadeau-Bengio corrected
t-test over the 12 CV splits (variance factor 1/12 + 0.336), Holm within a family, the DS2 difference paired by
seed, and the verdicts meaningful / comparable / inconclusive. Inputs are tables of runs with columns spec_split,
spec_seed, macro_f1 (and DS2 metrics)."""
import math
import numpy as np
from scipy import stats as sstats

NB_RATIO = 0.336          # plan: test/train ratio of the 3x4 patient-grouped CV (the replication used 0.25/0.75)


def per_split_seed(d, metric="macro_f1"):
    """The metric (Macro-F1 by default) by (split, seed)."""
    return d.groupby(["spec_split", "spec_seed"], dropna=False)[metric].mean()

def compare(a, b, require_seed_sign=False, metric="macro_f1"):
    """A vs B on `metric` (Macro-F1 by default): seeds averaged per CV split; paired Nadeau-Bengio corrected t-test
    over the 12 splits with its 95% interval; the DS2 difference paired by seed."""
    if a is None or b is None or a.empty or b.empty:
        return None
    A, B = per_split_seed(a, metric), per_split_seed(b, metric)
    if a["spec_seed"].isna().all() or b["spec_seed"].isna().all():      # a seed-free control pairs with every seed
        A = A.groupby(level=0).mean(); B = B.groupby(level=0).mean()
        Asp, Bsp = A, B
        ds2_seed = None
    else:
        Asp, Bsp = A.groupby(level=0).mean(), B.groupby(level=0).mean()
        ds2_seed = (A.xs("final", level=0) - B.xs("final", level=0)).dropna() if "final" in A.index.get_level_values(0) and "final" in B.index.get_level_values(0) else None
    cv = [s for s in Asp.index if str(s).startswith("cv-") and s in Bsp.index]
    if len(cv) < 3:
        return None
    d = (Asp[cv] - Bsp[cv]).values
    k = len(d)
    var = np.var(d, ddof=1) * (1 / k + NB_RATIO)
    se = math.sqrt(var)
    t = d.mean() / se if se > 0 else np.inf
    p = 2 * sstats.t.sf(abs(t), k - 1)
    q = sstats.t.ppf(0.975, k - 1)
    ds2 = float(Asp.get("final", np.nan) - Bsp.get("final", np.nan)) if ("final" in Asp.index and "final" in Bsp.index) else np.nan
    out = {"n_splits": k, "cv_diff": float(d.mean()), "ci_low": float(d.mean() - q * se), "ci_high": float(d.mean() + q * se),
           "t": float(t), "p": float(p), "ds2_diff": ds2,
           "ds2_diff_by_seed": None if ds2_seed is None else {str(s): float(v) for s, v in ds2_seed.items()}}
    # DS2 without record 232, and per-class SVEB / VEB recall and precision (plan: every comparison)
    fa, fb = a[a.spec_split == "final"], b[b.spec_split == "final"]
    for m in ("macro_f1_no232", "recall_SVEB", "precision_SVEB", "recall_VEB", "precision_VEB"):
        if m in fa and m in fb and len(fa) and len(fb):
            out[f"ds2_diff_{m}"] = float(fa[m].mean() - fb[m].mean())
    if require_seed_sign and not a["spec_seed"].isna().all():
        per_seed = []
        for sd in sorted(set(a.spec_seed.dropna()) & set(b.spec_seed.dropna())):
            xa = a[(a.spec_seed == sd) & a.spec_split.astype(str).str.startswith("cv-")].groupby("spec_split")[metric].mean()
            xb = b[(b.spec_seed == sd) & b.spec_split.astype(str).str.startswith("cv-")].groupby("spec_split")[metric].mean()
            j = xa.index.intersection(xb.index)
            per_seed.append(float((xa[j] - xb[j]).mean()))
        out["cv_diff_by_seed"] = per_seed
        out["same_sign_all_seeds"] = bool(per_seed and (all(x > 0 for x in per_seed) or all(x < 0 for x in per_seed)))
    return out

def compare_diff(a_on, a_off, b_on, b_off):
    """F7: (on - off, full) - (on - off, ablated), per split (seeds averaged), corrected t-test."""
    parts = [per_split_seed(x).groupby(level=0).mean() if x is not None and not x.empty else None for x in (a_on, a_off, b_on, b_off)]
    if any(p is None for p in parts):
        return None
    D = (parts[0] - parts[1]) - (parts[2] - parts[3])
    cv = [s for s in D.index if str(s).startswith("cv-") and np.isfinite(D[s])]
    if len(cv) < 3:
        return None
    d = D[cv].values; k = len(d)
    se = math.sqrt(np.var(d, ddof=1) * (1 / k + NB_RATIO)); t = d.mean() / se if se > 0 else np.inf
    q = sstats.t.ppf(0.975, k - 1)
    return {"n_splits": k, "cv_diff": float(d.mean()), "ci_low": float(d.mean() - q * se), "ci_high": float(d.mean() + q * se),
            "t": float(t), "p": float(2 * sstats.t.sf(abs(t), k - 1)), "ds2_diff": float(D.get("final", np.nan)),
            "plain_full_minus_ablated_on": float((parts[0] - parts[2])[cv].mean())}

def holm(rows):
    ps = [r["p"] if r else np.nan for r in rows]
    idx = [i for i, p in enumerate(ps) if np.isfinite(p)]
    order = sorted(idx, key=lambda i: ps[i]); m = len(order); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (m - rank) * ps[i]))
        rows[i]["p_holm"] = run
    return rows

def verdict(r, need_seed_sign=False):
    if r is None:
        return "missing"
    meaningful = abs(r["cv_diff"]) >= 0.01 and r.get("p_holm", 1) < 0.05 and np.sign(r["ds2_diff"]) == np.sign(r["cv_diff"])
    if need_seed_sign:
        meaningful = meaningful and r.get("same_sign_all_seeds", False)
    if meaningful:
        return "meaningful"
    if r["ci_low"] >= -0.02 and r["ci_high"] <= 0.02 and abs(r["ds2_diff"]) < 0.02:
        return "comparable"
    return "inconclusive"

def family(name, items, need_seed_sign=False):
    rows = [it[1] for it in items]
    rows = holm([dict(r) if r else None for r in rows])
    out = []
    for (label, _), r in zip(items, rows):
        out.append({"family": name, "comparison": label, **({} if r is None else {k: v for k, v in r.items() if not isinstance(v, (dict, list))}),
                    "verdict": verdict(r, need_seed_sign)})
    return out

'''
LSM2_FILES['lsm2/measure.py'] = r'''"""Label-free measurements: batching beats through the simulator, firing, the rank scores, the starting-state
rule, twin runs (level 3) and the memory test (level 4)."""
import math
import numpy as np
import torch

from . import config as C
from .sim import Simulator, start_voltages


class Ctx:
    """Where and how to simulate: device, precision and a fixed batch size (plan, determinism)."""
    def __init__(self, device="cpu", dtype=torch.float32, batch=C.CODE["sim_batch"], log=print):
        self.device, self.dtype, self.batch, self.log = device, dtype, batch, log


def simulate(ctx, liq, g, in_scale, spikes, u0, T, rec, recurrence=True):
    """spikes: (n, T_in, C) numpy input spikes; u0: (n, N). Returns the simulator outputs as numpy, batched."""
    sm = Simulator(liq, device=ctx.device, dtype=ctx.dtype, g=g, in_scale=in_scale, recurrence=recurrence)
    parts = []
    for s in range(0, len(spikes), ctx.batch):
        o = sm.run(spikes[s:s + ctx.batch], u0[s:s + ctx.batch], T, rec=rec)
        parts.append({k: v.cpu().numpy() if torch.is_tensor(v) else v for k, v in o.items() if k != "u_end"})
    out = {}
    for k in parts[0]:
        if k == "states_at_times":
            out[k] = parts[0][k]
        elif k == "raster":
            out[k] = np.concatenate([p[k] for p in parts], 1)
        else:
            out[k] = np.concatenate([p[k] for p in parts], 0)
    return out


# ---------------------------------------------------------------- rank scores
def effective_rank(M):
    """Roy & Vetterli 2007: exp of the entropy of the normalised singular values, of the centred matrix
    (float64). 0 for an all-zero matrix."""
    M = np.asarray(M, np.float64)
    M = M - M.mean(0, keepdims=True)
    s = np.linalg.svd(M, compute_uv=False)
    tot = s.sum()
    if tot <= 0:
        return 0.0
    p = s / tot
    p = p[p > 0]
    return float(np.exp(-(p * np.log(p)).sum()))


def plain_rank(M):
    """(e)'s plain rank (reported only)."""
    M = np.asarray(M, np.float64)
    return int(np.linalg.matrix_rank(M)) if M.any() else 0


def cutoff_rank(M, frac=0.01):
    """An earlier draft's measure: singular values above 1% of the largest, uncentred (reported only)."""
    s = np.linalg.svd(np.asarray(M, np.float64), compute_uv=False)
    return int((s > frac * s[0]).sum()) if len(s) and s[0] > 0 else 0


# ---------------------------------------------------------------- input sets
def jitter_spikes(spikes, rng, sd_ms=C.JITTER_MS, step_ms=C.SAMPLE_MS):
    """Each encoded spike moved by a Gaussian amount with SD 4 ms, rounded to the nearest sample; spikes moved
    outside the beat are dropped (plan, Generalisation score). spikes: (T, C) counts."""
    T, Cn = spikes.shape
    out = np.zeros((T, Cn), np.uint8)
    tt, cc = np.nonzero(spikes)
    reps = spikes[tt, cc].astype(int)
    tt, cc = np.repeat(tt, reps), np.repeat(cc, reps)
    nt = tt + np.rint(rng.normal(0, sd_ms / step_ms, len(tt))).astype(int)
    ok = (nt >= 0) & (nt < T)
    np.add.at(out, (nt[ok], cc[ok]), 1)
    return out


def twin_spike(spikes, rng, t=C.R_SAMPLE):
    """The same input with one extra spike at sample t (the R peak), on a channel silent there."""
    s = spikes.copy()
    free = np.nonzero(s[t] == 0)[0]
    ch = rng.choice(free) if len(free) else rng.integers(s.shape[1])
    s[t, ch] = min(255, int(s[t, ch]) + 1)
    return s


def level3_sets(sep_spikes, gen_base, seed_sets=0, n_twin=None):
    """sep_spikes: (500, T, C) spikes of 500 different inputs; gen_base: (4, T, C) the 4 inputs that are jittered.
    Returns the separation, generalisation and twin input sets (fixed for every candidate and g)."""
    n_twin = C.CODE["twin_beats"] if n_twin is None else n_twin
    gen = np.stack([jitter_spikes(gen_base[i], np.random.default_rng([seed_sets, 31, i, c]))
                    for i in range(len(gen_base)) for c in range(C.GEN_COPIES)])
    twin = np.stack([twin_spike(sep_spikes[i], np.random.default_rng([seed_sets, 37, i])) for i in range(min(n_twin, len(sep_spikes)))])
    return {"sep": sep_spikes, "gen": gen, "twin": twin}


# ---------------------------------------------------------------- level 3 at one g
def firing_stats(neuron_counts, n_samples, step_ms=C.SAMPLE_MS):
    """Mean rate over all neurons (silent ones included) and the share of neurons above 100 Hz.
    neuron_counts: (beats, N) spike counts over n_samples input steps."""
    dur_s = n_samples * step_ms / 1000.0
    per_neuron = neuron_counts.mean(0) / dur_s
    return float(per_neuron.mean()), float((per_neuron > C.HIGH_HZ).mean())


def measure_at_g(ctx, liq, g, in_scale, sets, seed, T=C.BEAT_SAMPLES, read=C.STATE_READ_SAMPLE,
                 with_start_test=False, with_twins=True, extra_ranks=False):
    """Every level-3 measure at one g for one liquid. Starting voltages: uniform between the resting level and
    the threshold for every candidate, drawn from (seed, stream, input index)."""
    nb = len(sets["sep"])
    u_sep = start_voltages(liq, seed, range(nb), 1, random_start=True)
    rec = {"states_at": [read, C.START_TEST_SAMPLE], "neuron_counts": (0, T)}
    o = simulate(ctx, liq, g, in_scale, sets["sep"], u_sep, T, rec)
    rate, high = firing_stats(o["neuron_counts"], T, liq.h_ms * liq.n_sub)
    sep_states = o["states_at"][:, 0]
    u_gen = start_voltages(liq, seed, range(len(sets["gen"])), 2, random_start=True)
    og = simulate(ctx, liq, g, in_scale, sets["gen"], u_gen, T, {"states_at": [read]})
    gen_states = og["states_at"][:, 0]
    res = {"g": float(g), "abs_g": liq.ee_mean_push(g), "rate_hz": rate, "frac_above_100hz": high,
           "in_band": bool(C.BAND_HZ[0] <= rate <= C.BAND_HZ[1] and high <= C.HIGH_FRAC),
           "sep": effective_rank(sep_states), "gen": effective_rank(gen_states)}
    res["gap"] = res["sep"] - res["gen"]
    if extra_ranks:
        res.update({"sep_plain": plain_rank(sep_states), "gen_plain": plain_rank(gen_states),
                    "sep_cut1": cutoff_rank(sep_states), "gen_cut1": cutoff_rank(gen_states)})
        res["gap_plain"], res["gap_cut1"] = res["sep_plain"] - res["gen_plain"], res["sep_cut1"] - res["gen_cut1"]
    if with_twins and len(sets["twin"]):
        nt = len(sets["twin"])
        u_t = u_sep[:nt]
        rt = {"states_every": (0, T)}
        a = simulate(ctx, liq, g, in_scale, sets["sep"][:nt], u_t, T, rt)["states_every"]
        b = simulate(ctx, liq, g, in_scale, sets["twin"], u_t, T, rt)["states_every"]
        dist = np.sqrt(((a - b) ** 2).sum(-1)).mean(0)          # mean over beats of the state distance, per sample
        e0, e1 = C.CODE["twin_early"]; l0, l1 = C.CODE["twin_late"]
        early, late = dist[e0:e1 + 1].mean(), dist[l0:l1 + 1].mean()
        res["twin_early"], res["twin_late"] = float(early), float(late)
        res["twin_ratio"] = float(late / early) if early > 0 else float("nan")
    if with_start_test:
        u2 = start_voltages(liq, seed, range(nb), 3, random_start=True)
        o2 = simulate(ctx, liq, g, in_scale, sets["sep"], u2, T, {"states_at": [C.START_TEST_SAMPLE]})
        s1, s2 = o["states_at"][:, 1].astype(np.float64), o2["states_at"][:, 0].astype(np.float64)
        same = float(np.sqrt(((s1 - s2) ** 2).sum(1)).mean())
        sq = (s1 ** 2).sum(1)
        D2 = np.maximum(sq[:, None] + sq[None, :] - 2 * s1 @ s1.T, 0)
        iu = np.triu_indices(len(s1), 1)
        diff = float(np.sqrt(D2[iu]).mean())
        res.update({"start_same": same, "start_diff": diff,
                    "start_rule_pass": bool(same < diff or (same == 0 and diff == 0))})
    return res


# ---------------------------------------------------------------- level 4: memory
def surrogate_walks(steps_pool, n_seq, T, seed):
    """Random walks on [0, 1], reflected at the edges, step sizes drawn from ECG's own sample-to-sample steps
    with random signs (plan, level 4). Starting levels uniform on [0, 1]."""
    rng = np.random.default_rng([seed, 41])
    X = np.empty((n_seq, T))
    for i in range(n_seq):
        st = rng.choice(steps_pool, T - 1) * rng.choice([-1.0, 1.0], T - 1)
        v = rng.uniform(0, 1)
        X[i, 0] = v
        for t in range(1, T):
            v = v + st[t - 1]
            while v < 0 or v > 1:
                v = -v if v < 0 else 2 - v
            X[i, t] = v
    return X


class _Stats:
    """Sufficient statistics of (x, y) rows for ridge with an intercept: n, sum x, sum y, sum y^2, X'X, X'y."""
    def __init__(self, N, dev):
        z = lambda *sh: torch.zeros(*sh, dtype=torch.float64, device=dev)
        self.n, self.sx, self.sy, self.syy, self.G, self.b = 0, z(N), z(()), z(()), z(N, N), z(N)

    def add(self, X, y):
        self.n += X.shape[0]; self.sx += X.sum(0); self.sy += y.sum(); self.syy += (y * y).sum()
        self.G += X.T @ X; self.b += X.T @ y

    def __add__(self, o):
        r = _Stats.__new__(_Stats)
        r.n = self.n + o.n
        for k in ("sx", "sy", "syy", "G", "b"):
            setattr(r, k, getattr(self, k) + getattr(o, k))
        return r


def _fit(st, alpha):
    mu, my = st.sx / st.n, st.sy / st.n
    Gc = st.G - st.n * torch.outer(mu, mu)
    bc = st.b - st.n * mu * my
    w = torch.linalg.solve(Gc + alpha * torch.eye(len(mu), dtype=Gc.dtype, device=Gc.device), bc)
    return w, mu, my


def _ss_res(st, w, mu, my):
    """sum over st's rows of (y - my - (x - mu)'w)^2, from its statistics."""
    c = my - mu @ w                                     # prediction = x'w + c
    return float(st.syy - 2 * (w @ st.b) - 2 * c * st.sy + w @ st.G @ w + 2 * c * (w @ st.sx) + st.n * c * c)


def _ss_tot(st, mean=None):
    m = st.sy / st.n if mean is None else mean
    return float(st.syy - 2 * m * st.sy + st.n * m * m)


def memory_curve(ctx, liq, g, in_scale, seq_spikes, target_counts, seed, T=C.BEAT_SAMPLES,
                 lags_ms=C.MEMORY_T_MS, warmup=C.MEMORY_WARMUP, recurrence=True, batch=100,
                 alphas=C.CODE["ridge_alphas"], k=C.CODE["ridge_inner_folds"]):
    """R^2 of reconstructing target[t - T] from the 30 ms-filtered state at t, for each lag: a ridge readout
    trained on the first half of the sequences (penalty by inner CV, folds grouped by sequence) and scored on
    the second half, at every sample after the warm-up. Statistics are accumulated batch by batch."""
    n = len(seq_spikes)
    step_ms = liq.h_ms * liq.n_sub
    half = n // 2
    fold = np.full(n, k)                                      # group k = test sequences
    fold[:half] = np.random.default_rng(0).permutation(half) % k
    lagL = {lag: int(round(lag / step_ms)) for lag in lags_ms}
    dev = ctx.device
    stats = {lag: [_Stats(liq.N, dev) for _ in range(k + 1)] for lag in lags_ms}
    sm = Simulator(liq, device=dev, dtype=ctx.dtype, g=g, in_scale=in_scale, recurrence=recurrence)
    u0 = start_voltages(liq, seed, range(n), 4)               # the liquid's own starting rule, as on ECG
    counts = 0.0
    for s0 in range(0, n, batch):
        o = sm.run(seq_spikes[s0:s0 + batch], u0[s0:s0 + batch], T, rec={"states_every": (0, T), "neuron_counts": (0, T)})
        Z = o["states_every"].to(torch.float64)               # (b, T, N)
        counts += float(o["neuron_counts"].sum())
        tg = torch.as_tensor(np.asarray(target_counts[s0:s0 + batch]), dtype=torch.float64, device=dev)
        for lag, L in lagL.items():
            t0 = max(warmup, L)
            for bi in range(Z.shape[0]):
                X = Z[bi, t0:T]
                y = tg[bi, t0 - L:T - L]
                stats[lag][fold[s0 + bi]].add(X, y)
        del o, Z
    out = {"rate_hz": counts / (n * liq.N * T * step_ms / 1000.0)}
    for lag in lags_ms:
        st = stats[lag]
        train_all = st[0]
        for f in range(1, k):
            train_all = train_all + st[f]
        best, best_a = -np.inf, alphas[0]
        for a in alphas:
            res = tot = 0.0
            for f in range(k):
                tr = None
                for f2 in range(k):
                    if f2 != f:
                        tr = st[f2] if tr is None else tr + st[f2]
                w, mu, my = _fit(tr, a)
                res += _ss_res(st[f], w, mu, my)
                tot += _ss_tot(st[f], float(train_all.sy / train_all.n))
            r2 = 1 - res / tot if tot > 0 else float("nan")
            if r2 > best:
                best, best_a = r2, a
        w, mu, my = _fit(train_all, best_a)
        tot = _ss_tot(st[k])
        out[f"r2_{lag}"] = float(1 - _ss_res(st[k], w, mu, my) / tot) if tot > 0 else float("nan")
        out[f"alpha_{lag}"] = float(best_a)
    return out
'''
LSM2_FILES['lsm2/gain.py'] = r'''"""Choosing the recurrent gain g (plan, Choosing the recurrent gain; decision 5).

Grid: g = 0 (control), 0.1 and 0.3, a coarse pass at 1, 3, ..., 1000 times the base weights, extensions in 3x
steps at an edge (at most 3), then a fine pass g* x 1.4^k, k = -3..3. Each setting is measured on all seeds and
averaged. Rule: among settings above 0 inside the firing band, the largest mean gap; a setting whose mean gap lies
within the range of the best setting's seed values is tied with it, and the smallest tied g is taken."""
import numpy as np

from . import config as C


def _agg(per_seed):
    keys = [k for k in per_seed[0] if isinstance(per_seed[0][k], (int, float, bool, np.floating, np.integer))]
    m = {k: float(np.nanmean([float(r[k]) for r in per_seed])) for k in keys}
    m["gap_seeds"] = [float(r["gap"]) for r in per_seed]
    m["rate_seeds"] = [float(r["rate_hz"]) for r in per_seed]
    m["in_band"] = bool(C.BAND_HZ[0] <= m["rate_hz"] <= C.BAND_HZ[1] and m["frac_above_100hz"] <= C.HIGH_FRAC)
    m["per_seed"] = per_seed
    return m


def choose(table):
    """table: {g: aggregated measures}. Returns (chosen g or None, best g or None)."""
    cand = [g for g in table if g > 0 and table[g]["in_band"]]
    if not cand:
        return None, None
    best = max(cand, key=lambda g: (table[g]["gap"], -g))
    lo, hi = min(table[best]["gap_seeds"]), max(table[best]["gap_seeds"])
    eps = 1e-12 * max(1.0, abs(lo), abs(hi))                 # the mean of equal seed values may round
    tied = [g for g in cand if lo - eps <= table[g]["gap"] <= hi + eps] + [best]
    return min(tied), best


def edge_status(table, g):
    """'upper' / 'lower' if g sits at that edge of the measured grid (g = 0 excluded), else None. A setting next
    to one outside the firing band counts as inside the grid (plan, Grid)."""
    gs = sorted(x for x in table if x > 0)
    i = gs.index(g)
    if i == len(gs) - 1:
        return "upper"
    if i == 0:
        return "lower"
    return None


def band_limited(table, g):
    gs = sorted(x for x in table if x > 0)
    i = gs.index(g)
    up = i + 1 < len(gs) and not table[gs[i + 1]]["in_band"]
    down = i > 0 and not table[gs[i - 1]]["in_band"]
    return bool(up or down)


def sweep(measure, seeds, log=print, coarse=None):
    """measure(g, seed) -> level-3 measures at g (measure.measure_at_g). Returns the sweep record."""
    table, order = {}, []

    def run(gs, stage):
        for g in gs:
            g = float(g)
            if g in table:
                continue
            table[g] = _agg([measure(g, s) for s in seeds])
            table[g]["stage"] = stage
            order.append(g)
            t = table[g]
            log(f"    g={g:<10.4g} abs={t['abs_g']:.4g} rate={t['rate_hz']:.2f} Hz in_band={t['in_band']} "
                f"sep={t['sep']:.2f} gen={t['gen']:.2f} gap={t['gap']:.2f}"
                + (f" twin={t['twin_ratio']:.2f}" if 'twin_ratio' in t else ""))

    run([0.0] + list(C.G_LOW) + list(coarse or C.G_COARSE), "coarse")
    chosen, best = choose(table)
    ext = 0
    while best is not None and ext < C.G_EXTEND_MAX:
        e = edge_status(table, best)
        if e is None:
            break
        gs = sorted(x for x in table if x > 0)
        nxt = gs[-1] * C.G_EXTEND_STEP if e == "upper" else gs[0] / C.G_EXTEND_STEP
        run([nxt], f"extension {ext + 1}")
        ext += 1
        chosen, best = choose(table)
    at_edge = best is not None and edge_status(table, best) is not None
    if best is not None and not at_edge:
        run([best * C.G_FINE_STEP ** k for k in range(-C.G_FINE_K, C.G_FINE_K + 1) if k != 0], "fine")
        chosen, best = choose(table)
        at_edge = edge_status(table, best) is not None
    rec = {"table": table, "order": order, "chosen_g": chosen, "best_g": best, "extensions": ext,
           "no_g_in_band": best is None, "peak_at_edge": bool(at_edge),
           "band_limited": bool(chosen is not None and band_limited(table, chosen))}
    if chosen is not None:
        gs = sorted(x for x in table if x > 0)
        i = gs.index(chosen)
        rec["next_higher_g"] = gs[i + 1] if i + 1 < len(gs) else None
        rec["next_higher_gap_lower"] = (rec["next_higher_g"] is not None
                                        and table[rec["next_higher_g"]]["gap"] < table[chosen]["gap"])
    rec["gap_peak_inside_grid"] = bool(best is not None and not at_edge)
    return rec
'''
LSM2_FILES['lsm2/calib.py'] = r'''"""Input size (plan, Input and calibration): the paper's rule, or with factor 5 a bisection on one factor s so the
input-receiving neurons fire, with recurrence off, at the rate Maass's own input-receiving neurons reach in the
Maass check (paper version). Factor 9 fixes Maass's own per-spike size instead."""
import math
import numpy as np

from . import config as C
from .liquid import paper_rule_scale
from .measure import simulate, firing_stats
from .sim import start_voltages


def ain_of(spikes):
    """Mean number of input spikes per sample, summed over channels (the authors' A_in)."""
    return float(np.asarray(spikes, np.float64).sum() / (spikes.shape[0] * spikes.shape[1]))


def input_neuron_rate(ctx, liqs, s, spikes, T=C.BEAT_SAMPLES):
    """Mean rate (Hz) of the input-receiving neurons, recurrence off, averaged over the liquids (seeds)."""
    rates = []
    for liq in liqs:
        u0 = start_voltages(liq, liq.meta.get("seed", 0), range(len(spikes)), 5)
        o = simulate(ctx, liq, 0.0, s, spikes, u0, T, {"neuron_counts": (0, T)}, recurrence=False)
        cnt = o["neuron_counts"][:, liq.in_neurons]
        rates.append(firing_stats(cnt, T, liq.h_ms * liq.n_sub)[0])
    return float(np.mean(rates))


def bisect_scale(ctx, liqs, spikes, target_hz, tol=C.CODE["calib_tol"], iters=C.CODE["calib_iters"], log=print):
    """Smallest bracket on s (log scale) until the rate is within tol of the target."""
    lo, hi = 0.0, 0.05
    r_hi = input_neuron_rate(ctx, liqs, hi, spikes)
    k = 0
    while r_hi < target_hz and k < 30:
        lo, hi = hi, hi * 2
        r_hi = input_neuron_rate(ctx, liqs, hi, spikes)
        k += 1
    if r_hi < target_hz:
        raise RuntimeError(f"input calibration: rate {r_hi:.2f} Hz < target {target_hz:.2f} Hz even at s = {hi}")
    best, best_r = hi, r_hi
    for i in range(iters):
        mid = math.sqrt(lo * hi) if lo > 0 else hi / 2
        r = input_neuron_rate(ctx, liqs, mid, spikes)
        if abs(r - target_hz) < abs(best_r - target_hz):
            best, best_r = mid, r
        if abs(r - target_hz) <= tol * target_hz:
            break
        if r < target_hz:
            lo = mid
        else:
            hi = mid
    log(f"    factor-5 input size s = {best:.5g} (rate {best_r:.3f} Hz, target {target_hz:.3f} Hz); Maass's own 0.12")
    return float(best), float(best_r)


def input_scale(ctx, liqs, spikes_cal, ain, maass_target_hz, log=print):
    """The candidate's input size: (scale, how it was set)."""
    liq = liqs[0]
    if liq.in_scale_fixed is not None:
        return float(liq.in_scale_fixed), {"rule": "fixed (factor 9 or Maass's own)", "scale": float(liq.in_scale_fixed)}
    if 5 in liq.factors:
        if maass_target_hz is None:
            raise RuntimeError("factor 5 needs the Maass check's input-neuron rate")
        s, r = bisect_scale(ctx, liqs, spikes_cal, maass_target_hz, log=log)
        return s, {"rule": "factor 5 calibration", "scale": s, "rate_hz": r, "target_hz": maass_target_hz}
    s = paper_rule_scale(ain, liq)
    fill_E = C.I_TARGET / (1 - float(liq.beta[liq.is_exc][0] ** liq.n_sub))
    fill_I = C.I_TARGET / (1 - float(liq.beta[~liq.is_exc][0] ** liq.n_sub))
    return s, {"rule": "paper's rule", "scale": s, "A_in": ain, "P_in": 1.0 if 6 in liq.factors else C.X["P_in"],
               "fill_E": fill_E, "fill_I": fill_I}
'''
LSM2_FILES['lsm2/data.py'] = r'''"""Label-free inputs for the search: the DS1 halves, the 500 calibration beats, the generalisation and twin sets,
the memory test's random surrogate, and Poisson trains for the Maass check."""
from dataclasses import dataclass
import numpy as np

from . import config as C
from .measure import level3_sets, surrogate_walks


def multithreshold(X, num_levels=200):
    """The authors' Multi-threshold encoder (detect_edges='both', include_endpoints=True): a spike on threshold j
    at sample t when x crosses it between t-1 and t. X: (n, T) in [0, 1] -> (n, T, K) uint8."""
    X = np.asarray(X)
    thr = np.linspace(0.0, 1.0, num_levels, dtype=X.dtype)
    above = X[:, :, None] >= thr
    out = np.zeros(above.shape, np.uint8)
    out[:, 1:] = (above[:, 1:] != above[:, :-1])
    return out


def split_records(records, seed=C.CODE["search_split_seed"]):
    """DS1's patients halved without labels: a fixed random permutation of the record list."""
    rec = sorted(records)
    perm = np.random.default_rng(seed).permutation(len(rec))
    half = len(rec) // 2
    return sorted(rec[i] for i in perm[:half]), sorted(rec[i] for i in perm[half:])


@dataclass
class SearchData:
    name: str
    sets: dict                 # sep (500, T, C), gen (500, T, C), twin (100, T, C)
    ain: float                 # input spikes per sample (A_in) on this half, labels unused
    mem_spikes: np.ndarray     # (500, T, C) surrogate encoded
    mem_targets: np.ndarray    # (500, T) input spike count per sample
    surrogate_ain: float
    real_spikes: np.ndarray = None    # secondary memory check on real beats
    real_targets: np.ndarray = None
    meta: dict = None


def make_search_data(name, beats, encode, seed, n_cal=None, n_mem=None):
    """beats: (n, 577) normalised config-11 beats of one half, in a fixed order (labels never used).
    encode: (m, 577) -> (m, 577, C) spikes."""
    n_cal = n_cal or C.N_CAL_BEATS
    n_mem = n_mem or C.CODE["memory_sequences"]
    rng = np.random.default_rng([seed, 51])
    pick = np.sort(rng.choice(len(beats), n_cal + C.GEN_BEATS, replace=False))
    perm = rng.permutation(pick)
    sep_idx, gen_idx = np.sort(perm[:n_cal]), np.sort(perm[n_cal:])
    sep = encode(beats[sep_idx])
    gen_base = encode(beats[gen_idx])
    sets = level3_sets(sep, gen_base, seed_sets=seed)
    # A_in over the whole half (the authors' quantity: input spikes per sample, summed over channels)
    tot, cnt = 0.0, 0
    for s in range(0, len(beats), 2000):
        e = encode(beats[s:s + 2000]); tot += float(e.sum(dtype=np.int64)); cnt += e.shape[0] * e.shape[1]
    ain = tot / cnt
    steps = np.abs(np.diff(beats, axis=1)).ravel()
    walks = surrogate_walks(steps, n_mem, beats.shape[1], seed)
    mem = encode(walks)
    targets = mem.sum(2).astype(np.int64)
    s_ain = float(mem.sum(dtype=np.int64) / (mem.shape[0] * mem.shape[1]))
    # secondary check: real beats, target = deviation from the R-aligned mean beat
    mean_beat = beats.mean(0)
    real_targets = beats[sep_idx] - mean_beat[None, :]
    return SearchData(name=name, sets=sets, ain=ain, mem_spikes=mem, mem_targets=targets, surrogate_ain=s_ain,
                      real_spikes=sep, real_targets=real_targets,
                      meta={"n_beats_half": int(len(beats)), "sep_idx": sep_idx.tolist(), "gen_idx": gen_idx.tolist(),
                            "surrogate_vs_ecg": s_ain / ain})


def poisson_trains(n, T, rate_hz, seed, stream, step_ms=C.SAMPLE_MS):
    """n Poisson trains of T samples, counts per sample (one channel)."""
    rng = np.random.default_rng([seed, stream])
    return rng.poisson(rate_hz * step_ms / 1000.0, (n, T, 1)).astype(np.uint8)


def make_maass_data(seed, T=C.BEAT_SAMPLES, n_cal=None, n_mem=None):
    """The Maass check's inputs: 20 Hz Poisson trains of beat length stand in for beats."""
    n_cal = n_cal or C.N_CAL_BEATS
    n_mem = n_mem or C.CODE["memory_sequences"]
    r = C.MAASS["input_rate_hz"]
    sep = poisson_trains(n_cal, T, r, seed, 61)
    gen_base = poisson_trains(C.GEN_BEATS, T, r, seed, 62)
    sets = level3_sets(sep, gen_base, seed_sets=seed)
    mem = poisson_trains(n_mem, T, r, seed, 63)
    return SearchData(name="maass_poisson", sets=sets, ain=float(sep.mean() * 1), mem_spikes=mem,
                      mem_targets=mem[:, :, 0].astype(np.int64), surrogate_ain=float(mem.mean()), meta={"rate_hz": r})
'''
LSM2_FILES['lsm2/stream.py'] = r'''"""Streaming whole records through a liquid (the continuous-liquid plan).

* stack(): several liquids side by side as one block-diagonal liquid (no synapse between them, the same input
  channels), so one simulator run serves them all. Each liquid's own recurrent gain and input size are folded into
  its weights (the stacked liquid runs at g = 1, input size 1); per-neuron settings are concatenated; the settings
  the simulator holds as scalars must agree.
* plan_streams(): the streams of one record: the whole record, or pieces that each start `warmup` samples early
  (from rest) and run `tail` samples past their own part. Every beat goes to the piece whose own part holds its
  window start.
* run_streams(): every stream side by side in segments of `seg_len` input steps, the state carried from segment to
  segment (Simulator.run with state / t0), each beat's `nwin` windows of `win` steps cut from the per-step spike
  counts; optionally the sum of the adaptation variable over each stream's own steps (for the frozen controls).
* frozen_thresholds() / frozen_copy(): the time-averaged threshold of every neuron (1 + c * mean b) and the liquid
  with its thresholds fixed there.
"""
from dataclasses import replace
import numpy as np
import torch

from .liquid import Liquid

_SAME = ("n_sub", "h_ms", "reset_mode", "reset_value", "start_lo", "start_hi", "rest", "clamp", "strict", "schedule",
         "syn_model", "snn_quirk", "in_split", "J_b")


def no_recurrence(liq):
    """The same liquid without its internal synapses (same neurons, input wiring and adaptation)."""
    e = np.zeros(0, np.int64)
    z = np.zeros(0)
    return replace(liq, pre=e, post=e.copy(), typ=e.copy(), w=z, U=z.copy(), D_ms=z.copy(), F_ms=z.copy(),
                   meta={**liq.meta, "recurrence": False})


def stack(liqs, g=None, in_scale=None):
    """One block-diagonal liquid from several. g / in_scale: per liquid (default 1 / the liquid's in_scale_fixed);
    they are multiplied into its synapse and input weights. Run the result with Simulator(g=1, in_scale=1)."""
    k = len(liqs)
    g = [1.0] * k if g is None else [float(v) for v in g]
    in_scale = [float(l.in_scale_fixed) for l in liqs] if in_scale is None else [float(v) for v in in_scale]
    L0 = liqs[0]
    for l in liqs[1:]:
        for f in _SAME:
            if getattr(l, f) != getattr(L0, f):
                raise ValueError(f"stack: liquids differ in {f}: {getattr(l, f)!r} vs {getattr(L0, f)!r}")
        if not np.array_equal(l.delay, L0.delay) or not np.array_equal(l.Ubar, L0.Ubar):
            raise ValueError("stack: liquids differ in delays or U-bar")
        if l.Win.shape[0] != L0.Win.shape[0]:
            raise ValueError("stack: liquids differ in input channels")
        if (l.cur_a is None) != (L0.cur_a is None) or (l.cur_a is not None and not np.array_equal(l.cur_a, L0.cur_a)):
            raise ValueError("stack: liquids differ in current decays")
        if (l.in_a is None) != (L0.in_a is None) or (l.kappa is None) != (L0.kappa is None):
            raise ValueError("stack: liquids differ in input-current settings")
    if L0.schedule != "ours":
        raise ValueError("stack: only the 'ours' schedule")
    # the threshold reset (subtract the current threshold) is the plain reset for a neuron whose threshold stays 1,
    # so liquids without it can join a stack that has it, if none of their thresholds move
    rtt = any(l.reset_to_thr for l in liqs)
    for l in liqs:
        if L0.reset_mode == "subtract" and not l.reset_to_thr and (l.thr_base is not None or (l.adapt_c is not None and np.any(np.asarray(l.adapt_c) > 0))):
            raise ValueError("stack: a liquid with moving or fixed thresholds but the plain reset")
    offs = np.cumsum([0] + [l.N for l in liqs])
    cat = lambda f: np.concatenate([getattr(l, f) for l in liqs])
    pre = np.concatenate([l.pre + o for l, o in zip(liqs, offs)]).astype(np.int64)
    post = np.concatenate([l.post + o for l, o in zip(liqs, offs)]).astype(np.int64)
    typ = cat("typ")
    w = np.concatenate([l.w * gi for l, gi in zip(liqs, g)])
    order = np.lexsort((post, pre, typ))               # by type, then pre, then post (E-presynaptic first), stable
    adapt = any(l.adapt_c is not None for l in liqs)
    thr = any(l.thr_base is not None for l in liqs)
    N = int(offs[-1])
    return Liquid(
        N=N, dims=(N, 1, 1), is_exc=cat("is_exc"), pos=np.concatenate([l.pos for l in liqs]), pre=pre[order],
        post=post[order], typ=typ[order], w=w[order], U=cat("U")[order], D_ms=cat("D_ms")[order],
        F_ms=cat("F_ms")[order], Ubar=L0.Ubar, Win=np.concatenate([l.Win * s for l, s in zip(liqs, in_scale)], 1),
        in_neurons=cat("in_neurons"), n_sub=L0.n_sub, h_ms=L0.h_ms, beta=cat("beta"), tap=cat("tap"),
        reset_mode=L0.reset_mode, reset_value=L0.reset_value, start_lo=L0.start_lo, start_hi=L0.start_hi,
        rest=L0.rest, gap=cat("gap"), clamp=L0.clamp, strict=L0.strict, schedule=L0.schedule, syn_model=L0.syn_model,
        delay=L0.delay, cur_a=L0.cur_a, in_a=None if L0.in_a is None else cat("in_a"), J_b=L0.J_b,
        kappa=None if L0.kappa is None else cat("kappa"), syn_tau_ms=L0.syn_tau_ms, factors=L0.factors,
        snn_quirk=L0.snn_quirk, g_maass=None, in_scale_fixed=1.0, in_split=L0.in_split,
        adapt_c=np.concatenate([l.adapt_c if l.adapt_c is not None else np.zeros(l.N) for l in liqs]) if adapt else None,
        adapt_tau_ms=np.concatenate([l.adapt_tau_ms if l.adapt_tau_ms is not None else np.full(l.N, np.inf)
                                     for l in liqs]) if adapt else None,
        thr_base=np.concatenate([l.thr_base if l.thr_base is not None else np.ones(l.N) for l in liqs]) if thr else None,
        reset_to_thr=rtt,
        meta={"kind": "stacked", "parts": [dict(l.meta) for l in liqs], "offsets": offs.tolist(), "g": g,
              "in_scale": in_scale},
    )


def plan_streams(rec_len, starts, pieces=1, warmup=0, tail=640):
    """The streams of one record of rec_len samples whose beats have window starts `starts` (record samples,
    ascending). pieces = 1: the whole record, from its first sample to `tail` samples past the last beat's window
    start (at least to the record's end). pieces > 1: equal own parts; each piece starts `warmup` samples before
    its own part (not before the record's start) and runs to `tail` samples past its own part.
    Returns dicts: offset (record sample of the stream's step 0), length (steps), beat_idx (into starts),
    starts (relative to the offset), own_from / own_to (the stream's own record samples, relative)."""
    starts = np.asarray(starts, np.int64)
    if len(starts) > 1 and np.any(np.diff(starts) < 0):
        raise ValueError("plan_streams: starts must be ascending")
    bounds = np.linspace(0, rec_len, pieces + 1).round().astype(np.int64)
    out = []
    for c0, c1 in zip(bounds[:-1], bounds[1:]):
        off = max(0, int(c0) - int(warmup))
        idx = np.nonzero((starts >= c0) & (starts < c1))[0]
        length = int(c1) + (int(tail) if pieces > 1 else 0) - off
        if len(idx):
            length = max(length, int(starts[idx].max()) + int(tail) - off)
        out.append({"offset": off, "length": length, "beat_idx": idx, "starts": starts[idx] - off,
                    "own_from": int(c0) - off, "own_to": int(c1) - off})
    return out


def period(sm):
    """Input steps after which the simulator's spike-history ring repeats (segment lengths must be multiples)."""
    return -(-sm.L // sm.liq.n_sub)


@torch.no_grad()
def run_streams(sm, enc, streams, seg_len, u0=None, nwin=32, win=20, ab=False, counts=False, sink=None,
                beat_chunk=None, progress=None):
    """All streams side by side. enc(a, b) -> (B, b - a, C) uint8 input spikes of steps [a, b) of every stream
    (zero past a stream's end). streams: dicts with length, starts (ascending), own_from, own_to. u0: (B, N)
    starting voltages (default 0). sink(i, ks, wc): receives the windows of stream i's beats ks (indices into its
    starts) as wc (len(ks), nwin, N) uint8; default: kept and returned as out["windows"][i].
    Returns {"windows" (no sink), "own_steps": (B,), "ab_sum": (B, N) the adaptation variable summed over each
    stream's own steps (ab=True; zeros for a liquid that does not adapt), "spike_sum": (B, N) spikes over the own steps (counts=True), "steps": T}."""
    dev, N, B = sm.device, sm.N, len(streams)
    span = nwin * win
    m = period(sm)
    if seg_len % m:
        raise ValueError(f"seg_len must be a multiple of {m}")
    T = max(int(s["length"]) for s in streams)
    starts = [np.asarray(s["starts"], np.int64) for s in streams]
    for s, st in zip(streams, starts):
        if len(st) and (st.min() < 0 or st.max() + span > s["length"] or np.any(np.diff(st) < 0)):
            raise ValueError("a beat's windows run outside its stream, or its starts are not ascending")
    kept = None
    if sink is None:
        kept = [np.zeros((len(st), nwin, N), np.uint8) for st in starts]
        def sink(i, ks, wc):
            kept[i][ks] = wc
    if beat_chunk is None:
        beat_chunk = max(1, int(2 ** 28 // (span * N)))     # gather buffer of about 256 MB
    nxt = [0] * B
    own_from = np.array([s["own_from"] for s in streams]); own_to = np.array([s["own_to"] for s in streams])
    ab_sum = torch.zeros(B, N, dtype=torch.float64, device=dev) if ab else None
    spike_sum = torch.zeros(B, N, dtype=torch.float64, device=dev) if counts else None
    own_steps = np.zeros(B, np.int64)
    if u0 is None:
        u0 = np.zeros((B, N))
    state, buf, buf0 = None, None, 0
    ar = torch.arange(span, device=dev)
    for a in range(0, T, seg_len):
        b = min(a + seg_len, T)
        x = enc(a, b)
        t = np.arange(a, b)
        w = (t[None, :] >= own_from[:, None]) & (t[None, :] < own_to[:, None])
        own_steps += w.sum(1)
        rec = {"step_counts": True, "keep_state": True}
        if ab and sm.adapt:
            rec["ab_weight"] = w.astype(np.float64)
        o = sm.run(x, u0 if state is None else None, b - a, rec=rec, state=state, t0=a)
        state, sc = o["state"], o["step_counts"]
        if ab and sm.adapt:
            ab_sum += o["ab_sum"]
        if counts:                                           # own steps of a stream are one range: sum it directly
            for i in range(B):
                lo, hi = max(int(own_from[i]), a) - a, min(int(own_to[i]), b) - a
                if hi > lo:
                    spike_sum[i] += sc[i, lo:hi].sum(0, dtype=torch.int64).to(torch.float64)
        buf = sc if buf is None else torch.cat([buf, sc], 1)
        jobs = []                                            # (stream, beat, start in the buffer)
        for i in range(B):
            st = starts[i]
            while nxt[i] < len(st) and st[nxt[i]] + span <= b:
                jobs.append((i, nxt[i], int(st[nxt[i]]) - buf0))
                nxt[i] += 1
        for c in range(0, len(jobs), beat_chunk):
            J = jobs[c:c + beat_chunk]
            ii = torch.as_tensor([j[0] for j in J], device=dev)
            pp = torch.as_tensor([j[2] for j in J], device=dev)
            seg = buf[ii[:, None], pp[:, None] + ar[None, :]]           # (K, span, N)
            wc = seg.view(len(J), nwin, win, N).sum(2, dtype=torch.int32).clamp(max=255).to(torch.uint8).cpu().numpy()
            by = {}
            for r, (i, k, _) in enumerate(J):
                by.setdefault(i, []).append((k, r))
            for i, kr in by.items():
                ks = np.array([k for k, _ in kr]); rs = np.array([r for _, r in kr])
                sink(i, ks, wc[rs])
        keep_from = max(buf0, b - span)                      # every beat not yet cut starts after b - span
        buf = buf[:, keep_from - buf0:]
        buf0 = keep_from
        if progress is not None:
            progress(b, T)
    if any(nxt[i] != len(starts[i]) for i in range(B)):
        raise RuntimeError("some beats were not cut")
    out = {"own_steps": own_steps, "steps": T}
    if kept is not None:
        out["windows"] = kept
    if ab:
        out["ab_sum"] = ab_sum.cpu().numpy()
    if counts:
        out["spike_sum"] = spike_sum.cpu().numpy()
    return out


def frozen_thresholds(liq, ab_sum, steps):
    """Time-averaged threshold of every neuron from summed adaptation variables: 1 + c * (sum b / steps).
    ab_sum: (N,) summed over the averaging steps (b in spikes per ms); neurons that do not adapt get 1."""
    c = np.zeros(liq.N) if liq.adapt_c is None else np.asarray(liq.adapt_c, float)
    return 1.0 + c * (np.asarray(ab_sum, float) / float(steps))


def frozen_copy(liq, thr):
    """The same liquid with each neuron's threshold fixed at thr (no adaptation): the frozen control."""
    return replace(liq, adapt_c=None, adapt_tau_ms=None, thr_base=np.asarray(thr, float).copy(),
                   meta={**liq.meta, "frozen": True})
'''

import shutil
shutil.rmtree(os.path.join(SCRATCH_DIR, "lsm2"), ignore_errors=True)
for _rel, _src in LSM2_FILES.items():
    _p = os.path.join(SCRATCH_DIR, _rel)
    os.makedirs(os.path.dirname(_p), exist_ok=True)
    with open(_p, "w") as _f:
        _f.write(_src)
open(os.path.join(SCRATCH_DIR, "lsm2", "__init__.py"), "w").close()
if SCRATCH_DIR not in sys.path:
    sys.path.insert(0, SCRATCH_DIR)
for _m in [m for m in sys.modules if m == "lsm2" or m.startswith("lsm2.")]:
    del sys.modules[_m]
import lsm2
print(f"lsm2: {len(LSM2_FILES)} files at {os.path.join(SCRATCH_DIR, 'lsm2')}")


## 4. Utilities: logging, metrics, the authors' readout

In [ ]:
# Utilities: output folders, logging, cache.txt, resume, time/disk guards, metrics, the authors' readout.
CODE_VERSION = "v1"   # part of every run's hash; bump only if experiment code changes meaning
for sub in ["runs", "reservoirs", "states", "tables", "figures", "logs", "checks"]:
    os.makedirs(os.path.join(WORK_DIR, sub), exist_ok=True)

def elapsed_h():
    return (time.time() - SESSION_START) / 3600

LOCK = threading.RLock()     # cache.txt, the log, speeds and disk counters are shared with the fit workers

def log(*parts):
    line = f"[{elapsed_h():6.2f} h] " + " ".join(str(p) for p in parts)
    with LOCK:
        print(line, flush=True)               # the tee writes it to LOG_PATH

def _json_default(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, torch.Tensor): return o.detach().cpu().tolist()
    return str(o)

def atomic_write_text(path, text):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        f.write(text)
    os.replace(tmp, path)

def jdump(obj, path):
    atomic_write_text(path, json.dumps(obj, indent=2, default=_json_default))

def stable_hash(obj, n=8):
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=_json_default).encode()).hexdigest()[:n]

# ---------------- cache.txt ----------------
CACHE_PATH = os.path.join(WORK_DIR, "cache.txt")
MACHINE_ID = f"{MACHINE}|{ENV.get('gpu', DEVICE.type)}|{N_CPU} cores"      # identifies the machine in merged outputs

def out_tag(share, machine_id):
    """Folder-safe name of an output: its share and machine."""
    return re.sub(r"[^A-Za-z0-9.]+", "_", f"{share if share not in (None, 'None') else 'all'}__{machine_id or 'unknown'}").strip("_")

class RunCache:
    """cache.txt = the record of what has run. Rewritten (atomically) after every run.
    Sections: [DONE] run_id, finished_utc, seconds, macro_f1, accuracy, group
              [FAILED] run_id, when_utc, error
              [PENDING] run_id, group"""
    def __init__(self):
        self.done, self.failed, self.pending = {}, {}, []

    def merge_file(self, path):
        if not os.path.exists(path):
            return
        section = None
        for line in open(path):
            line = line.rstrip("\n")
            if line.startswith("[") and line.endswith("]"):
                section = line[1:-1]
                continue
            if not line or line.startswith("#"):
                continue
            parts = line.split("\t")
            if section == "DONE":
                self.done[parts[0]] = parts[1:]
                self.failed.pop(parts[0], None)
            elif section == "FAILED" and parts[0] not in self.done:
                self.failed[parts[0]] = parts[1:]

    def is_done(self, rid):
        return rid in self.done

    def mark_done(self, rid, seconds, metrics, group):
        with LOCK:
            self._mark_done(rid, seconds, metrics, group)

    def _mark_done(self, rid, seconds, metrics, group):
        mf1 = f"{metrics['macro_f1']:.4f}" if metrics and "macro_f1" in metrics else "-"
        acc = f"{metrics['accuracy']:.4f}" if metrics and "accuracy" in metrics else "-"
        self.done[rid] = [datetime.datetime.utcnow().isoformat(timespec="seconds"), f"{seconds:.1f}", mf1, acc, group]
        self.failed.pop(rid, None)
        self.write()

    def mark_failed(self, rid, err):
        with LOCK:
            self._mark_failed(rid, err)

    def _mark_failed(self, rid, err):
        self.failed[rid] = [datetime.datetime.utcnow().isoformat(timespec="seconds"), err.replace("\t", " ").replace("\n", " | ")[:500]]
        self.write()

    def write(self):
        with LOCK:
            self._write()

    def _write(self):
        lines = ["# LSM-ECG replication: run cache. Rewritten after every run; read by the next session to skip finished work.",
                 f"# updated_utc: {datetime.datetime.utcnow().isoformat(timespec='seconds')}   session_elapsed_h: {elapsed_h():.2f}",
                 f"# totals: DONE {len(self.done)} | FAILED {len(self.failed)} | PENDING {len(self.pending)}",
                 f"# smoke_test: {SMOKE_TEST}",
                 f"# share: {SHARE}",
                 f"# machine_id: {MACHINE_ID}",
                 f"# code_version: {CODE_VERSION}",
                 f"# notebook_build: {NOTEBOOK_BUILD}",
                 "# DONE columns: run_id, finished_utc, seconds, macro_f1, accuracy, group", "[DONE]"]
        lines += ["\t".join([k] + v) for k, v in self.done.items()]
        lines += ["", "# FAILED columns: run_id, when_utc, error", "[FAILED]"]
        lines += ["\t".join([k] + v) for k, v in self.failed.items()]
        lines += ["", "# PENDING columns: run_id, group", "[PENDING]"]
        lines += [f"{rid}\t{g}" for rid, g in self.pending]
        atomic_write_text(CACHE_PATH, "\n".join(lines) + "\n")

CACHE = RunCache()

# ---------------- resume (previous session) and merge mode (two machines' outputs) ----------------
RESUME_SUBDIRS = ["runs", "reservoirs", "states", "logs", "checks"]   # outputs worth carrying; tables/figures are rebuilt

def _cache_is_smoke(p):
    return any(l.strip() == "# smoke_test: True" for l in itertools.islice(open(p), 12))

def cache_header(p):
    h = {}
    for l in itertools.islice(open(p), 12):
        m = re.match(r"^# (smoke_test|share|machine_id|code_version|notebook_build): (.*)$", l.rstrip("\n"))
        if m:
            h[m.group(1)] = m.group(2).strip()
    return h

def copy_forward(src_root):
    n = 0
    for sub in RESUME_SUBDIRS:
        for root, _, files in os.walk(os.path.join(src_root, sub)):
            rel = os.path.relpath(root, src_root)
            for fn in files:
                dst = os.path.normpath(os.path.join(WORK_DIR, rel, fn))
                if not os.path.exists(dst):
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    shutil.copy2(os.path.join(root, fn), dst)
                    n += 1
    return n

if MERGE_MODE:
    for md in MERGE_DIRS:
        if os.path.abspath(md) == os.path.abspath(WORK_DIR):
            raise ValueError("merge mode: WORK_DIR must be a new folder, not one of MERGE_DIRS")
        if not os.path.exists(os.path.join(md, "cache.txt")):
            raise FileNotFoundError(f"merge mode: no cache.txt in {md} (give the output folders themselves)")
    _hdrs = [cache_header(os.path.join(md, "cache.txt")) for md in MERGE_DIRS]
    if len({h.get("notebook_build") for h in _hdrs}) > 1:
        print("WARNING: the folders were made by different notebook builds: "
              + ", ".join(f"{md}: {h.get('notebook_build', '?')}" for md, h in zip(MERGE_DIRS, _hdrs)))

if RESUME_DIRS == "auto":
    RESUME_DIRS = sorted({os.path.dirname(p) for p in glob.glob(os.path.join(AUTO_RESUME_ROOT, "**", "cache.txt"), recursive=True)})
for rd in (MERGE_DIRS if MERGE_MODE else RESUME_DIRS):
    cp = os.path.join(rd, "cache.txt")
    if os.path.abspath(rd) == os.path.abspath(WORK_DIR):
        continue
    if not os.path.exists(cp):
        print(f"resume: no cache.txt in {rd} -> fresh start")
        continue
    if _cache_is_smoke(cp) != SMOKE_TEST:
        msg = f"{rd}: its smoke_test flag differs from this session's SMOKE_TEST={SMOKE_TEST}"
        if MERGE_MODE:
            raise ValueError("merge mode: " + msg)
        print(f"resume: skipping {msg}")
        continue
    before = len(CACHE.done)
    _o = RunCache(); _o.merge_file(cp); n_there = len(_o.done)
    CACHE.merge_file(cp)
    n = copy_forward(rd)
    n_dirs = len(os.listdir(os.path.join(rd, "runs"))) if os.path.isdir(os.path.join(rd, "runs")) else 0
    h = cache_header(cp)
    print(f"{'merged' if MERGE_MODE else 'resumed'} from {rd} (share {h.get('share', '?')}, {h.get('machine_id', '?')}): "
          f"{n_there} finished runs there, {len(CACHE.done) - before} of them new here, {n} files copied")
    if n_dirs < len(CACHE.done) - before - 2:
        print(f"WARNING: its cache.txt lists {len(CACHE.done) - before} finished runs but only {n_dirs} run folders are there. "
              "They will be skipped, but their results will be missing from tables and figures. "
              "Bring the whole output folder, not only cache.txt.")
if os.path.exists(CACHE_PATH) and _cache_is_smoke(CACHE_PATH) != SMOKE_TEST:
    raise RuntimeError(f"{WORK_DIR} holds {'smoke-test' if _cache_is_smoke(CACHE_PATH) else 'real'} results but SMOKE_TEST = "
                       f"{SMOKE_TEST}. Use a separate WORK_DIR for the smoke test so the two never mix.")
CACHE.merge_file(CACHE_PATH)       # this machine's earlier sessions (same folder, reused in place)
print(f"cache: {len(CACHE.done)} done, {len(CACHE.failed)} failed")

# ---------------- time and disk guards ----------------
def seconds_left():
    return SESSION_BUDGET_HOURS * 3600 - (time.time() - SESSION_START) - ANALYSIS_RESERVE_MIN * 60

_DISK = {"bytes": None, "since": 0}
def work_dir_gb(force=False):
    if force or _DISK["bytes"] is None or _DISK["since"] >= 50:
        total = 0
        for root, _, files in os.walk(WORK_DIR):
            for fn in files:
                try:
                    total += os.path.getsize(os.path.join(root, fn))
                except OSError:
                    pass
        _DISK.update(bytes=total, since=0)
    return _DISK["bytes"] / 1e9

def disk_add(nbytes):
    with LOCK:
        if _DISK["bytes"] is not None:
            _DISK["bytes"] += nbytes
            _DISK["since"] += 1

# Throughput model, measured as the session runs (and carried over between sessions in checks/speeds.json).
# Units: beats/s for simulation and encoding; seconds per (training beats x features / 1e9) for a classifier fit;
# seconds per epoch per 1,000 training beats for the CNN/GRU.
_GPU = DEVICE.type == "cuda"
# GPU defaults: roughly what a Kaggle T4 measured (used only until this machine has measured its own)
SPEED = {"sim_beats_per_s": {e: (2000.0 if _GPU else 300.0) for e in ("multi", "rate", "latency")},
         "encode_beats_per_s": {e: (20000.0 if _GPU else 1500.0) for e in ("multi", "rate", "latency")},
         "fit_s_per_Gcell": {"1": 400.0, "2": 220.0, "3": 160.0, "pipe": 400.0}, "fit_threads": float(READOUT_JOBS),
         "nn_s_per_epoch_kbeat": {"cnn": 0.04 if _GPU else 3.0, "gru": 0.025 if _GPU else 8.0},
         "nn_unit_s_per_epoch_kbeat": {"cnn": 0.03 if _GPU else 3.0, "gru": 0.04 if _GPU else 8.0},
         "lsm_job_overhead_s": 3.0, "huang_s_per_run": 1200.0}
SPEED_SAMPLES = {}                # recent measurements of the robust (median) entries
# fit_s_per_Gcell: "1"/"2"/"3" = one fit using that many cores (older runner); "pipe" = one fit per core while all
# the pipeline's workers are busy (includes their competition for memory bandwidth).
# nn_unit_*: seconds per epoch per 1,000 beats for a tuned architecture, divided by its relative cost (see nn_cost).
SPEED_MEASURED = set()            # which entries were measured (vs default guesses)
SPEEDS_PATH = os.path.join(WORK_DIR, "checks", "speeds.json")
if os.path.exists(SPEEDS_PATH):
    try:
        _sp = json.load(open(SPEEDS_PATH))
        if _sp.get("device") == DEVICE.type and _sp.get("gpu") == ENV.get("gpu") and _sp.get("cpu_count") == N_CPU:
            for k, v in _sp["speed"].items():
                if isinstance(SPEED.get(k), dict) and isinstance(v, dict):
                    SPEED[k].update(v)
                elif k in SPEED and not isinstance(SPEED[k], dict) and not isinstance(v, dict):
                    SPEED[k] = v                                               # entries whose format changed are skipped
            SPEED_MEASURED.update(m for m in _sp["measured"] if not (m == "fit_s_per_Gcell"))
            SPEED_SAMPLES.update(_sp.get("samples", {}))
            print("speeds carried over from an earlier session:", sorted(SPEED_MEASURED))
    except Exception as ex:
        print("could not read speeds.json:", ex)

def update_speed(key, value, sub=None, alpha=0.3, robust=False):
    with LOCK:
        if robust:            # median of the last 40 measurements: one odd fit does not move the projection
            tag = f"{key}:{sub}" if sub else key
            xs = SPEED_SAMPLES.setdefault(tag, []); xs.append(float(value)); del xs[:-40]
            value, alpha = float(np.median(xs)), 1.0
        _update_speed(key, value, sub, alpha)

def _update_speed(key, value, sub, alpha):
    tag = f"{key}:{sub}" if sub else key
    if sub is None:
        SPEED[key] = value if tag not in SPEED_MEASURED else alpha * value + (1 - alpha) * SPEED[key]
    else:
        SPEED[key][sub] = value if tag not in SPEED_MEASURED else alpha * value + (1 - alpha) * SPEED[key][sub]
    SPEED_MEASURED.add(tag)

def save_speeds():
    with LOCK:
        jdump({"device": DEVICE.type, "gpu": ENV.get("gpu"), "cpu_count": N_CPU, "speed": SPEED, "samples": SPEED_SAMPLES, "measured": sorted(SPEED_MEASURED)}, SPEEDS_PATH)

# ---------------- metrics ----------------
CLASS_NAMES = ["N", "SVEB", "VEB"]

def compute_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    tp = np.diag(cm).astype(float)
    support, predicted, total = cm.sum(1).astype(float), cm.sum(0).astype(float), cm.sum()
    rec = np.divide(tp, support, out=np.zeros(3), where=support > 0)
    prec = np.divide(tp, predicted, out=np.zeros(3), where=predicted > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(3), where=(prec + rec) > 0)
    fp = predicted - tp
    tn = total - support - fp
    spec = np.divide(tn, tn + fp, out=np.zeros(3), where=(tn + fp) > 0)
    m = {"accuracy": float(tp.sum() / total), "macro_f1": float(f1.mean()), "macro_recall": float(rec.mean()),
         "macro_precision": float(prec.mean()), "n": int(total), "confusion_matrix": cm.tolist()}
    for i, c in enumerate(CLASS_NAMES):
        m.update({f"recall_{c}": float(rec[i]), f"precision_{c}": float(prec[i]), f"f1_{c}": float(f1[i]),
                  f"specificity_{c}": float(spec[i]), f"fpr_{c}": float(1 - spec[i]), f"support_{c}": int(support[i])})
    return m

# ---------------- the authors' readout (exact settings) ----------------
def soft_class_weight(y, p=0.6):
    classes = np.unique(y)
    base = compute_class_weight(class_weight="balanced", classes=classes, y=y).astype(np.float64)
    return dict(zip(classes.astype(int), (base ** p).astype(float)))

CURVE_EVAL_EPOCHS = {1, 3, 5, 10, 15}      # epochs at which the validation curve is measured (CV runs only)

class Cancelled(Exception):
    """Raised inside a fit when the session's time is up; the run stays pending for the next session."""

def prepare_train(Xtr_fn, ntr, prescale):
    """StandardScaler fitted with batched partial_fit (authors' way); optionally the whole scaled training matrix."""
    bs = 4096
    scaler = StandardScaler(with_mean=True, with_std=True)
    for i in range(0, ntr, bs):
        scaler.partial_fit(Xtr_fn(np.arange(i, min(i + bs, ntr))))
    XS = None
    if prescale:
        XS = np.concatenate([scaler.transform(Xtr_fn(np.arange(i, min(i + bs, ntr)))).astype(np.float32) for i in range(0, ntr, bs)])
    return scaler, XS

def fit_readout(Xtr_fn, ntr, ytr, Xev_fn, nev, loss, rng_seed, yev_for_curve=None, curve_sub=4000, prescale=False, n_jobs=None,
                prepared=None, cancel=None, epochs=None):
    """Authors' readout: StandardScaler (batched partial_fit) + SGDClassifier(hinge|log_loss, l2, alpha=1e-4,
    optimal LR, average=True, random_state=2026, soft class weights ^0.6), 15 epochs, batch 4096,
    shuffling with np.random.default_rng(rng_seed). Batches are scaled on the fly as in the authors' code, or, when
    prescale=True and RAM allows, the whole matrix is scaled once (row-wise identical values, faster).
    prepared=(scaler, XS) reuses a scaler / scaled matrix shared by several fits on the same training rows.
    Xtr_fn(positions) / Xev_fn(positions) return float32 feature rows."""
    bs = 4096
    scaler, XS = prepared if prepared is not None else prepare_train(Xtr_fn, ntr, prescale)
    tf = lambda X: scaler.transform(X).astype(np.float32)
    get_tr = (lambda p: XS[p]) if XS is not None else (lambda p: tf(Xtr_fn(p)))
    clf = SGDClassifier(loss=loss, penalty="l2", alpha=1e-4, fit_intercept=True, learning_rate="optimal",
                        max_iter=1, tol=None, class_weight=soft_class_weight(ytr), average=True, random_state=2026,
                        n_jobs=n_jobs)   # n_jobs only parallelises the 3 one-vs-rest problems: results are bit-identical
    classes = np.unique(ytr)
    rng = np.random.default_rng(rng_seed)
    sub = np.sort(np.random.default_rng(0).choice(ntr, min(curve_sub, ntr), replace=False))
    Xsub = get_tr(sub)
    def predict_eval():
        preds, scores = [], []
        for i in range(0, nev, bs):
            Xe = tf(Xev_fn(np.arange(i, min(i + bs, nev))))
            preds.append(clf.predict(Xe)); scores.append(clf.decision_function(Xe).astype(np.float32))
        return np.concatenate(preds), np.concatenate(scores)
    curve = []
    for ep in range(epochs or READOUT_EPOCHS):          # epochs: only the core search uses fewer
        idx = rng.permutation(ntr)
        for i in range(0, ntr, bs):
            if cancel is not None and cancel.is_set():
                raise Cancelled()
            sel = idx[i:i + bs]
            Xb = get_tr(sel)
            if ep == 0 and i == 0:
                clf.partial_fit(Xb, ytr[sel], classes=classes)
            else:
                clf.partial_fit(Xb, ytr[sel])
        row = {"epoch": ep + 1, "train_sub_macro_f1": compute_metrics(ytr[sub], clf.predict(Xsub))["macro_f1"]}
        if loss == "hinge":
            dec = clf.decision_function(Xsub)
            onehot = -np.ones_like(dec); onehot[np.arange(len(sub)), ytr[sub]] = 1
            row["train_sub_hinge_loss"] = float(np.maximum(0, 1 - onehot * dec).mean())
        if yev_for_curve is not None and (ep + 1) in CURVE_EVAL_EPOCHS:
            row["eval_macro_f1"] = compute_metrics(yev_for_curve, predict_eval()[0])["macro_f1"]
        curve.append(row)
    y_pred, scores = predict_eval()
    return scaler, clf, y_pred, scores, curve

# ---------------- saving a run ----------------
def save_run(rid, spec, group, metrics, eval_idx, eval_pid, y_true, y_pred, scores, model_obj,
             curve, liquid_stats, timing, extra=None):
    d = os.path.join(WORK_DIR, "runs", rid)
    os.makedirs(d, exist_ok=True)
    jdump({"run_id": rid, "group": group, "spec": spec, "env": ENV, "code_version": CODE_VERSION,
           "session": SESSION_TAG, **(extra or {})}, os.path.join(d, "config.json"))
    jdump(metrics, os.path.join(d, "metrics.json"))
    np.savez_compressed(os.path.join(d, "predictions.npz"), eval_index=np.asarray(eval_idx, np.int32),
                        patient_id=np.asarray(eval_pid).astype("U4"), y_true=np.asarray(y_true, np.int8),
                        y_pred=np.asarray(y_pred, np.int8),
                        scores=np.asarray(scores, np.float32) if scores is not None else np.zeros((0, 3), np.float32))
    if model_obj is not None:
        if isinstance(model_obj, dict) and model_obj.get("_torch_state"):
            torch.save(model_obj, os.path.join(d, "model.pt"))
        else:
            joblib.dump(model_obj, os.path.join(d, "readout.joblib"), compress=3)
    if curve:
        pd.DataFrame(curve).to_csv(os.path.join(d, "train_curve.csv"), index=False)
    if liquid_stats:
        jdump(liquid_stats, os.path.join(d, "liquid_stats.json"))
    jdump(timing, os.path.join(d, "timing.json"))
    with open(os.path.join(d, "log.txt"), "w") as f:
        f.write(f"{rid}\nmacro_f1={metrics.get('macro_f1')}\naccuracy={metrics.get('accuracy')}\n"
                f"confusion_matrix={metrics.get('confusion_matrix')}\ntiming={timing}\n")
    disk_add(sum(os.path.getsize(os.path.join(d, f)) for f in os.listdir(d)))


## 5. Data: MIT-BIH → DS1 / DS2 and the authors' folds

In [ ]:
# Data: locate + checksum MIT-BIH, build DS1/DS2 exactly as the authors do (both filters),
# track each beat's position for RR-interval features, verify against the authors' released files.
def locate_input():
    cands = [INPUT_DIR] + sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/100.hea", recursive=True)})
    for c in cands:
        if os.path.exists(os.path.join(c, "100.hea")) and os.path.exists(os.path.join(c, "234.atr")):
            return c
    raise FileNotFoundError(f"MIT-BIH not found at {INPUT_DIR} or anywhere under /kaggle/input")

DATA_DIR = locate_input()
print("MIT-BIH at:", DATA_DIR)

CHECKS = {}
_recs = pd.read_csv(os.path.join(REPO_DIR, "splits/ds1_ds2_records.csv")).set_index("set")["records"]
DS_RECORDS = {"DS1": _recs["DS1"].split(","), "DS2": _recs["DS2"].split(",")}
DS_RECORDS_ALL = DS_RECORDS["DS1"] + DS_RECORDS["DS2"]
sums_path = os.path.join(DATA_DIR, "SHA256SUMS.txt")
if os.path.exists(sums_path):
    bad, missing, n = [], [], 0
    for line in open(sums_path):
        h, fn = line.strip().split(maxsplit=1)
        fp = os.path.join(DATA_DIR, fn)
        if os.path.isfile(fp):
            n += 1
            if hashlib.sha256(open(fp, "rb").read()).hexdigest() != h:
                bad.append(fn)
        else:
            missing.append(fn)
    used = {f"{r}.{x}" for r in DS_RECORDS_ALL for x in ("dat", "hea", "atr")}
    CHECKS["checksums"] = {"files_listed": n + len(missing), "files_checked": n, "mismatches": bad, "missing": missing,
                           "missing_files_used_by_pipeline": sorted(set(missing) & used)}
    print(f"checksums: {n} files checked, {len(bad)} mismatches, {len(missing)} missing"
          + (f" ({', '.join(missing[:12])}{' ...' if len(missing) > 12 else ''})" if missing else ""))
    if set(missing) & used:
        raise FileNotFoundError(f"record files needed by the pipeline are missing: {sorted(set(missing) & used)}")
else:
    CHECKS["checksums"] = {"files_checked": 0, "note": "SHA256SUMS.txt not present"}


NORMAL_S, SVEB_S, VEB_S = ['N', 'L', 'R', 'e', 'j'], ['A', 'a', 'J', 'S'], ['V', 'E']
BEAT_SYMBOLS = set("NLRBAaJSVrFejnE/fQ?")      # all beat annotations (for RR intervals)
RR_NAMES = ["pre", "post", "local10", "rec_median", "pre_over_local", "post_over_local", "pre_over_rec", "pre_over_post"]

def rr_features(all_beats, pos, fs):
    """RR features (seconds / ratios) for beats at sample positions `pos`, from every beat annotation."""
    all_beats = np.sort(np.asarray(all_beats))
    rr = np.diff(all_beats) / fs
    rec_med = float(np.median(rr))
    i = np.searchsorted(all_beats, pos)
    pre = np.where(i > 0, rr[np.clip(i - 1, 0, len(rr) - 1)], rec_med)
    post = np.where(i < len(all_beats) - 1, rr[np.clip(i, 0, len(rr) - 1)], rec_med)
    csum = np.concatenate([[0], np.cumsum(rr)])
    lo = np.clip(i - 10, 0, None)
    cnt = np.maximum(i - lo, 1)
    local = np.where(i > 0, (csum[np.clip(i, 0, len(rr))] - csum[lo]) / cnt, rec_med)
    f = np.stack([pre, post, local, np.full_like(pre, rec_med), pre / local, post / local, pre / rec_med, pre / post], 1)
    return f.astype(np.float32)

def rr_mean_ratio(all_beats, pos, fs):
    """pre-RR / mean RR of the whole recording (Huang et al. 2014); the same pre-RR as the "pre" column."""
    rr = np.diff(np.sort(np.asarray(all_beats))) / fs
    return (rr_features(all_beats, pos, fs)[:, 0] / rr.mean()).astype(np.float32)

def huang_windows(sig, pos, half=100):
    """Huang et al.: 200 raw (unfiltered) samples, R-100 .. R+99 at 360 Hz, minus their mean, divided by their SD."""
    W = np.stack([sig[p - half:p + half] for p in pos]).astype(np.float64)
    return ((W - W.mean(1, keepdims=True)) / W.std(1, keepdims=True)).astype(np.float32)

def build_set(records, filt):
    Xs, syms, pids, poss, rrs = [], [], [], [], []
    rrms, hws = [], []          # extensions: pre-RR / record mean RR, Huang windows
    for rec in records:
        sig, atr_sym, atr_sample, fs = A03.load_ecg(os.path.join(DATA_DIR, rec), target_fs=360)
        if filt == "median":
            sig_f, _ = A_sig.median_filters(sig, fs=fs, kernel_size_ms1=200, kernel_size_ms2=600, mode='reflect')
        else:   # the authors' (commented-out) morphological line
            _, sig_f, _, _ = A_sig.morph_filter_baseline(sig, fs=360, k1_ratio=0.20, k2_ratio=0.35, k3_ratio=0.005, mode='reflect')
        df_all = pd.DataFrame({'atr_sym': atr_sym, 'atr_sample': atr_sample})
        df = df_all[df_all.atr_sym.isin(NORMAL_S + SVEB_S + VEB_S)].reset_index(drop=True)
        Xp, _, sp, pp = A03.build_XY(sig_f, df, 0.8, 0.8, fs, NORMAL_S, SVEB_S + VEB_S, patient_id=rec)
        L = int(round(0.8 * fs))
        keep = ((df.atr_sample - L) >= 0) & ((df.atr_sample + L + 1) <= len(sig_f))
        pos = df.atr_sample[keep].to_numpy()
        assert len(pos) == len(Xp) and list(df.atr_sym[keep]) == list(sp)
        beats_all = [s for s, y in zip(atr_sample, atr_sym) if y in BEAT_SYMBOLS]
        Xs.append(Xp); syms += list(sp); pids += list(pp); poss.append(pos); rrs.append(rr_features(beats_all, pos, fs))
        rrms.append(rr_mean_ratio(beats_all, pos, fs)); hws.append(huang_windows(sig, pos))
    X = np.vstack(Xs); pos = np.concatenate(poss); rr = np.vstack(rrs); pid = np.array(pids)
    num = A03.Numerical_Labels(syms)
    X_sh, y_sh, pid_sh = A03.Shuffle_data(X, num, pid, seed=42)
    np.random.seed(42); perm = np.random.permutation(len(X))           # the same permutation Shuffle_data used
    assert np.array_equal(X[perm], X_sh)
    with contextlib.redirect_stdout(io.StringIO()):     # silence the authors' "taking all of them" prints
        Xd, yd, pidd = A_strat.Make_Data_Stratified(X_sh, y_sh, pid_sh, num_data=50000, Type_Class='SuperC')
        idx_d, _, _ = A_strat.Make_Data_Stratified(np.arange(len(X))[:, None], y_sh, pid_sh, num_data=50000, Type_Class='SuperC')
    src = perm[np.asarray(idx_d)[:, 0].astype(int)]
    assert np.array_equal(X[src], Xd), "beat-order tracking failed"
    return {"X": np.asarray(Xd, np.float64), "y": np.asarray(yd, int), "pid": np.asarray(pidd).astype(str),
            "pos": pos[src], "rr": rr[src], "sym": np.array(syms)[src],
            "rr_mean_ratio": np.concatenate(rrms)[src], "huang_beats": np.vstack(hws)[src]}

t0 = time.time()
DATA = {f: {ds: build_set(DS_RECORDS[ds], f) for ds in ("DS1", "DS2")} for f in ("morph", "median")}
print(f"datasets built in {time.time()-t0:.0f}s")
for ds in ("DS1", "DS2"):
    assert np.array_equal(DATA["morph"][ds]["y"], DATA["median"][ds]["y"])
    assert np.array_equal(DATA["morph"][ds]["pid"], DATA["median"][ds]["pid"])

# verify against the authors' released DS1 order and counts
_auth = pd.read_csv(os.path.join(REPO_DIR, "splits/patient_ids_train.csv"))
CHECKS["ds1_matches_authors_order"] = bool(np.array_equal(_auth.patient_id.astype(str).values, DATA["morph"]["DS1"]["pid"])
                                           and np.array_equal(_auth.label.values, DATA["morph"]["DS1"]["y"]))
CHECKS["class_counts"] = {ds: np.bincount(DATA["morph"][ds]["y"], minlength=3).tolist() for ds in ("DS1", "DS2")}
CHECKS["expected_counts"] = {"DS1": [45827, 943, 3787], "DS2": [44216, 1837, 3219]}
print("DS1 order identical to authors' file:", CHECKS["ds1_matches_authors_order"], "| counts:", CHECKS["class_counts"])

# folds: the authors' exact indices, and a check that they regenerate
_fd = json.load(open(os.path.join(REPO_DIR, "splits/fold_definitions.json")))
FOLDS = [{"rep": f["rep"], "fold": f["fold"], "train_idx": np.array(f["train_idx"]), "val_idx": np.array(f["val_idx"])}
         for f in _fd["folds"]]
_ok, _k = True, 0
_y1, _p1 = DATA["morph"]["DS1"]["y"], DATA["morph"]["DS1"]["pid"]
for rep, sd in enumerate([11, 23, 37], 1):
    for fold, (tr, va) in enumerate(StratifiedGroupKFold(4, shuffle=True, random_state=sd).split(np.zeros(len(_y1)), _y1, _p1), 1):
        f = FOLDS[_k]; _k += 1
        _ok &= (f["rep"], f["fold"]) == (rep, fold) and np.array_equal(np.sort(f["train_idx"]), tr) and np.array_equal(np.sort(f["val_idx"]), va)
CHECKS["folds_regenerate_exactly"] = bool(_ok)
print("authors' fold indices regenerate exactly:", _ok)

# smoke test: keep a few beats per record and class, remap folds
if SMOKE_TEST:
    def _keep(D):
        keep = []
        for p in np.unique(D["pid"]):
            for c, k in SMOKE_PER_CLASS.items():
                keep += list(np.where((D["pid"] == p) & (D["y"] == c))[0][:k])
        return np.sort(np.array(keep))
    _k1, _k2 = _keep(DATA["morph"]["DS1"]), _keep(DATA["morph"]["DS2"])
    for f in DATA:
        for ds, kk in (("DS1", _k1), ("DS2", _k2)):
            DATA[f][ds] = {k: v[kk] for k, v in DATA[f][ds].items()}
    _remap_arr = -np.ones(int(_k1.max()) + 1, int); _remap_arr[_k1] = np.arange(len(_k1))
    def _remap(idx):
        r = _remap_arr[idx[idx <= _k1.max()]]
        return r[r >= 0]
    FOLDS = [{"rep": f["rep"], "fold": f["fold"], "train_idx": _remap(f["train_idx"]), "val_idx": _remap(f["val_idx"])}
             for f in FOLDS]
    print("SMOKE subset:", {ds: np.bincount(DATA['morph'][ds]['y'], minlength=3).tolist() for ds in ("DS1", "DS2")})

CV_SPLITS = [(f["rep"], f["fold"]) for f in FOLDS]
if SMOKE_TEST:
    CV_SPLITS = SMOKE_CV_SPLITS
FOLD_BY = {(f["rep"], f["fold"]): f for f in FOLDS}

# ---------------- normalization (two paths, as the authors have them) ----------------
def normalize_cv(X, tr_idx, norm):
    """CV path (script 03 get_fold_data + float32 cast before encoding)."""
    if norm == "standard":
        return np.asarray(A_strat.minmax_per_beat(X), np.float32), {}
    mn, mx = A_strat.robust_minmax_fit(X[tr_idx], low=0.1, high=99.9)
    return np.asarray(A_strat.robust_minmax_apply(X, mn, mx), np.float32), {"p_low": mn, "p_high": mx}

def normalize_final(X1, X2, norm):
    """Final DS1 -> DS2 path (script 01): float64, global DS1 percentiles for robust."""
    if norm == "standard":
        return A_strat.Rowwise_Normalization(X1), A_strat.Rowwise_Normalization(X2), {}
    lo, hi = np.percentile(X1, [0.1, 99.9])
    return A_strat.minmax_apply(X1, lo, hi), A_strat.minmax_apply(X2, lo, hi), {"p_low": float(lo), "p_high": float(hi)}


## 6. Smoke subset (SMOKE only)

In [ ]:
# SMOKE only: keep the beats whose 32 windows end inside each record's first SMOKE_SECONDS, so every record can be
# streamed whole in seconds. The data cell above kept every beat (SMOKE_PER_CLASS is unlimited in this notebook);
# its folds are remapped the same way the data cell remaps them.
# If a class is missing from either set, the stretch is doubled until every class is there (the readout needs all 3).
if SMOKE_TEST:
    while True:
        _lim = int(SMOKE_SECONDS * 360)
        _k1 = np.where(DATA["morph"]["DS1"]["pos"] + 352 <= _lim)[0]
        _k2 = np.where(DATA["morph"]["DS2"]["pos"] + 352 <= _lim)[0]
        if all(len(np.unique(DATA["morph"][d]["y"][k])) == 3 for d, k in (("DS1", _k1), ("DS2", _k2))) or SMOKE_SECONDS >= 1806:
            break
        SMOKE_SECONDS *= 2
    for _f in DATA:
        for _ds, _kk in (("DS1", _k1), ("DS2", _k2)):
            DATA[_f][_ds] = {k: v[_kk] for k, v in DATA[_f][_ds].items()}
    _map = -np.ones(int(max(_k1.max(), max(f["train_idx"].max() for f in FOLDS), max(f["val_idx"].max() for f in FOLDS))) + 1, int)
    _map[_k1] = np.arange(len(_k1))
    def _remap(idx):
        r = _map[idx]
        return r[r >= 0]
    FOLDS = [{"rep": f["rep"], "fold": f["fold"], "train_idx": _remap(f["train_idx"]), "val_idx": _remap(f["val_idx"])}
             for f in FOLDS]
    FOLD_BY = {(f["rep"], f["fold"]): f for f in FOLDS}
    print(f"SMOKE: beats in the first {SMOKE_SECONDS} s of each record: DS1 {len(_k1)}, DS2 {len(_k2)}; "
          f"classes {np.bincount(DATA['morph']['DS1']['y'], minlength=3).tolist()} / {np.bincount(DATA['morph']['DS2']['y'], minlength=3).tolist()}")


## 7. Spike encoders, RR channels, measured A_in

In [ ]:
# Encoders (the authors' functions, device passed explicitly), RR spike channels, A_in measurement.
PAPER_AIN = {"multi": 0.05892, "rate": 20.8245, "latency": 0.9473}   # values hard-coded by the authors

def multithreshold_t(Xb, num_levels=200):
    """GPU version of the authors' MultiThreshold (detect_edges='both', include_endpoints=True,
    separate_polarity=False). Same thresholds (np.linspace in the data's dtype) and the same comparisons:
    rise = x_t >= thr & x_{t-1} < thr, fall = x_t < thr & x_{t-1} >= thr, spike = rise | fall."""
    X = np.ascontiguousarray(np.asarray(Xb))
    thr = torch.from_numpy(np.linspace(0.0, 1.0, num_levels, dtype=X.dtype)).to(DEVICE)
    x = torch.from_numpy(X).to(DEVICE)
    out = torch.zeros((X.shape[0], X.shape[1], num_levels), dtype=torch.uint8, device=DEVICE)
    above = x[:, :, None] >= thr                     # (B, T, K)
    out[:, 1:] = (above[:, 1:] != above[:, :-1]).to(torch.uint8)
    return out

def roundrobin_t(counts, bin_pop=10):
    """GPU version of the authors' counts_to_binary_population_roundrobin. Their loop keeps a pointer p per
    (beat, channel), sets sub-channels (p + j) % bin_pop for j < min(k, bin_pop), then p = (p + k) % bin_pop.
    So p_t = (counts before t) % bin_pop, and sub-channel b fires iff (b - p_t) % bin_pop < min(k_t, bin_pop)."""
    c = torch.as_tensor(np.asarray(counts).astype(np.int64), device=DEVICE)          # (B, L, C)
    p = torch.remainder(torch.cumsum(c, dim=1) - c, bin_pop)
    keff = torch.clamp(c, max=bin_pop)
    b = torch.arange(bin_pop, device=DEVICE)
    rel = torch.remainder(b[None, None, None, :] - p[..., None], bin_pop)
    out = (rel < keff[..., None]).to(torch.uint8)
    return out.reshape(c.shape[0], c.shape[1], c.shape[2] * bin_pop)

def encode_t(Xb, enc, num_levels=200):
    """(B, 577) normalized beats -> (B, T_in, C) uint8 spikes as a tensor on DEVICE."""
    if enc == "multi":
        return multithreshold_t(Xb, num_levels)
    if enc == "rate":
        s = A_enc.TemporalRateCoding(Xb, None, gain=1.0, offset=0.0, pop_size=64, batch_size=1024, device=DEVICE,
                                     out_dir=None, spike_filename=None, labels_filename=None, save_to_disk=False)
        return torch.from_numpy(np.asarray(s, dtype=np.uint8)).to(DEVICE)
    if enc == "latency":
        cnt = A_enc.BoundedDelayLatencyCoding(Xb, Dmax=64, pop_size=1, batch_size=1024, device=DEVICE)
        return roundrobin_t(cnt, bin_pop=10)
    raise ValueError(enc)

def encode(Xb, enc, num_levels=200):
    """Same as encode_t, returned as a NumPy array."""
    return encode_t(Xb, enc, num_levels).cpu().numpy()

N_IN = {"multi": 200, "rate": 64, "latency": 10}

# RR timing as extra input channels: thermometer code of 3 ratio features, 10 levels each.
RR_LIQ_FEATS = [RR_NAMES.index(n) for n in ("pre_over_local", "post_over_local", "pre_over_rec")]
RR_LEVELS, RR_PERIOD = 10, 5            # an active channel spikes every 5th step

def rr_liq_feats(rr_rows):
    """Columns turned into RR channels: the 3 ratios; with the 9-column array of rr_inputs(..., True) also the
    9th column, pre-RR / recording mean RR (40 channels instead of 30)."""
    return RR_LIQ_FEATS + ([len(RR_NAMES)] if rr_rows.shape[1] > len(RR_NAMES) else [])

def rr_thresholds(rr_train):
    q = np.linspace(0.05, 0.95, RR_LEVELS)
    return np.stack([np.quantile(rr_train[:, j], q) for j in rr_liq_feats(rr_train)])     # (3 or 4, 10)

def rr_channels(rr_rows, thr, T):
    active = (rr_rows[:, rr_liq_feats(rr_rows)][:, :, None] >= thr[None]).reshape(len(rr_rows), -1)   # (B, 30 or 40)
    pulse = (np.arange(T) % RR_PERIOD == 0).astype(np.uint8)
    return (pulse[None, :, None] * active[:, None, :]).astype(np.uint8)                   # (B, T, 30)

def rr_window_counts(rr_rows, thr, T):
    """rr_channels(rr_rows, thr, T) summed over the 32 windows of 20 steps (steps >= T are silent), (B, 32, 30) uint8."""
    active = (rr_rows[:, rr_liq_feats(rr_rows)][:, :, None] >= thr[None]).reshape(len(rr_rows), -1)
    steps = np.arange(NWIN * WIN)
    per_window = ((steps % RR_PERIOD == 0) & (steps < T)).reshape(NWIN, WIN).sum(1)
    return (per_window[None, :, None] * active[:, None, :]).astype(np.uint8)

def measure_ain(Xn, rows, enc, num_levels=200, rr=None, rr_thr=None, n=5000):
    """Average number of active input spikes per time step (summed over channels), over a random
    subset of training beats. Same quantity as the authors' estimate_K_spikes_per_timestep."""
    rows = np.asarray(rows)
    torch.manual_seed(12345)           # the Rate encoder is stochastic: make A_in independent of job order
    pick = np.sort(np.random.default_rng(0).choice(rows, min(n, len(rows)), replace=False))
    tot, cnt = 0.0, 0
    for i in range(0, len(pick), 1000):
        r = pick[i:i + 1000]
        S = encode(Xn[r], enc, num_levels)
        if rr is not None:
            S = np.concatenate([S, rr_channels(rr[r], rr_thr, S.shape[1])], axis=2)
        tot += float(S.sum(dtype=np.int64)); cnt += S.shape[0] * S.shape[1]
    return tot / cnt


## 8. The paper's liquid class

In [ ]:
# The liquid: built with the authors' class; modes patch the two bugs. GPU simulator with diagnostics.
LIQ = dict(P_in=0.1, betaE=0.95, betaI=0.8, Uthr=1.0, rho=0.9, reservoir_dim=(7, 7, 7), excit_ratio=0.8,
           dt=1.0, num_sim_steps=650, C_vals={"exc_exc": 0.3, "exc_inh": 0.2, "inh_exc": 0.4, "inh_inh": 0.1},
           lambda_val=0.8, scl_v=1, mu_rsv=[0.020, 0.025, 0.050, 0.040], g_rec=1, tau_syn=0.01)
I_TARGET = LIQ["rho"] * (1 - LIQ["betaE"]) * LIQ["Uthr"]
MODES = {   # gamma_fix: Gamma(shape, scale) as intended; ain: "paper" = hard-coded value, "measured" = Eq. as described
    "published": {"gamma_fix": False, "ain": "paper"},
    "fixed":     {"gamma_fix": True,  "ain": "measured"},
    "fix_gamma": {"gamma_fix": True,  "ain": "paper"},
    "fix_ain":   {"gamma_fix": False, "ain": "measured"},
}
_TorchGamma = torch.distributions.Gamma

def build_liquid(input_size, mu_in, seed, gamma_fix, lambda_val=0.8, recurrence=True, g_rec=None):
    """seed_everything(seed) then the authors' constructor, exactly as in their scripts."""
    seed_everything(seed, deterministic=False)
    if gamma_fix:
        torch.distributions.Gamma = lambda conc, scale: _TorchGamma(conc, 1.0 / scale)   # (shape, scale) as intended
    try:
        model = LSM3D(input_size, LIQ["P_in"], mu_in, LIQ["betaE"], LIQ["betaI"], LIQ["reservoir_dim"],
                      LIQ["excit_ratio"], LIQ["dt"], LIQ["num_sim_steps"], LIQ["C_vals"], lambda_val, LIQ["scl_v"],
                      np.array(LIQ["mu_rsv"]), LIQ["g_rec"] if g_rec is None else g_rec, LIQ["tau_syn"], DEVICE).to(DEVICE)
    finally:
        torch.distributions.Gamma = _TorchGamma
    if not recurrence:
        model.rsv_weights = torch.zeros_like(model.rsv_weights)
    return model

def structure_stats(model):
    W = model.rsv_weights.detach().cpu().numpy(); Win = model.input_rsv_weights.detach().cpu().numpy()
    E = model.is_excit.cpu().numpy()
    out = {"n_recurrent_synapses": int((W != 0).sum()), "mean_recurrent_in_degree": float((W != 0).sum(0).mean()),
           "neurons_without_recurrent_input": int(((W != 0).sum(0) == 0).sum()),
           "n_input_synapses": int((Win != 0).sum()), "mean_input_in_degree": float((Win != 0).sum(0).mean()),
           "mean_nonzero_input_weight": float(Win[Win != 0].mean()) if (Win != 0).any() else 0.0,
           "threshold": LIQ["Uthr"], "n_excitatory": int(E.sum())}
    for name, pre, post in (("EE", E, E), ("EI", E, ~E), ("IE", ~E, E), ("II", ~E, ~E)):
        blk = W[np.ix_(pre, post)]
        nz = blk[blk != 0]
        out[f"{name}_synapses"] = int(nz.size)
        out[f"{name}_mean_abs_weight"] = float(np.abs(nz).mean()) if nz.size else 0.0
    return out

def save_reservoir(model, meta):
    rid = stable_hash(meta, 12)
    path = os.path.join(WORK_DIR, "reservoirs", f"{rid}.pt")
    if not os.path.exists(path):
        Win = model.input_rsv_weights.detach().cpu(); W = model.rsv_weights.detach().cpu()
        iw, ir = Win.nonzero().T, W.nonzero().T
        torch.save({"meta": meta, "input_size": Win.shape[0], "n_neurons": W.shape[0],
                    "win_index": iw.to(torch.int16), "win_value": Win[iw[0], iw[1]].float(),
                    "wrec_index": ir.to(torch.int16), "wrec_value": W[ir[0], ir[1]].float(),
                    "is_excit": model.is_excit.cpu(), "beta": torch.where(model.is_excit.cpu(), LIQ["betaE"], LIQ["betaI"]).float(),
                    "structure": structure_stats(model), "torch": torch.__version__, "device": str(DEVICE)}, path)
        disk_add(os.path.getsize(path))
    return rid

WIN, NWIN, N_NEUR = 20, 32, 343          # fr_windows(win=20, stride=20) over 650 steps -> 32 windows

def simulate(model, Xn, enc, batch, num_levels=200, rr=None, rr_thr=None, y=None, enc_seed=0):
    """Encode + run the liquid exactly as the authors' loop does. Returns spike counts per (window, neuron)
    as uint8 (N, 32*343) in the authors' feature order (window-major), plus liquid diagnostics."""
    torch.manual_seed(enc_seed)
    N = len(Xn)
    out = np.empty((N, NWIN * N_NEUR), np.uint8)
    Win, Wr = model.input_rsv_weights, model.rsv_weights
    neur = torch.zeros(N_NEUR, dtype=torch.float64, device=DEVICE)
    per_beat = np.empty(N, np.float32)
    acc = {"in_spikes": 0.0, "in_steps": 0, "Iin_sum": 0.0, "Iin_steps": 0, "Irec_abs_sum": 0.0}
    t_enc = t_sim = 0.0
    with torch.no_grad():
        for s in range(0, N, batch):
            t0 = time.time()
            x = encode_t(Xn[s:s + batch], enc, num_levels)
            if rr is not None:
                x = torch.cat([x, torch.from_numpy(rr_channels(rr[s:s + batch], rr_thr, x.shape[1])).to(DEVICE)], dim=2)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t_enc += time.time() - t0; t0 = time.time()
            B, Tin, C = x.shape
            spk = torch.zeros(B, N_NEUR, device=DEVICE); mem = torch.zeros(B, N_NEUR, device=DEVICE)
            zero_in = torch.zeros(B, C, device=DEVICE)
            cnt = torch.zeros(B, NWIN, N_NEUR, dtype=torch.int16, device=DEVICE)
            Iin_acc = torch.zeros((), dtype=torch.float64, device=DEVICE); Irec_acc = torch.zeros((), dtype=torch.float64, device=DEVICE)
            for t in range(LIQ["num_sim_steps"]):
                x_t = x[:, t, :].float() if t < Tin else zero_in
                I_in = x_t @ Win
                I_rec = spk @ Wr
                I = (I_in + I_rec) * model.dt
                spk, mem = model.lif(I, mem)
                if t < NWIN * WIN:
                    cnt[:, t // WIN] += spk.to(torch.int16)
                if t < Tin:
                    Iin_acc += I_in.mean(dtype=torch.float64)
                Irec_acc += I_rec.abs().mean(dtype=torch.float64)
            c = cnt.reshape(B, -1)
            out[s:s + B] = c.to(torch.uint8).cpu().numpy()
            neur += cnt.sum((0, 1), dtype=torch.float64)
            per_beat[s:s + B] = c.sum(1).float().cpu().numpy()
            acc["in_spikes"] += float(x.sum(dtype=torch.int64)); acc["in_steps"] += B * Tin
            acc["Iin_sum"] += float(Iin_acc) * B; acc["Iin_steps"] += Tin * B
            acc["Irec_abs_sum"] += float(Irec_acc) * B
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t_sim += time.time() - t0
            del x, cnt, spk, mem
    neur = neur.cpu().numpy()
    steps = NWIN * WIN
    E = model.is_excit.cpu().numpy()
    stats = {"liquid_firing_prob_per_step": float(neur.sum() / (N * steps * N_NEUR)),
             "excitatory_firing_prob": float(neur[E].sum() / (N * steps * E.sum())),
             "inhibitory_firing_prob": float(neur[~E].sum() / (N * steps * (~E).sum())),
             "silent_neurons": int((neur == 0).sum()), "silent_fraction": float((neur == 0).mean()),
             "liquid_spikes_per_beat_mean": float(per_beat.mean()),
             "input_spikes_per_step": acc["in_spikes"] / max(acc["in_steps"], 1),
             "mean_input_current_per_neuron_step": acc["Iin_sum"] / max(acc["Iin_steps"], 1),
             "input_current_over_target": (acc["Iin_sum"] / max(acc["Iin_steps"], 1)) / I_TARGET,
             "mean_abs_recurrent_current": acc["Irec_abs_sum"] / max(N * LIQ["num_sim_steps"], 1),
             "n_beats": N, "encode_seconds": t_enc, "simulate_seconds": t_sim}
    if y is not None:
        for c, nm in enumerate(CLASS_NAMES):
            if (y == c).any():
                stats[f"liquid_spikes_per_beat_{nm}"] = float(per_beat[y == c].mean())
    return out, stats

def liquid_raster(model, S, batch=256):
    """Full spike raster (B, 650, 343) with the same update as simulate(), on the device. Also returns the mean
    |recurrent current| and mean input current per neuron and step (input steps only)."""
    Win, Wr = model.input_rsv_weights, model.rsv_weights
    T = LIQ["num_sim_steps"]; out = np.zeros((len(S), T, N_NEUR), bool); irec = iin = 0.0; tin = 1
    with torch.no_grad():
        for s in range(0, len(S), batch):
            x = torch.from_numpy(np.ascontiguousarray(S[s:s + batch])).to(DEVICE).float()
            B, tin, Cin = x.shape
            spk = torch.zeros(B, N_NEUR, device=DEVICE); mem = torch.zeros(B, N_NEUR, device=DEVICE)
            zero_in = torch.zeros(B, Cin, device=DEVICE)
            r = torch.zeros(B, T, N_NEUR, dtype=torch.bool, device=DEVICE)
            for t in range(T):
                I_in = (x[:, t, :] if t < tin else zero_in) @ Win
                I_rec = spk @ Wr
                spk, mem = model.lif((I_in + I_rec) * model.dt, mem)
                r[:, t] = spk > 0
                irec += float(I_rec.abs().mean()) * B
                if t < tin:
                    iin += float(I_in.mean()) * B
            out[s:s + B] = r.cpu().numpy()
    return out, irec / (len(S) * T), iin / (len(S) * tin)

def counts_to_rates(counts_rows):
    """Authors' features: window mean of 0/1 spikes in float64, stored float32."""
    return (counts_rows.astype(np.float64) / WIN).astype(np.float32)


## 9. Replication functions reused

In [ ]:
# From the replication notebook: the split names (verbatim), and its readout (verbatim, except that the power
# of the class weights is a parameter; 0.6, the paper's, is the default and gives the same fit).
def parse_split(s):
    m = re.match(r"cv-r(\d+)f(\d+)", s)
    return (int(m.group(1)), int(m.group(2))) if m else None

def fit_readout_p(Xtr_fn, ntr, ytr, Xev_fn, nev, loss, rng_seed, yev_for_curve=None, curve_sub=4000, prescale=False, n_jobs=None,
                  prepared=None, cancel=None, epochs=None, weight_power=0.6):
    """Authors' readout: StandardScaler (batched partial_fit) + SGDClassifier(hinge|log_loss, l2, alpha=1e-4,
    optimal LR, average=True, random_state=2026, soft class weights ^0.6), 15 epochs, batch 4096,
    shuffling with np.random.default_rng(rng_seed). Batches are scaled on the fly as in the authors' code, or, when
    prescale=True and RAM allows, the whole matrix is scaled once (row-wise identical values, faster).
    prepared=(scaler, XS) reuses a scaler / scaled matrix shared by several fits on the same training rows.
    Xtr_fn(positions) / Xev_fn(positions) return float32 feature rows."""
    bs = 4096
    scaler, XS = prepared if prepared is not None else prepare_train(Xtr_fn, ntr, prescale)
    tf = lambda X: scaler.transform(X).astype(np.float32)
    get_tr = (lambda p: XS[p]) if XS is not None else (lambda p: tf(Xtr_fn(p)))
    clf = SGDClassifier(loss=loss, penalty="l2", alpha=1e-4, fit_intercept=True, learning_rate="optimal",
                        max_iter=1, tol=None, class_weight=soft_class_weight(ytr, weight_power), average=True, random_state=2026,
                        n_jobs=n_jobs)   # n_jobs only parallelises the 3 one-vs-rest problems: results are bit-identical
    classes = np.unique(ytr)
    rng = np.random.default_rng(rng_seed)
    sub = np.sort(np.random.default_rng(0).choice(ntr, min(curve_sub, ntr), replace=False))
    Xsub = get_tr(sub)
    def predict_eval():
        preds, scores = [], []
        for i in range(0, nev, bs):
            Xe = tf(Xev_fn(np.arange(i, min(i + bs, nev))))
            preds.append(clf.predict(Xe)); scores.append(clf.decision_function(Xe).astype(np.float32))
        return np.concatenate(preds), np.concatenate(scores)
    curve = []
    for ep in range(epochs or READOUT_EPOCHS):          # epochs: only the core search uses fewer
        idx = rng.permutation(ntr)
        for i in range(0, ntr, bs):
            if cancel is not None and cancel.is_set():
                raise Cancelled()
            sel = idx[i:i + bs]
            Xb = get_tr(sel)
            if ep == 0 and i == 0:
                clf.partial_fit(Xb, ytr[sel], classes=classes)
            else:
                clf.partial_fit(Xb, ytr[sel])
        row = {"epoch": ep + 1, "train_sub_macro_f1": compute_metrics(ytr[sub], clf.predict(Xsub))["macro_f1"]}
        if loss == "hinge":
            dec = clf.decision_function(Xsub)
            onehot = -np.ones_like(dec); onehot[np.arange(len(sub)), ytr[sub]] = 1
            row["train_sub_hinge_loss"] = float(np.maximum(0, 1 - onehot * dec).mean())
        if yev_for_curve is not None and (ep + 1) in CURVE_EVAL_EPOCHS:
            row["eval_macro_f1"] = compute_metrics(yev_for_curve, predict_eval()[0])["macro_f1"]
        curve.append(row)
    y_pred, scores = predict_eval()
    return scaler, clf, y_pred, scores, curve


## 10. Shared task code (the fixed liquid, features, readout, statistics)

In [ ]:
# Shared by the four task notebooks.
# The liquid is the paper's with the three fixes: Gamma weights drawn as the paper describes, A_in measured on each
# split's training rows (Eq. 12-13 as described), and the reset of Eqs. 17-19 (lsm2's simulator; the authors' snnTorch
# neuron subtracts once more while the stored voltage stays above 1). Everything else is config 11: morphological
# filter, robust normalisation, Multi-Threshold coding (200 levels), 7x7x7 liquid, windowed firing rates, the authors'
# linear SVM readout, the authors' splits (DS2 and the 3 x 4 grouped CV of DS1) and readout seeds.
import threading
from concurrent.futures import ThreadPoolExecutor
from lsm2 import config as LC, liquid as LL, sim as LS, stats as LSt

LS.USE_CUDA_GRAPHS = LS.USE_CUDA_GRAPHS and DEVICE.type == "cuda"
ENC = "multi"
TASK_DIR = os.path.join(WORK_DIR, TASK)
os.makedirs(TASK_DIR, exist_ok=True)
SFX = "_smoke" if SMOKE_TEST else ""
SPLITS_CV = [f"cv-r{r}f{f}" for r, f in CV_SPLITS]
ALL_SPLITS = ["final"] + SPLITS_CV
D1, D2 = DATA["morph"]["DS1"], DATA["morph"]["DS2"]

def tpath(name):
    return os.path.join(TASK_DIR, f"{name}{SFX}")

def geometry(n_samples):
    """(windows, steps) for an input of n_samples steps, by the paper's rule: 577 samples -> 32 windows of 20 steps,
    650 steps simulated (the windows cover the input, the last 10 steps are simulated but not read)."""
    nwin = math.ceil(n_samples / WIN) + 3
    return nwin, nwin * WIN + 10

assert geometry(D1["X"].shape[1]) == (NWIN, LIQ["num_sim_steps"]), "window rule does not give the paper's 32 x 20 / 650"


# ---------------------------------------------------------------- front end and splits (the replication's)
def front_end(split, cast, X1=None, X2=None, folds=None):
    """Robust normalisation for one split. cast=True: the replication's feature path (float32 also for the final
    split); cast=False: its liquid path (the final split stays float64, as execute_lsm passes it to the encoder)."""
    X1 = D1["X"] if X1 is None else X1
    X2 = D2["X"] if X2 is None else X2
    folds = FOLD_BY if folds is None else folds
    if split == "final":
        X1n, X2n, normp = normalize_final(X1, X2, "robust")
        if cast:
            X1n, X2n = np.asarray(X1n, np.float32), np.asarray(X2n, np.float32)
        return X1n, X2n, np.arange(len(X1)), normp
    f = folds[parse_split(split)]
    X1n, normp = normalize_cv(X1, f["train_idx"], "robust")
    return X1n, None, f["train_idx"], normp

def split_rows(split, n1, n2, folds=None):
    """Training rows (DS1), evaluation rows (DS2 for the final split, DS1 otherwise), readout seed (the authors')."""
    if split == "final":
        return np.arange(n1), np.arange(n2), 2026
    rep, fold = parse_split(split)
    f = (FOLD_BY if folds is None else folds)[(rep, fold)]
    return f["train_idx"], f["val_idx"], 2026 + rep * 10 + fold


# ---------------------------------------------------------------- features
def window_counts(Xn, nwin, batch=2048):
    """The replication's encoder_window_counts with nwin windows: Multi-Threshold spikes counted per 20 steps."""
    C = 200
    out = np.zeros((len(Xn), nwin, C), np.uint8)
    for s in range(0, len(Xn), batch):
        S = encode(Xn[s:s + batch], ENC)
        T = min(S.shape[1], nwin * WIN)
        Sp = np.zeros((S.shape[0], nwin * WIN, C), np.uint8); Sp[:, :T] = S[:, :T]
        out[s:s + batch] = Sp.reshape(S.shape[0], nwin, WIN, C).sum(2)
    return out

def rr_counts(rr_rows, thr, T_in, nwin):
    """The replication's rr_window_counts with nwin windows (the RR channels, counted per window)."""
    active = (rr_rows[:, rr_liq_feats(rr_rows)][:, :, None] >= thr[None]).reshape(len(rr_rows), -1)
    steps = np.arange(nwin * WIN)
    per_window = ((steps % RR_PERIOD == 0) & (steps < T_in)).reshape(nwin, WIN).sum(1)
    return (per_window[None, :, None] * active[:, None, :]).astype(np.uint8)

def liquid_model(X1n, tr_rows, seed, rr1=None, rr_thr=None):
    """The paper's liquid (the authors' class) with the fixed input rule: A_in measured on the training rows,
    Gamma weights as described. With RR, the 30 RR channels are extra inputs (and count in A_in)."""
    ain = measure_ain(X1n, tr_rows, ENC, rr=rr1, rr_thr=rr_thr)
    mu_in = I_TARGET / (ain * LIQ["P_in"])
    n_in = 200 + (len(rr_thr) * RR_LEVELS if rr_thr is not None else 0)
    return build_liquid(n_in, mu_in, seed, True), float(ain), float(mu_in)

def simulate_x(model, Xn, T, nwin, rr=None, rr_thr=None, recurrence=True, quirk=False):
    """Window spike counts (n, nwin * 343), window-major as the replication stores them. lsm2's simulator on the
    authors' weights, voltages start at 0. quirk=True: the authors' snnTorch reset (used only by the check that
    this path reproduces the replication's stored runs); quirk=False: Eqs. 17-19."""
    liq = LL.from_authors_model(model, snn_quirk=bool(quirk))
    sm = LS.Simulator(liq, device=DEVICE, dtype=torch.float32, g=1.0, in_scale=1.0, recurrence=recurrence)
    n, N = len(Xn), liq.N
    out = np.zeros((n, nwin * N), np.uint8)
    spikes, t0 = 0.0, time.time()
    for s in range(0, n, NEW_SIM_BATCH):
        x = encode_t(Xn[s:s + NEW_SIM_BATCH], ENC)
        if rr is not None:
            x = torch.cat([x, torch.from_numpy(rr_channels(rr[s:s + NEW_SIM_BATCH], rr_thr, x.shape[1])).to(DEVICE)], dim=2)
        o = sm.run(x, np.zeros((len(x), N)), T, rec={"windows": (WIN, nwin)})
        w = o["windows"]
        spikes += float(w.sum())
        out[s:s + len(x)] = w.clamp(max=255).to(torch.uint8).reshape(len(x), -1).cpu().numpy()
    rate = spikes / (n * N * nwin * WIN * LC.SAMPLE_MS / 1000.0)
    return out, {"rate_hz": rate, "sim_seconds": time.time() - t0}

def rates_fn(C):
    """Features of a counts array (n, ...): firing rate per window and neuron (counts / 20), float32."""
    return lambda r: counts_to_rates(C[r].reshape(len(r), -1))

def elm_fns(c1, c2, model, tr_rows):
    """The replication's random-features control: window counts x the liquid's input weights, centred on the
    training mean, ReLU (no time dynamics, no recurrence)."""
    Wf = model.input_rsv_weights.detach().cpu().numpy().astype(np.float32)
    nwin = c1.shape[1]
    proj = lambda c: np.einsum("nwc,ck->nwk", c.astype(np.float32), Wf).reshape(len(c), -1)
    tot = np.zeros(nwin * N_NEUR, np.float64)
    for i in range(0, len(tr_rows), 4096):
        tot += proj(c1[tr_rows[i:i + 4096]]).sum(0)
    mean = (tot / len(tr_rows)).astype(np.float32)
    F1 = lambda r: np.maximum(proj(c1[r]) - mean, 0)
    F2 = (lambda r: np.maximum(proj(c2[r]) - mean, 0)) if c2 is not None else None
    return F1, F2


# ---------------------------------------------------------------- readout and the fit queue
def readout(F_tr, tr, y_tr_all, F_ev, ev, y_ev_all, rng_seed, weight_power=0.6, prepared=None):
    """The authors' readout (linear SVM) on rows tr -> rows ev. F_tr / F_ev map row indices to features."""
    ftr = lambda p: F_tr(tr[p]); fev = lambda p: F_ev(ev[p])
    _, _, yp, sc, _ = fit_readout_p(ftr, len(tr), y_tr_all[tr], fev, len(ev), "hinge", rng_seed, prescale=PRESCALE,
                                    n_jobs=1, prepared=prepared, weight_power=weight_power)
    m = compute_metrics(y_ev_all[ev], yp)
    return {k: v for k, v in m.items() if k != "confusion_matrix"}, m.get("confusion_matrix"), yp, sc

def prepare(F_tr, tr):
    """A scaler (and the scaled training matrix) shared by several fits on the same training rows."""
    return prepare_train(lambda p: F_tr(tr[p]), len(tr), PRESCALE)

class FitQueue:
    """Readout fits run side by side (FIT_THREADS); one row per fit in TASK_DIR/rows.json, saved after every fit,
    so a restarted session skips what is done."""
    def __init__(self, keys):
        self.keys, self.path = keys, tpath("rows") + ".json"
        self.rows = json.load(open(self.path)) if os.path.exists(self.path) else []
        self.done = {self.key(r) for r in self.rows}
        self.lock = threading.Lock()
        self.ex = ThreadPoolExecutor(max(1, FIT_THREADS))
        self.futs = []
    def key(self, r):
        return tuple(r.get(k) for k in self.keys)
    def is_done(self, **meta):
        return self.key(meta) in self.done
    def submit(self, meta, job):
        if self.is_done(**meta):
            return
        def run():
            t0 = time.time()
            m, cm, _, _ = job()
            row = {**meta, **m, "confusion_matrix": cm, "fit_seconds": time.time() - t0, "session": SESSION_TAG}
            with self.lock:
                self.rows.append(row); self.done.add(self.key(row))
                jdump(self.rows, self.path)
            log("    " + ", ".join(f"{k} {meta[k]}" for k in self.keys) +
                f": Macro-F1 {m['macro_f1']:.4f}, SVEB recall {m['recall_SVEB']:.3f}, SVEB F1 {m['f1_SVEB']:.3f}, "
                f"VEB recall {m['recall_VEB']:.3f} ({time.time() - t0:.0f} s)")
        self.futs.append(self.ex.submit(run))
    def wait(self, max_pending=0):
        """Block until at most max_pending fits are pending (keeps at most a few states in memory)."""
        while True:
            for f in [f for f in self.futs if f.done()]:
                f.result()                                   # re-raises a failed fit here
                self.futs.remove(f)
            if len(self.futs) <= max_pending:
                return
            time.sleep(0.5)

def budget_left():
    if time.time() - SESSION_START > SESSION_BUDGET_HOURS * 3600:
        log(f"{TASK}: session budget reached; run the notebook again to continue")
        return False
    return True


# ---------------------------------------------------------------- comparisons and reporting
def as_runs(df):
    """A table in lsm2.stats' format (spec_split, spec_seed, metrics)."""
    if df is None or len(df) == 0:
        return None
    d = df.rename(columns={"split": "spec_split", "seed": "spec_seed"}).copy()
    d["spec_seed"] = pd.to_numeric(d["spec_seed"], errors="coerce")
    return d

def compare_family(name, items):
    """items: [(label, table A, table B)] -> lsm2.stats' paired comparisons (Nadeau-Bengio over the CV splits,
    seeds averaged per split; DS2 difference by seed), Holm within the family, verdicts."""
    res = [(lab, LSt.compare(as_runs(a), as_runs(b), require_seed_sign=True) if a is not None and b is not None and len(a) and len(b) else None)
           for lab, a, b in items]
    return pd.DataFrame(LSt.family(name, res))

def means_table(groups):
    """groups: [(label, table)] -> mean Macro-F1 and per-class recall / F1, over the CV runs and over the DS2 runs."""
    rows = []
    for lab, df in groups:
        if df is None or len(df) == 0:
            rows.append({"model": lab, "cv_runs": 0, "ds2_runs": 0}); continue
        cv = df[df.split.astype(str).str.startswith("cv-")]; f = df[df.split == "final"]
        r = {"model": lab, "cv_runs": int(len(cv)), "ds2_runs": int(len(f))}
        for k in ("macro_f1", "recall_SVEB", "f1_SVEB", "precision_SVEB", "recall_VEB", "f1_VEB"):
            r[f"cv_{k}"] = float(cv[k].mean()) if len(cv) else np.nan
            r[f"ds2_{k}"] = float(f[k].mean()) if len(f) else np.nan
        rows.append(r)
    return pd.DataFrame(rows)

def md_table(df, cols, fmt=None):
    fmt = fmt or {}
    cols = [c for c in cols if c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        cells = []
        for c in cols:
            v = r.get(c)
            if isinstance(v, (float, np.floating)):
                cells.append("" if not np.isfinite(v) else fmt.get(c, "{:.4f}").format(v))
            else:
                cells.append("" if v is None or (isinstance(v, float) and np.isnan(v)) else str(v))
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)

MEANS_COLS = ["model", "cv_runs", "cv_macro_f1", "cv_recall_SVEB", "cv_f1_SVEB", "cv_recall_VEB", "ds2_runs",
              "ds2_macro_f1", "ds2_recall_SVEB", "ds2_f1_SVEB", "ds2_precision_SVEB", "ds2_recall_VEB"]
COMP_COLS = ["comparison", "n_splits", "cv_diff", "ci_low", "ci_high", "p_holm", "ds2_diff", "ds2_diff_recall_SVEB", "verdict"]

def log_comparisons(C):
    for _, r in C.iterrows():
        if pd.notna(r.get("cv_diff")):
            log(f"{TASK}: {r['comparison']}: CV {r['cv_diff']:+.4f} [{r['ci_low']:+.4f}, {r['ci_high']:+.4f}], "
                f"DS2 {r['ds2_diff']:+.4f}: {r['verdict']}")
        else:
            log(f"{TASK}: {r['comparison']}: missing")


# ---------------------------------------------------------------- check against the replication's stored runs
def stored_runs():
    """Config 11's stored runs of REPLICATION_DIR that the check uses (fixed-mode liquid, random features, no liquid)."""
    if not REPLICATION_DIR:
        return None
    root = os.path.expanduser(REPLICATION_DIR)
    if not os.path.isdir(os.path.join(root, "runs")):
        log(f"{TASK} check: no runs/ folder in {root}; the check is skipped")
        return None
    rows = []
    for d in sorted(glob.glob(os.path.join(root, "runs", "*"))):
        if os.path.basename(d).startswith("SMOKE__") != bool(SMOKE_TEST):
            continue                       # a smoke test is only checked against smoke runs, a real run against real ones
        try:
            c = json.load(open(os.path.join(d, "config.json")))
        except Exception:
            continue
        s = c.get("spec", {})
        if s.get("filter") != "morph" or s.get("norm") != "robust" or s.get("encoder") != ENC or s.get("rr_set") is not None:
            continue
        lsm_ok = (s.get("kind") == "lsm" and c.get("group") in ("main_cv", "main_final") and s.get("mode") == "fixed"
                  and s.get("readout") == "svm" and not s.get("rr_in_liquid") and s.get("recurrence") is True
                  and s.get("gain") == 1 and s.get("lambda_val") == 0.8 and s.get("ain_override") is None)
        feat_ok = s.get("kind") == "feat" and s.get("model") in ("elm", "no_liquid") and c.get("group") in ("ablation_cv", "ablation_final")
        if not (lsm_ok or feat_ok):
            continue
        try:
            m = json.load(open(os.path.join(d, "metrics.json")))
        except Exception:
            continue
        rows.append({"run_dir": d, "model": "liquid" if lsm_ok else s["model"], "seed": s.get("seed"),
                     "split": s.get("split"), "macro_f1": m.get("macro_f1")})
    return pd.DataFrame(rows) if rows else None

def check_against_replication():
    """Recompute three stored runs of the replication (first seed, first CV split) through this notebook's code:
    the liquid with the authors' reset (quirk), random features, no liquid. Same predictions, beat by beat, and the
    same decision scores mean the front end, A_in, weights, simulator, features and readout are the replication's;
    the task's runs then differ from it only by the reset of Eqs. 17-19."""
    p = tpath("check") + ".json"
    if os.path.exists(p):
        v = json.load(open(p)); log(f"{TASK} check (saved): {v['verdict']}"); return v
    R = stored_runs()
    seed, split = TASK_SEEDS[0], SPLITS_CV[0]
    out = {"seed": seed, "split": split, "checks": []}
    if R is None:
        out["verdict"] = "not run (no REPLICATION_DIR)"; jdump(out, p); log(f"{TASK} check: {out['verdict']}"); return out
    tr, ev, rs = split_rows(split, len(D1["y"]), len(D2["y"]))
    X1l, _, trl, _ = front_end(split, cast=False)
    X1f, _, _, _ = front_end(split, cast=True)
    c1 = window_counts(X1f, NWIN)
    def feats(model):
        if model == "liquid":
            mdl, _, _ = liquid_model(X1l, trl, seed)
            C1, _ = simulate_x(mdl, X1l, LIQ["num_sim_steps"], NWIN, quirk=True)
            return rates_fn(C1)
        if model == "elm":
            mdl, _, _ = liquid_model(X1f, tr, seed)
            return elm_fns(c1, None, mdl, tr)[0]
        return rates_fn(c1)
    for model in ("liquid", "elm", "no_liquid"):
        st = R[(R.model == model) & (R.split == split) & ((R.seed == seed) if model != "no_liquid" else R.seed.isna())]
        if st.empty:
            out["checks"].append({"what": model, "status": "no stored run"}); continue
        F = feats(model)
        m, _, yp, sc = readout(F, tr, D1["y"], F, ev, D1["y"], rs)
        z = np.load(os.path.join(st.run_dir.iloc[0], "predictions.npz"))
        if len(z["y_pred"]) != len(yp):
            out["checks"].append({"what": model, "status": "differs", "note": f"different beats: stored {len(z['y_pred'])}, here {len(yp)}"})
            log(f"{TASK} check: {model}: differs (the stored run evaluated {len(z['y_pred'])} beats, this one {len(yp)})")
            continue
        same = float(np.mean(z["y_pred"].astype(int) == np.asarray(yp).astype(int)))
        dmax = float(np.abs(z["scores"].astype(np.float64) - np.asarray(sc, np.float64)).max()) if same == 1.0 else np.nan
        ok = same == 1.0 and dmax < 1e-4
        out["checks"].append({"what": model, "status": "match" if ok else "differs", "same_predictions": same,
                              "largest_score_difference": dmax, "macro_f1": m["macro_f1"],
                              "stored_macro_f1": float(st.macro_f1.iloc[0])})
        log(f"{TASK} check: {model}: {'match' if ok else 'differs'} (same predictions {100 * same:.2f}%, "
            f"Macro-F1 {m['macro_f1']:.6f} vs stored {float(st.macro_f1.iloc[0]):.6f})")
    st_ = [c["status"] for c in out["checks"]]
    out["verdict"] = ("all match" if st_ and all(s == "match" for s in st_) else
                      "not run (no stored runs" + (" from a smoke test" if SMOKE_TEST else "") + ")" if all(s == "no stored run" for s in st_) else
                      "DIFFERS: see check.json (a GPU of another type can flip a few spikes; a large difference is a bug)")
    log(f"{TASK} check: {out['verdict']}")
    jdump(out, p)
    return out


## 11. Streaming: records, encoder, liquids, runs, states, readouts

In [ ]:
# Shared streaming code: records, the stream encoder, the liquids of every row, stream runs, states, readouts.
# The plan: "LSM-ECG plan: config 11 vs continuous liquids with memory" (Claude doc). Everything before and after the
# liquid is config 11's final path (task notebooks' code); only the liquid and how it sees the record change.
import gc, shutil, joblib
from dataclasses import replace as dc_replace
from lsm2 import stream as LSS, measure as LMe, gain as LGa, data as LDa, calib as LCa

TASK_DIR = WORK_DIR                          # the plan's files live in WORK_DIR itself
for _d in ("liquids", "states", "runs", "benchmark", "figures"):
    os.makedirs(os.path.join(WORK_DIR, _d), exist_ok=True)
CV_TMP = os.path.join(SCRATCH_DIR, "cv_states")    # CV states: fitted, then deleted
os.makedirs(CV_TMP, exist_ok=True)
SPAN = NWIN * WIN                            # 640 steps: the 32 readout windows
PRE = 288                                    # a beat's window starts 288 samples (0.8 s) before its R peak
P_ROWS = ("P1", "P2", "P3", "P4", "P5")
M_ROWS = ("M1", "M3")
ADAPTIVE = ("P3", "P4", "P5", "M3")
FROZEN_OF = {"P3f": "P3", "M3f": "M3"}
RUN_ROWS = [r for r in ROWS]
if SMOKE_TEST:
    LC.N_SUB_REFERENCE = 4                   # as the ytest smoke test: Maass's grid coarse, the numbers mean nothing
    LC.N_CAL_BEATS, LC.GEN_COPIES = 24, 6
    LC.CODE.update(twin_beats=4, memory_sequences=16)
    LC.G_COARSE, LC.G_LOW, LC.G_FINE_K = (1, 10, 100), (0.3,), 1
SEEDS_RUN = tuple(STREAM_SEEDS[:2]) if SMOKE_TEST else tuple(STREAM_SEEDS)
SPLITS_RUN = ["final"] + (SPLITS_CV if EXACT_CV else [])
STEP_S = LC.SAMPLE_MS / 1000.0

def wpath(*p):
    return os.path.join(WORK_DIR, *p)

def key_of(row, seed):
    return row if seed is None else f"{row}_s{seed}"


# ---------------------------------------------------------------- records (the replication's loader and filter)
RECS = {ds: list(DS_RECORDS[ds]) for ds in ("DS1", "DS2")}
DS_OF = {r: ds for ds in RECS for r in RECS[ds]}
SIG = {}

def load_records():
    """Every record through config 11's morphological filter, as build_set does it (cut in SMOKE)."""
    t0 = time.time()
    for r in RECS["DS1"] + RECS["DS2"]:
        if r in SIG:
            continue
        sig, _, _, fs = A03.load_ecg(os.path.join(DATA_DIR, r), target_fs=360)
        _, sig_f, _, _ = A_sig.morph_filter_baseline(sig, fs=360, k1_ratio=0.20, k2_ratio=0.35, k3_ratio=0.005, mode='reflect')
        sig_f = np.asarray(sig_f)
        SIG[r] = sig_f[:int(SMOKE_SECONDS * 360)] if SMOKE_TEST else sig_f
    log(f"records: {len(SIG)} filtered ({sum(len(s) for s in SIG.values()) / 360 / 3600:.2f} h of ECG, {time.time() - t0:.0f} s)")

def beats_of(ds):
    """Per record: its rows in DATA (ascending window start) and their window starts (R - 288)."""
    D = DATA["morph"][ds]
    out = {}
    for r in RECS[ds]:
        idx = np.where(D["pid"] == r)[0]
        st = D["pos"][idx].astype(np.int64) - PRE
        o = np.argsort(st, kind="stable")
        out[r] = (idx[o], st[o])
    return out

load_records()
BEATS = {ds: beats_of(ds) for ds in ("DS1", "DS2")}
N_BEATS = {ds: len(DATA["morph"][ds]["y"]) for ds in ("DS1", "DS2")}


# ---------------------------------------------------------------- normalisation and the stream encoder
_FE = {}
def fe(split):
    """front_end(split, cast=False): the liquid path (final: float64 with DS1's percentiles; CV: float32 with the
    split's training rows' percentiles). Cached for the split in use."""
    if split not in _FE:
        _FE.clear(); gc.collect()
        _FE[split] = front_end(split, cast=False)
    return _FE[split]

def norm_signal(sig, split):
    """A whole record through the same normalisation as its beats (elementwise, so equal sample for sample)."""
    p = fe(split)[3]
    if split == "final":
        return A_strat.minmax_apply(sig, p["p_low"], p["p_high"])
    return np.asarray(A_strat.robust_minmax_apply(sig, p["p_low"], p["p_high"]), np.float32)

class StreamEncoder:
    """Multi-threshold spikes (200 levels, config 11's encoder) of every stream, steps [a, b): a level spikes when a
    record sample's level test differs from the previous sample's; nothing at a record's first sample and nothing
    past its end (silence, as config 11 has after a beat). sigs: one normalised signal per record; rec_of / offset:
    per stream, its record and the record sample of its step 0."""
    def __init__(self, sigs, rec_of, offset, levels=200):
        dt = sigs[0].dtype
        lens = np.array([len(s) for s in sigs], np.int64)
        base = np.concatenate([[0], np.cumsum(lens)[:-1]]).astype(np.int64)
        self.flat = torch.from_numpy(np.concatenate(sigs)).to(DEVICE)
        self.base = torch.as_tensor(base[rec_of], device=DEVICE)
        self.last = torch.as_tensor(lens[rec_of] - 1, device=DEVICE)
        self.off = torch.as_tensor(np.asarray(offset, np.int64), device=DEVICE)
        self.thr = torch.from_numpy(np.linspace(0.0, 1.0, levels, dtype=dt)).to(DEVICE)
    def __call__(self, a, b):
        t = torch.arange(a - 1, b, device=DEVICE)
        idx = torch.minimum((self.off[:, None] + t[None, :]).clamp(min=0), self.last[:, None])
        above = self.flat[self.base[:, None] + idx][:, :, None] >= self.thr
        return (above[:, 1:] != above[:, :-1]).to(torch.uint8)


# ---------------------------------------------------------------- the liquids of every row
_BASE = {}
def paper_base(split, seed):
    """Config 11's liquid for a split (the authors' class, Gamma fix, A_in on the split's training rows) as lsm2
    runs it with the reset of Eqs. 17-19, plus its A_in and mu_in."""
    k = (split, seed)
    if k not in _BASE:
        X1n, _, trl, _ = fe(split)
        model, ain, mu = liquid_model(X1n, trl, seed)
        _BASE[k] = (LL.from_authors_model(model, snn_quirk=False), float(ain), float(mu))
        del model
    return _BASE[k]

def adapt_settings(row):
    a = dict(ADAPT_SALAJ)
    if row == "P5":
        a["tau_ms"] = (ADAPT_ONE_CLOCK_MS, ADAPT_ONE_CLOCK_MS)
    return a

def paper_row(row, split, seed, thr=None):
    """(liquid, g, input size, info) of a paper's-liquid row. thr: the frozen thresholds (P3f)."""
    base, ain, mu = paper_base(split, seed)
    liq = dc_replace(base, meta=dict(base.meta))
    if row in ("P3", "P4", "P5", "P3f"):
        LL.add_adaptation(liq, seed, **adapt_settings("P3" if row == "P3f" else row))
        liq.reset_to_thr = True                 # Salaj's reset: subtract the current threshold
    if row in ("P2", "P4"):
        liq = LSS.no_recurrence(liq)
    if row == "P3f":
        liq = LSS.frozen_copy(liq, thr)
    liq.meta.update(row=row, seed=int(seed), split=split)
    return liq, 1.0, 1.0, {"A_in": ain, "mu_in": mu}

M_CAL = None                                    # Maass's input size and gain (set by maass_calibration)
def maass_liquid(seed, adapt=False, n_channels=200):
    """Maass's liquid as in the ytest notebook (15 x 3 x 3, every factor), input to MAASS_INPUT_FRACTION of the
    neurons. Its recurrent structure depends only on the seed, so the 1-channel (Poisson) and 200-channel (ECG)
    versions of a seed are the same liquid."""
    liq = LL.build_candidate(LC.ALL_FACTORS, seed, LC.N_SUB_REFERENCE, n_channels, in_fraction=MAASS_INPUT_FRACTION)
    if adapt:
        LL.add_adaptation(liq, seed, **ADAPT_SALAJ)
    liq.meta["seed"] = int(seed)
    return liq

def maass_row(row, seed, thr=None):
    liq = maass_liquid(seed, adapt=row in ("M3", "M3f"))
    if row == "M3f":
        liq = LSS.frozen_copy(liq, thr)
    liq.meta.update(row=row, seed=int(seed), split="final")
    return liq, float(M_CAL["g"][str(seed)]), float(M_CAL["input_scale"]), {"input_scale": M_CAL["input_scale"],
                                                                            "g": M_CAL["g"][str(seed)]}

def save_liquid(key, liq, g, sc, info):
    p = wpath("liquids", f"{key}.pt")
    if os.path.exists(p):
        return
    d = {k: getattr(liq, k) for k in ("N", "is_exc", "pre", "post", "typ", "w", "U", "D_ms", "F_ms", "Win", "beta",
                                       "gap", "delay", "adapt_c", "adapt_tau_ms", "thr_base", "reset_to_thr",
                                       "reset_mode", "rest", "n_sub", "h_ms", "snn_quirk")}
    d.update({"g": g, "in_scale": sc, "meta": liq.meta})
    d.update(info)
    torch.save(d, p + ".tmp"); os.replace(p + ".tmp", p)


# ---------------------------------------------------------------- stream runs
def stream_plan(split, dss, pieces=1, warm_s=0.0, seconds=None):
    """The streams of every record of dss (all its beats): per stream, its record, offset, length, starts, own part,
    and the rows of its beats. seconds: only that much of each record (benchmark)."""
    sigs, recs, streams = [], [], []
    for ds in dss:
        for r in RECS[ds]:
            s = SIG[r] if seconds is None else SIG[r][:int(seconds * 360)]
            rows, st = BEATS[ds][r]
            keep = np.ones(len(st), bool) if seconds is None else st + 2 * PRE + 1 <= len(s)   # benchmark: whole beats
            rows, st = rows[keep], st[keep]
            ri = len(sigs)
            sigs.append(norm_signal(s, split)); recs.append((ds, r))
            for p in LSS.plan_streams(len(s), st, pieces=pieces, warmup=int(round(warm_s * 360)), tail=SPAN):
                p.update(rec=ri, ds=ds, record=r, rows=rows[p["beat_idx"]])
                streams.append(p)
    return sigs, recs, streams

def seg_len_for(sm, B):
    m = LSS.period(sm)
    s = min(SEGMENT_STEPS, max(m, (2 ** 28) // max(1, B * sm.N)))     # step-count buffers of about 0.25 GB
    return max(m, s // m * m)

class StateSink:
    """Writes each liquid's windows into its own (beats, 32 x N) uint8 array (an .npy file, memory-mapped), rows in
    DATA order. One sink per (liquid, data set)."""
    def __init__(self, path, n, N):
        self.path, self.tmp = path, path + ".tmp.npy"
        self.A = np.lib.format.open_memmap(self.tmp, mode="w+", dtype=np.uint8, shape=(n, NWIN * N))
        self.filled = np.zeros(n, bool)
    def put(self, rows, wc):
        self.A[rows] = wc.reshape(len(rows), -1)
        self.filled[rows] = True
    def close(self, require_all=True):
        self.A.flush()
        if require_all and not self.filled.all():
            raise RuntimeError(f"{self.path}: {int((~self.filled).sum())} beats were not written")
        del self.A
        os.replace(self.tmp, self.path)

def run_liquids(items, split, dss, stack=True, pieces=1, warm_s=0.0, seconds=None, sinks=None, counts=True,
                label=""):
    """Stream the records of dss through every liquid of items [(key, liq, g, in_scale)], together (stack) or one
    after another. sinks: {(key, ds): StateSink} (missing = windows not kept). Returns per key: per-record sums of
    the adaptation variable and spikes over the own steps (DS1 and DS2 records separately), steps, timing."""
    sigs, recs, streams = stream_plan(split, dss, pieces, warm_s, seconds)
    enc = StreamEncoder(sigs, [s["rec"] for s in streams], [s["offset"] for s in streams])
    groups = [items] if stack else [[it] for it in items]
    out, t_all = {}, time.time()
    for grp in groups:
        if len(grp) == 1:                        # a liquid on its own runs as itself, with its own g and input size
            S = grp[0][1]
            sm = LS.Simulator(S, device=DEVICE, dtype=torch.float32, g=grp[0][2], in_scale=grp[0][3])
            offs = [0, S.N]
        else:
            S = LSS.stack([it[1] for it in grp], g=[it[2] for it in grp], in_scale=[it[3] for it in grp])
            sm = LS.Simulator(S, device=DEVICE, dtype=torch.float32, g=1.0, in_scale=1.0)
            offs = S.meta["offsets"]
        u0 = np.full((len(streams), S.N), S.rest)
        def sink(i, ks, wc, grp=grp, offs=offs):
            s = streams[i]
            for j, it in enumerate(grp):
                k = sinks.get((it[0], s["ds"])) if sinks else None
                if k is not None:
                    k.put(s["rows"][ks], np.ascontiguousarray(wc[:, :, offs[j]:offs[j + 1]]))
        seg = seg_len_for(sm, len(streams))
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        o = LSS.run_streams(sm, enc, streams, seg, u0=u0, nwin=NWIN, win=WIN, ab=True, counts=counts, sink=sink)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        sec = time.time() - t0
        for j, it in enumerate(grp):
            sl = slice(offs[j], offs[j + 1])
            per = {}
            for ri, (ds, r) in enumerate(recs):
                ii = [i for i, s in enumerate(streams) if s["rec"] == ri]
                per[r] = {"ds": ds, "steps": int(o["own_steps"][ii].sum()), "ab_sum": o["ab_sum"][ii][:, sl].sum(0),
                          "spike_sum": o["spike_sum"][ii][:, sl].sum(0) if counts else None}
            out[it[0]] = {"records": per, "seconds": sec, "steps": int(o["steps"]), "streams": len(streams),
                          "stacked_with": len(grp), "seg_len": seg}
        log(f"    {label}{'stacked ' if len(grp) > 1 else ''}{', '.join(it[0] for it in grp)}: {len(streams)} streams x "
            f"{o['steps']} steps in {sec:.0f} s ({1e6 * sec / (o['steps'] * len(streams) * len(grp)):.2f} us per stream-step)")
        del sm, S, o
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    del enc
    return out

def frozen_from(res, liq, ds_rows=None, records=None):
    """The time-averaged threshold of every neuron of liq over the records chosen (default: every DS1 record)."""
    per = res["records"]
    recs = [r for r, v in per.items() if (v["ds"] == "DS1" if records is None else r in records)]
    if records is None:
        assert all(per[r]["ds"] == "DS1" for r in recs)
    ab = np.sum([per[r]["ab_sum"] for r in recs], 0); steps = sum(per[r]["steps"] for r in recs)
    return LSS.frozen_thresholds(liq, ab, steps), recs

def save_thresholds(key, res, liq, thr, recs, tag=""):
    per = res["records"]
    names = sorted(per)
    p = wpath("liquids", f"{key}_thresholds{tag}.npz")
    np.savez(p + ".tmp.npz", records=np.array(names),
             ab_sum=np.stack([per[r]["ab_sum"] for r in names]), steps=np.array([per[r]["steps"] for r in names]),
             adapt_c=np.zeros(liq.N) if liq.adapt_c is None else liq.adapt_c, frozen=thr,
             averaged_over=np.array(recs))
    os.replace(p + ".tmp.npz", p)                # a half-written file is never taken as finished

def liquid_stats(key, res, liq, extra=None):
    """Firing by neuron type, silent share and adaptation level on the DS1 records (own steps)."""
    per = res["records"]
    d1 = [r for r, v in per.items() if v["ds"] == "DS1"]
    steps = sum(per[r]["steps"] for r in d1)
    out = {"key": key, "records": len(d1), "seconds_of_ecg": steps * STEP_S, **(extra or {})}
    if per[d1[0]]["spike_sum"] is not None:
        sp = np.sum([per[r]["spike_sum"] for r in d1], 0)
        hz = sp / (steps * STEP_S)
        E = np.asarray(liq.is_exc, bool)
        out.update(rate_hz=float(hz.mean()), rate_E_hz=float(hz[E].mean()), rate_I_hz=float(hz[~E].mean()),
                   silent_share=float((sp == 0).mean()))
    if liq.adapt_c is not None:
        ab = np.sum([per[r]["ab_sum"] for r in d1], 0) / steps
        a = np.asarray(liq.adapt_c) > 0
        out.update(n_adaptive=int(a.sum()), n_adaptive_inhibitory=int((a & ~np.asarray(liq.is_exc, bool)).sum()),
                   mean_threshold_adaptive=float((1 + np.asarray(liq.adapt_c)[a] * ab[a]).mean()),
                   max_threshold_adaptive=float((1 + np.asarray(liq.adapt_c)[a] * ab[a]).max()))
    if liq.thr_base is not None:
        out["mean_frozen_threshold"] = float(np.asarray(liq.thr_base).mean())
    return out


# ---------------------------------------------------------------- states files
def state_path(key, ds):
    return wpath("states", f"{key}__{ds}.npy")

def states_done(key, dss=("DS1", "DS2")):
    return all(os.path.exists(state_path(key, ds)) for ds in dss)

def load_states(key, ds):
    return np.load(state_path(key, ds), mmap_mode="r")

def save_beats_info():
    """Per beat (DATA order): record, R sample, label, patient, and the gaps used only by the probes."""
    for ds in ("DS1", "DS2"):
        p = wpath("states", f"beats__{ds}.npz")
        if os.path.exists(p):
            continue
        D = DATA["morph"][ds]
        pre = D["rr"][:, RR_NAMES.index("pre")]
        np.savez(p + ".tmp.npz", record=D["pid"], r_sample=D["pos"], label=D["y"], patient=D["pid"], prev_gap_s=pre,
                 mean10_gap_s=D["rr"][:, RR_NAMES.index("local10")], rec_mean_gap_s=pre / D["rr_mean_ratio"])
        os.replace(p + ".tmp.npz", p)

def cv_path(key, split):
    return os.path.join(CV_TMP, f"{key}__{split}.npy")


# ---------------------------------------------------------------- readouts
Q = FitQueue(["row", "seed", "split"])

def readout_m(F_tr, tr, y_tr_all, F_ev, ev, y_ev_all, rng_seed):
    """t20_common.readout plus the fitted model (to save it)."""
    ftr = lambda p: F_tr(tr[p]); fev = lambda p: F_ev(ev[p])
    scaler, clf, yp, sc, _ = fit_readout_p(ftr, len(tr), y_tr_all[tr], fev, len(ev), "hinge", rng_seed,
                                           prescale=PRESCALE, n_jobs=1, weight_power=0.6)
    m = compute_metrics(y_ev_all[ev], yp)
    return m, yp, sc, {"scaler": scaler, "clf": clf}

def extra_metrics(m, y, yp, sc, ev_rows):
    """SVEB score AUC (one vs rest) and DS2 SVEB F1 without record 232."""
    from sklearn.metrics import roc_auc_score
    y = np.asarray(y); yp = np.asarray(yp)
    try:
        m["auc_SVEB"] = float(roc_auc_score(y == 1, np.asarray(sc)[:, 1]))
    except ValueError:
        m["auc_SVEB"] = float("nan")
    D = DATA["morph"]["DS2"]
    if len(y) == len(D["y"]):
        k = D["pid"][ev_rows] != "232"
        m2 = compute_metrics(y[k], yp[k])
        m["f1_SVEB_no232"], m["macro_f1_no232"], m["recall_SVEB_no232"] = m2["f1_SVEB"], m2["macro_f1"], m2["recall_SVEB"]
    return m

def submit_fit(row, seed, split, F1, F2, after=None, rows=None):
    """One readout of config 11 (linear SVM, weight power 0.6) for a row on a split; DS2 runs saved in runs/.
    split "final_drop30s" with rows = (DS1 rows, DS2 rows): the keep-vs-drop refit (DS1 -> DS2 on those beats)."""
    if Q.is_done(row=row, seed=seed, split=split):
        if after:
            after()
        return
    tr, ev, rs = split_rows("final" if split.startswith("final") else split, N_BEATS["DS1"], N_BEATS["DS2"])
    if rows is not None:
        tr, ev = rows
    final = split.startswith("final")
    y_ev = DATA["morph"]["DS2"]["y"] if final else DATA["morph"]["DS1"]["y"]
    def job():
        try:
            t0 = time.time()
            m, yp, sc, mdl = readout_m(F1, tr, DATA["morph"]["DS1"]["y"], F2 if final else F1, ev, y_ev, rs)
            m = extra_metrics(m, y_ev[ev], yp, sc, ev) if split == "final" else m
            if final:
                rid = f"{key_of(row, seed)}__svm" + ("" if split == "final" else "_" + split[len("final_"):])
                save_run(rid, {"row": row, "seed": seed, "split": split, "readout": "svm", "weight_power": 0.6,
                               "mode": MODE}, "stream", m, ev, DATA["morph"]["DS2"]["pid"][ev], y_ev[ev], yp, sc, mdl,
                         None, None, {"fit_seconds": time.time() - t0})
            cm = m.pop("confusion_matrix")
            return m, cm, yp, sc
        finally:
            if after:
                after()
    Q.submit({"row": row, "seed": seed, "split": split}, job)

def remover(*paths):
    def f():
        for p in paths:
            try:
                os.remove(p)
            except OSError:
                pass
    return f


## 12. Maass's liquid: input size and gain (the ytest rules), M0 per beat

In [ ]:
# Maass's liquid: input size and gain, chosen once (for M1, adaptation off) with the ytest notebook's label-free
# rules, then reused by M0, M3 and M3f. The functions are the ytest notebook's (y_build's reference rate, PILOT form:
# no gain sweep on Maass's own input; y_ecg_setup's bisection and rank-measure sweep), with Y_ADAPTATION = False,
# Y_INPUT_FRACTION = MAASS_INPUT_FRACTION and seeds STREAM_SEEDS.
LC.BAND_HZ = (0.0, float("inf"))      # as the ytest notebook: the gain is chosen by (e)'s rank measure only
LC.HIGH_FRAC = 1.0
Y_CTX = LMe.Ctx(DEVICE, torch.float32, NEW_SIM_BATCH, log)
Y_START_STREAM = 101                  # per-beat starting voltages from (seed, stream, beat index), as the ytest runs

def sweep_rows(sw):
    """The gain sweep as a list of rows (JSON-safe)."""
    rows = []
    for g in sorted(sw["table"]):
        t = sw["table"][g]
        rows.append({"g": float(g), "abs_g": t["abs_g"], "rate_hz": t["rate_hz"], "sep": t["sep"], "gen": t["gen"],
                     "gap": t["gap"], "gap_seeds": t["gap_seeds"], "rate_seeds": t["rate_seeds"],
                     "twin_ratio": t.get("twin_ratio"), "gap_plain": t.get("gap_plain"), "stage": t["stage"]})
    return {"rows": rows, "chosen_g": sw["chosen_g"], "best_g": sw["best_g"], "peak_at_edge": sw["peak_at_edge"],
            "gap_peak_inside_grid": sw["gap_peak_inside_grid"], "extensions": sw["extensions"]}

def run_sweep(liqs, sets, label):
    """(e)'s measure at every g of the grid (old plan's grid rule), on every seed. Returns the sweep record."""
    def measure(g, s):
        r = LMe.measure_at_g(Y_CTX, liqs[s], g, liqs[s].in_scale_fixed, sets, s, extra_ranks=True)
        r["seed"] = s
        return r
    log(f"  gain sweep on {label}, seeds {list(liqs)}")
    return LGa.sweep(measure, list(liqs), log=log)

def y_ds1_normalised():
    """All DS1 beats through config 11's filter and robust normalisation (percentiles of all DS1; labels unused)."""
    D = DATA["morph"]["DS1"]
    mn, mx = A_strat.robust_minmax_fit(D["X"], low=0.1, high=99.9)
    return np.asarray(A_strat.robust_minmax_apply(D["X"], mn, mx), np.float32), {"p_low": float(mn), "p_high": float(mx)}

def enc_np(X):
    return np.concatenate([encode(X[s:s + 2000], "multi") for s in range(0, len(X), 2000)])

# ---- the ytest notebook's build check (Maass 2002, Figures 4-5), reported here, not a stop
SEGMENT_MS, SEGMENTS = 250.0, 4          # (c), Figures 4-5: four 250 ms segments
SEGMENT_TRAIN, SEGMENT_TEST = 1000, 500  # (c): 1000 training and 500 test inputs
SEGMENT_RATE_HZ = 20.0                   # (c): each template is a 20 Hz Poisson train
if SMOKE_TEST:
    SEGMENT_TRAIN, SEGMENT_TEST = 40, 20

def segment_inputs(seed, n):
    """(c)'s Figures 4-5 task: four 250 ms segments, each one of two templates (20 Hz Poisson), the whole train
    jittered by 4 ms (the 4 ms jitter of the plan's generalisation set). Labels: which template, per segment."""
    rng = np.random.default_rng([seed, 71])
    seg = int(round(SEGMENT_MS / LC.SAMPLE_MS))
    tmpl = rng.poisson(SEGMENT_RATE_HZ * LC.SAMPLE_MS / 1000.0, (SEGMENTS, 2, seg, 1)).astype(np.uint8)
    lab = rng.integers(0, 2, (n, SEGMENTS))
    X = np.zeros((n, SEGMENTS * seg, 1), np.uint8)
    for i in range(n):
        base = np.concatenate([tmpl[k, lab[i, k]] for k in range(SEGMENTS)])
        X[i] = LMe.jitter_spikes(base, np.random.default_rng([seed, 72, i]))
    return X, lab

def segment_task(liq, g, seed, recurrence):
    """Accuracy of a linear readout (least squares to +-1, as (c)) of the 30 ms-filtered state at the end of the
    1 s input, for each segment (oldest first). Trained on the first 1000 inputs, scored on the next 500."""
    n_tr, n_te = SEGMENT_TRAIN, SEGMENT_TEST
    X, lab = segment_inputs(seed, n_tr + n_te)
    T = X.shape[1]
    u0 = LS.start_voltages(liq, seed, range(len(X)), 4)
    o = LMe.simulate(Y_CTX, liq, g, liq.in_scale_fixed, X, u0, T, {"states_at": [T - 1], "neuron_counts": (0, T)},
                     recurrence=recurrence)
    Z = o["states_at"][:, 0].astype(np.float64)
    Zb = np.hstack([Z, np.ones((len(Z), 1))])
    acc = []
    for k in range(SEGMENTS):
        y = 2.0 * lab[:, k] - 1
        w = np.linalg.lstsq(Zb[:n_tr], y[:n_tr], rcond=None)[0]
        acc.append(float(((Zb[n_tr:] @ w >= 0) == (lab[n_tr:, k] == 1)).mean()))
    rate = float(o["neuron_counts"].sum() / (len(X) * liq.N * T * LC.SAMPLE_MS / 1000.0))
    return {"accuracy": acc, "rate_hz": rate}

def segment_gate(B):
    """The ytest notebook's stop point, here only reported: at Maass's own scale, on every seed, our Maass liquid
    (this notebook's build: input to MAASS_INPUT_FRACTION, no adaptation, STREAM_SEEDS) keeps the three older
    segments better with its internal connections on than off. Saved in liquids/maass_segment_task.json."""
    p = wpath("liquids", "maass_segment_task.json")
    if os.path.exists(p):
        out = json.load(open(p))
    else:
        t0 = time.time()
        out = {"seeds": {}, "input_fraction": MAASS_INPUT_FRACTION, "train": SEGMENT_TRAIN, "test": SEGMENT_TEST}
        for s in SEEDS_RUN:
            liq = maass_liquid(s, n_channels=1)
            g = float(liq.g_maass)
            on = segment_task(liq, g, s, True)
            off = segment_task(liq, g, s, False)
            older_on, older_off = float(np.mean(on["accuracy"][:-1])), float(np.mean(off["accuracy"][:-1]))
            out["seeds"][str(s)] = {"g": g, "on": on, "off": off, "older_on": older_on, "older_off": older_off,
                                    "on_beats_off": bool(older_on > older_off)}
            log(f"    segment task (g = {g:.3g}), seed {s}: on {np.round(on['accuracy'], 3).tolist()} "
                f"({on['rate_hz']:.2f} Hz), off {np.round(off['accuracy'], 3).tolist()} ({off['rate_hz']:.2f} Hz)")
        out["pass"] = all(v["on_beats_off"] for v in out["seeds"].values())
        out["seconds"] = time.time() - t0
        jdump(out, p)
    ck("maass_segment_task", out["pass"],
       **{f"seed_{s}": f"older segments on {v['older_on']:.3f} vs off {v['older_off']:.3f}" for s, v in out["seeds"].items()},
       note="the ytest notebook's build check; reported, the run goes on")
    B["maass_segment_task"] = out
    return out

def maass_calibration():
    p = wpath("liquids", "maass_calibration.json")
    if os.path.exists(p):
        return json.load(open(p))
    t0 = time.time()
    seeds = list(SEEDS_RUN)
    # 1. the reference: Maass's input-receiving neurons on his own 20 Hz Poisson input, recurrence off
    data = LDa.make_maass_data(seed=1)
    lq1 = {s: maass_liquid(s, n_channels=1) for s in seeds}
    target = LCa.input_neuron_rate(Y_CTX, list(lq1.values()), lq1[seeds[0]].in_scale_fixed, data.sets["sep"])
    log(f"  Maass's input: input-receiving neurons fire {target:.2f} Hz (recurrence off)")
    # 2. ECG beats (DS1, labels unused): the input size that gives that rate, then the gain by the rank measure
    xn, normp = y_ds1_normalised()
    setup = LDa.make_search_data("ecg_setup", xn, enc_np, seed=1)
    del xn
    liqs = {s: maass_liquid(s) for s in seeds}
    per_spike = liqs[seeds[0]].in_scale_fixed
    rate_per_spike = LCa.input_neuron_rate(Y_CTX, list(liqs.values()), per_spike, setup.sets["sep"])
    scale, rate_in = LCa.bisect_scale(Y_CTX, list(liqs.values()), setup.sets["sep"], target, log=log)
    for s in seeds:
        liqs[s].in_scale_fixed = float(scale)
    sw = run_sweep(liqs, setup.sets, "ECG beats")
    out = {"sweep": sweep_rows(sw), "input_rule": "match_maass_rate", "input_scale": float(scale),
           "input_neuron_rate_hz": rate_in, "input_scale_maass_per_spike": per_spike,
           "input_neuron_rate_at_maass_per_spike_hz": rate_per_spike, "input_neuron_rate_maass_hz": target,
           "A_in": setup.ain, "normalisation": normp, "g_maass": {str(s): float(liqs[s].g_maass) for s in seeds},
           "input_fraction": MAASS_INPUT_FRACTION, "adaptation": False, "seeds": seeds, "seconds": time.time() - t0}
    g = sw["chosen_g"]
    if g is None:
        out["g"] = {str(s): float(liqs[s].g_maass) for s in seeds}
        out["g_rule"] = "Maass's own scale (the rank measure gave no choice)"
    else:
        out["g"] = {str(s): float(g) for s in seeds}
        out["g_rule"] = "(e)'s rank measure on ECG beats"
    out["peak_at_edge"] = bool(sw["peak_at_edge"])
    if sw["peak_at_edge"]:
        out["note"] = "no interior optimum: the rank measure peaks at the grid's edge after its extensions; that g is used"
        log("  WARNING: " + out["note"])
    jdump(out, p)
    log(f"Maass calibration: input size {scale:.4g} (input neurons {rate_in:.2f} Hz vs Maass's {target:.2f} Hz), "
        f"g = {out['g']} ({out['g_rule']}), {time.time() - t0:.0f} s")
    return out

def per_beat_maass(seed, X1n, X2n):
    """M0: Maass's liquid (M1's input size and gain) one beat at a time, the ytest notebook's per-beat run
    (starting voltages from (seed, stream, beat)), config 11's front end. Writes states/M0_s<seed>__DS1/DS2.npy."""
    key = key_of("M0", seed)
    if states_done(key):
        return
    liq, g, sc, info = maass_row("M1", seed)
    save_liquid(key, liq, g, sc, info)
    sm = LS.Simulator(liq, device=DEVICE, dtype=torch.float32, g=g, in_scale=sc)
    for ds, X, stream in (("DS1", X1n, Y_START_STREAM), ("DS2", X2n, Y_START_STREAM + 1)):
        sink = StateSink(state_path(key, ds), len(X), liq.N)
        t0 = time.time(); spk = 0
        for s in range(0, len(X), NEW_SIM_BATCH):
            x = encode_t(X[s:s + NEW_SIM_BATCH], "multi")
            u0 = LS.start_voltages(liq, seed, np.arange(s, s + len(x)), stream)
            o = sm.run(x, u0, LIQ["num_sim_steps"], rec={"windows": (LC.WIN, LC.NWIN)})
            w = o["windows"].clamp(max=255).to(torch.uint8).cpu().numpy()
            spk += int(w.sum(dtype=np.int64))
            sink.put(np.arange(s, s + len(x)), w)
        sink.close()
        log(f"    M0 seed {seed} {ds}: {len(X)} beats, {spk / (len(X) * liq.N * SPAN * STEP_S):.2f} Hz, {time.time() - t0:.0f} s")


## 13. Checks

In [ ]:
# Checks (BENCHMARK and SMOKE; the machinery check again at the start of FULL). Results: check.json.
CHECK = {}

def ck(name, ok, report=False, **info):
    """Record one check: ok True / False / None (not run). report=True: a measurement, reported, not a pass/fail."""
    CHECK[name] = {"ok": None if (ok is None or report) else bool(ok), **({"report": True} if report else {}), **info}
    log(f"  check {name}: {'reported' if report else ('PASS' if ok else ('not run' if ok is None else 'FAIL'))}"
        + (f" ({', '.join(f'{k} {v}' for k, v in info.items() if not isinstance(v, (list, dict)))})" if info else ""))
    return ok

def fast_path_check():
    """The streaming loop is the simulator's fast path (one input step per call, a CUDA graph per step on a GPU).
    On this card, compare it bit for bit with the plain loop on a few inputs of the paper's liquid and of Maass's
    (the ytest notebook's check). A difference switches the graphs off; if the fast path still differs, stop."""
    if DEVICE.type != "cuda":
        LS.USE_CUDA_GRAPHS = False
        return ck("fast_path", True, note="no CUDA card: graphs not used")
    rng = np.random.default_rng(0)
    x = (rng.random((64, 120, 200)) < 0.006).astype(np.uint8)
    s0 = SEEDS_RUN[0]
    p3 = paper_row("P3", "final", s0)[0]
    p3f = LSS.frozen_copy(p3, 1.0 + 0.5 * np.random.default_rng(3).random(p3.N) * (np.asarray(p3.adapt_c) > 0))
    cases = [("paper", paper_base("final", s0)[0], 1.0, 1.0), ("paper + adaptation (P3)", p3, 1.0, 1.0),
             ("paper, fixed thresholds (P3f-like)", p3f, 1.0, 1.0), ("Maass", maass_liquid(s0), None, 0.01),
             ("Maass + adaptation (M3)", maass_liquid(s0, adapt=True), None, 0.01)]
    rec = {"windows": (20, 6), "states_at": [59, 119], "neuron_counts": (0, 120)}
    res = {}
    for mode in ("graphs", "fast"):
        LS.USE_CUDA_GRAPHS = mode == "graphs"
        ok = True
        for name, liq, g, sc in cases:
            u0 = LS.start_voltages(liq, 1, range(len(x)), 4)
            sm = LS.Simulator(liq, device=DEVICE, dtype=torch.float32, g=g if g else liq.g_maass * 2, in_scale=sc)
            try:
                a = sm.run(x, u0, 120, rec=rec, fast=False)
                b = sm.run(x, u0, 120, rec=rec, fast=True)
                eq = all(torch.equal(a[k], b[k]) for k in ("windows", "states_at", "neuron_counts"))
            except Exception as ex:          # noqa: BLE001
                eq = False; res[f"{name} ({mode}) error"] = repr(ex)[:300]
            res[f"{name} ({mode})"] = eq
            ok = ok and eq
        if ok:
            break
    LS.USE_CUDA_GRAPHS = ok and mode == "graphs"
    if not ok:
        raise RuntimeError(f"the simulator's fast path differs from its plain loop on this card: {res}")
    return ck("fast_path", True, graphs=LS.USE_CUDA_GRAPHS, cases=res)

def check_machinery():
    """With the authors' snnTorch reset, B1 seed 2026 reproduces the replication's stored config 11 run (DS2) exactly,
    and its beats, order and labels match the stored eval_index and y_true."""
    R = stored_runs()
    seed = STREAM_SEEDS[0]
    st = None if R is None else R[(R.model == "liquid") & (R.split == "final") & (R.seed == seed)]
    if st is None or st.empty:
        ck("beat_order_labels", None, note="no stored run")
        return ck("machinery", None, note="no stored replication run (REPLICATION_DIR)")
    z = np.load(os.path.join(st.run_dir.iloc[0], "predictions.npz"))
    D2 = DATA["morph"]["DS2"]
    ck("beat_order_labels", len(z["y_true"]) == len(D2["y"]) and np.array_equal(z["eval_index"], np.arange(len(D2["y"])))
       and np.array_equal(z["y_true"].astype(int), D2["y"]), stored=os.path.basename(st.run_dir.iloc[0]), beats=len(D2["y"]))
    X1n, X2n, trl, _ = fe("final")
    model, ain, mu = liquid_model(X1n, trl, seed)
    C1, _ = simulate_x(model, X1n, LIQ["num_sim_steps"], NWIN, quirk=True)
    C2, _ = simulate_x(model, X2n, LIQ["num_sim_steps"], NWIN, quirk=True)
    tr, ev, rs = split_rows("final", N_BEATS["DS1"], N_BEATS["DS2"])
    m, yp, sc, _ = readout_m(rates_fn(C1), tr, DATA["morph"]["DS1"]["y"], rates_fn(C2), ev, D2["y"], rs)
    del C1, C2
    if len(z["y_pred"]) != len(yp):
        return ck("machinery", False, note=f"different beats: stored {len(z['y_pred'])}, here {len(yp)}")
    same = float(np.mean(z["y_pred"].astype(int) == np.asarray(yp).astype(int)))
    dmax = float(np.abs(z["scores"].astype(np.float64) - np.asarray(sc, np.float64)).max())
    return ck("machinery", same == 1.0 and dmax < 1e-4, same_predictions=same, largest_score_difference=dmax,
              macro_f1=m["macro_f1"], stored_macro_f1=float(st.macro_f1.iloc[0]), stored=os.path.basename(st.run_dir.iloc[0]))

def check_t1_rows():
    """B1 (Eqs. 17-19) reproduces task 1's confusion matrices (3 DS2 + 36 CV rows), B2 its no-liquid ones (1 + 12)."""
    p = os.path.expanduser(T1_ROWS) if T1_ROWS else None
    if SMOKE_TEST and not (p and "smoke" in os.path.basename(p)):
        ck("B1_vs_t1", None, note="SMOKE: other beats than task 1's runs")
        return ck("B2_vs_t1", None, note="SMOKE: other beats than task 1's runs")
    if not p or not os.path.exists(p):
        ck("B1_vs_t1", None, note=f"task 1's rows.json not found at {T1_ROWS}")
        return ck("B2_vs_t1", None, note="task 1's rows.json not found")
    t1 = json.load(open(p))
    mine = {(r["row"], r["seed"], r["split"]): r for r in Q.rows}
    for row, model in (("B1", "liquid"), ("B2", "no_liquid")):
        ref = [r for r in t1 if r["model"] == model and r["split"] in SPLITS_RUN
               and (r["seed"] in SEEDS_RUN if row == "B1" else r["seed"] is None)]
        got, bad, missing = 0, [], 0
        for r in ref:
            m = mine.get((row, r["seed"], r["split"]))
            if m is None:
                missing += 1; continue
            got += 1
            if m["confusion_matrix"] != r["confusion_matrix"]:
                bad.append(f"{r['seed']}/{r['split']}")
        ck(f"{row}_vs_t1", (not bad and got > 0) if got else None, compared=got, differ=len(bad), missing=missing,
           which=bad[:12])

def check_encoding(n=3000):
    """The stream encoder's slice R - 288 .. R + 288 equals config 11's per-beat encoding bit for bit (after zeroing
    step 0), for DS1 and DS2 beats (final path, float64) and DS1 beats of the first CV split (float32)."""
    rng = np.random.default_rng(5)
    ok, info = True, {}
    for split, ds in (("final", "DS1"), ("final", "DS2"), (SPLITS_CV[0], "DS1")):
        X = fe(split)[0 if ds == "DS1" else 1]
        rows = np.sort(rng.choice(N_BEATS[ds], min(n, N_BEATS[ds]), replace=False))
        recs = RECS[ds]
        sigs = [norm_signal(SIG[r], split) for r in recs]
        ri = {r: i for i, r in enumerate(recs)}
        D = DATA["morph"][ds]
        bad = 0
        for c in range(0, len(rows), 500):
            sl = slice(c, c + 500)
            sub = StreamEncoder(sigs, [ri[D["pid"][i]] for i in rows[sl]], D["pos"][rows[sl]] - PRE)
            a = sub(0, 2 * PRE + 1); a[:, 0] = 0
            b = encode_t(X[rows[sl]], "multi")
            bad += int((a != b).any(2).any(1).sum())
        info[f"{split}/{ds}"] = f"{bad} of {len(rows)} beats differ"
        ok &= bad == 0
        del sigs
    return ck("encoding_alignment", ok, **info)

def window_alignment(n=500):
    """The stream path on one beat's slice, from rest, input zeroed from step 577 on and at step 0, gives B1's
    window counts (the per-beat simulator), for P1 of the first seed."""
    rng = np.random.default_rng(6)
    D = DATA["morph"]["DS2"]
    X1n, X2n, trl, _ = fe("final")
    rows = np.sort(rng.choice(N_BEATS["DS2"], min(n, N_BEATS["DS2"]), replace=False))
    seed = SEEDS_RUN[0]
    model, _, _ = liquid_model(X1n, trl, seed)
    ref, _ = simulate_x(model, X2n[rows], LIQ["num_sim_steps"], NWIN)
    liq, g, sc, _ = paper_row("P1", "final", seed)
    sigs = [norm_signal(SIG[r], "final") for r in RECS["DS2"]]
    ri = {r: i for i, r in enumerate(RECS["DS2"])}
    enc0 = StreamEncoder(sigs, [ri[D["pid"][i]] for i in rows], D["pos"][rows] - PRE)
    def enc(a, b):
        x = enc0(a, b)
        t = torch.arange(a, b, device=DEVICE)
        x[:, (t == 0) | (t >= 2 * PRE + 1)] = 0
        return x
    streams = [{"length": LIQ["num_sim_steps"], "starts": np.array([0]), "own_from": 0, "own_to": 1} for _ in rows]
    sm = LS.Simulator(liq, device=DEVICE, dtype=torch.float32, g=g, in_scale=sc)
    o = LSS.run_streams(sm, enc, streams, LIQ["num_sim_steps"], nwin=NWIN, win=WIN)
    got = np.stack([w[0].reshape(-1) for w in o["windows"]])
    nd = int((got != ref).any(1).sum())
    return ck("window_alignment", nd == 0, beats=len(rows), differ=nd,
              agrees_with_existing_simulator="one beat's 650-step input from rest")

class KeepSink:
    """Keeps the windows of the beats of a few records (for the equality checks), keyed by (data set, row)."""
    def __init__(self, ds, records):
        self.ds, self.records, self.A = ds, set(records), {}
    def put(self, rows, wc):
        pid = DATA["morph"][self.ds]["pid"]
        for r, w in zip(rows, wc):
            if pid[r] in self.records:
                self.A[(self.ds, int(r))] = w.copy()
    def close(self, require_all=False):
        pass

KEEP_RECORDS = {ds: RECS[ds][:4] for ds in ("DS1", "DS2")}

def keep_sinks(keys):
    """KeepSinks for the beats of the first 4 records of DS1 and of DS2."""
    return {(k, ds): KeepSink(ds, KEEP_RECORDS[ds]) for k in keys for ds in ("DS1", "DS2")}

def merge_kept(sinks, key):
    out = {}
    for ds in ("DS1", "DS2"):
        out.update(sinks[(key, ds)].A)
    return out

def compare_kept(a, b):
    """Share of beats whose windows differ (and of window counts); whether the differing beats sit in the first or
    the second half of each record's kept beats (a gap that spreads shows up as more in the second half)."""
    keys = sorted(set(a) & set(b))
    diff = {k: not np.array_equal(a[k], b[k]) for k in keys}
    nc = sum(int((a[k] != b[k]).sum()) for k in keys)
    tot = sum(a[k].size for k in keys)
    first = second = 0
    for ds in ("DS1", "DS2"):
        D = DATA["morph"][ds]
        for r in KEEP_RECORDS[ds]:
            ks = sorted((k for k in keys if k[0] == ds and D["pid"][k[1]] == r), key=lambda k: D["pos"][k[1]])
            h = len(ks) // 2
            first += sum(diff[k] for k in ks[:h]); second += sum(diff[k] for k in ks[h:])
    nb = sum(diff.values())
    return {"beats": len(keys), "beats_differ": int(nb), "share_beats_differ": nb / max(len(keys), 1),
            "counts_differ": int(nc), "share_counts_differ": nc / max(tot, 1),
            "differ_first_half": int(first), "differ_second_half": int(second)}

def segment_check(seconds=60.0):
    """A record run in several segments with carried state equals one segment (P3, the most state), every beat."""
    seed = SEEDS_RUN[0]
    liq, g, sc, _ = paper_row("P3", "final", seed)
    sm = LS.Simulator(liq, device=DEVICE, dtype=torch.float32, g=g, in_scale=sc)
    sigs, recs, streams = stream_plan("final", ("DS1",), seconds=seconds)
    streams = streams[:4]
    enc = StreamEncoder(sigs, [s["rec"] for s in streams], [s["offset"] for s in streams])
    T = max(s["length"] for s in streams)
    a = LSS.run_streams(sm, enc, streams, T + T % 2, nwin=NWIN, win=WIN, ab=True)
    b = LSS.run_streams(sm, enc, streams, 2 * 499, nwin=NWIN, win=WIN, ab=True)
    nd = sum(int((x != y).any(axis=(1, 2)).sum()) for x, y in zip(a["windows"], b["windows"]))
    nb = sum(len(x) for x in a["windows"])
    rel = float(np.abs(a["ab_sum"] - b["ab_sum"]).max() / max(np.abs(a["ab_sum"]).max(), 1e-30))
    return ck("segments_vs_one_run", nd == 0 and rel < 1e-9, beats=nb, differ=nd, segment_steps=998,
              adaptation_sum_rel_diff=rel)

def end_of_record_check():
    """Past a record's end the stream input is silent (as config 11 after a beat), and the last beats are cut."""
    r = RECS["DS2"][0]
    s = norm_signal(SIG[r], "final")
    enc = StreamEncoder([s], [0], [len(s) - 100])
    x = enc(0, 800)
    return ck("end_of_record", bool(x[:, 100:].sum() == 0), steps_after_end=700)

def one_neuron_check():
    """Salaj's rule as lsm2 runs it: one spike raises the threshold by 0.0567 / tau_a (s); a steady 10 Hz holds it
    near 1.57; the reset subtracts the current threshold."""
    from lsm2.liquid import Liquid
    def neuron(c, tau, push=3.0):
        e = np.zeros(0, np.int64); z = np.zeros(0)
        return Liquid(N=1, dims=(1, 1, 1), is_exc=np.array([True]), pos=np.zeros((1, 3)), pre=e, post=e, typ=e, w=z,
                      U=z, D_ms=z, F_ms=z, Ubar=np.ones(4), Win=np.array([[push]]), in_neurons=np.array([True]),
                      n_sub=1, h_ms=LC.SAMPLE_MS, beta=np.array([0.0]), tap=np.zeros(1), reset_mode="subtract",
                      reset_value=0.0, start_lo=0.0, start_hi=0.0, rest=0.0, gap=np.array([1]), clamp=False,
                      strict=False, schedule="ours", syn_model="static", delay=np.ones(4, int), cur_a=None, in_a=None,
                      in_scale_fixed=1.0, adapt_c=np.array([c]), adapt_tau_ms=np.array([tau]), reset_to_thr=True)
    c = 1.0 * ADAPT_SALAJ["beta"] / ADAPT_SALAJ["b0"]
    res = {}
    for tau in (1000.0, 6000.0):
        sm = LS.Simulator(neuron(c, tau), device="cpu", dtype=torch.float64, g=1.0, in_scale=1.0)
        x = np.zeros((1, 3, 1), np.uint8); x[0, 0, 0] = 1
        o = sm.run(x, np.zeros((1, 1)), 3, rec={"keep_state": True})
        rise = c * float(o["state"]["ab"][0, 0]) * math.exp(2 * LC.SAMPLE_MS / tau)
        res[f"rise_tau_{tau / 1000:g}s"] = rise
        res[f"want_tau_{tau / 1000:g}s"] = 0.0567 / (tau / 1000)
    sm = LS.Simulator(neuron(c, 1000.0), device="cpu", dtype=torch.float64, g=1.0, in_scale=1.0)
    per, T = 36, 36 * 400
    x = np.zeros((1, T, 1), np.uint8); x[0, ::per, 0] = 1
    w = np.zeros((1, T)); w[0, T // 2:] = 1
    o = sm.run(x, np.zeros((1, 1)), T, rec={"ab_weight": w, "step_counts": True})
    thr10 = 1 + c * float(o["ab_sum"][0, 0]) / w.sum()
    x1 = np.zeros((1, 1, 1), np.uint8); x1[0, 0, 0] = 1
    u1 = float(sm.run(x1, np.zeros((1, 1)), 1)["u_end"][0, 0])
    # two spikes in a row at tau_a = 100 ms: the second crosses the raised threshold 1 + c (1 - rho) / h and the reset
    # subtracts exactly that (Salaj et al. 2021, Eqs. 3 and 5), not 1 and not the threshold after its own rise
    sm2 = LS.Simulator(neuron(c, 100.0), device="cpu", dtype=torch.float64, g=1.0, in_scale=1.0)
    x2 = np.zeros((1, 2, 1), np.uint8); x2[0, :, 0] = 1
    o2 = sm2.run(x2, np.zeros((1, 1)), 2, rec={"step_counts": True})
    rho = math.exp(-LC.SAMPLE_MS / 100.0)
    thr2 = 1.0 + c * (1.0 - rho) / LC.SAMPLE_MS
    u2 = float(o2["u_end"][0, 0])
    ok = all(abs(res[f"rise_tau_{t}s"] - res[f"want_tau_{t}s"]) / res[f"want_tau_{t}s"] < 0.01 for t in ("1", "6")) \
        and abs(thr10 - 1.567) < 0.01 and abs(u1 - 2.0) < 1e-12 \
        and o2["step_counts"][0, :, 0].tolist() == [1, 1] and abs(u2 - (3.0 - thr2)) < 1e-12
    return ck("adaptation_one_neuron", ok, threshold_at_10Hz=round(thr10, 4), u_after_spike=u1,
              second_spike_threshold=round(thr2, 5), u_after_second_spike=round(u2, 5), want_after_second=round(3.0 - thr2, 5),
              **{k: round(v, 5) for k, v in res.items()})

def run_checks_core():
    one_neuron_check()
    check_encoding()
    window_alignment()
    segment_check()
    end_of_record_check()


## 14. The rows

In [ ]:
# The rows: B1 and B2 (per beat, task 1's code), P1-P5 and P3f (the paper's liquid, streaming), M0 (Maass per beat),
# M1, M3 and M3f (Maass streaming). Each writes its states and submits its readout fits; a finished step is skipped.

def run_B(split):
    """B1 (config 11, three fixes, Eqs. 17-19) per seed and B2 (no liquid) on one split, as task 1 ran them."""
    final = split == "final"
    want_b1 = "B1" in RUN_ROWS and any(not Q.is_done(row="B1", seed=s, split=split) for s in SEEDS_RUN)
    want_b1 = want_b1 or (final and "B1" in RUN_ROWS and not all(states_done(key_of("B1", s)) for s in SEEDS_RUN))
    want_b2 = "B2" in RUN_ROWS and not Q.is_done(row="B2", seed=None, split=split)
    if not (want_b1 or want_b2):
        return
    X1l, X2l, trl, _ = fe(split)
    wait = 2 * FIT_THREADS if final else CV_PENDING_FITS
    if want_b2:
        X1f, X2f, _, _ = front_end(split, cast=True)
        c1 = window_counts(X1f, NWIN); c2 = window_counts(X2f, NWIN) if final else None
        if final:
            submit_fit("B2", None, split, rates_fn(c1), rates_fn(c2))
        else:                                   # held on disk, not in RAM, until its fit is done
            p = cv_path("B2", split); np.save(p, c1)
            submit_fit("B2", None, split, rates_fn(np.load(p, mmap_mode="r")), None, after=remover(p))
        del X1f, X2f, c1, c2
        Q.wait(max_pending=wait)
    if not want_b1:
        return
    for seed in SEEDS_RUN:
        key = key_of("B1", seed)
        need_states = final and not states_done(key)
        if Q.is_done(row="B1", seed=seed, split=split) and not need_states:
            continue
        mdl, ain, mu = liquid_model(X1l, trl, seed)
        t0 = time.time()
        C1, st = simulate_x(mdl, X1l, LIQ["num_sim_steps"], NWIN)
        C2 = simulate_x(mdl, X2l, LIQ["num_sim_steps"], NWIN)[0] if final else None
        TIMES.setdefault("B1_sim", []).append({"split": split, "seed": seed, "seconds": time.time() - t0})
        log(f"    B1 seed {seed} {split}: A_in {ain:.4f}, firing {st['rate_hz']:.2f} Hz, {time.time() - t0:.0f} s")
        if final:
            liq = LL.from_authors_model(mdl, snn_quirk=False)
            save_liquid(key, liq, 1.0, 1.0, {"A_in": float(ain), "mu_in": float(mu)})
            for ds, C in (("DS1", C1), ("DS2", C2)):
                if not os.path.exists(state_path(key, ds)):
                    np.save(state_path(key, ds) + ".tmp.npy", C); os.replace(state_path(key, ds) + ".tmp.npy", state_path(key, ds))
            del C1, C2
            submit_fit("B1", seed, split, rates_fn(load_states(key, "DS1")), rates_fn(load_states(key, "DS2")))
        else:
            p = cv_path(key, split); np.save(p, C1); del C1
            submit_fit("B1", seed, split, rates_fn(np.load(p, mmap_mode="r")), None, after=remover(p))
        Q.wait(max_pending=wait)
TIMES = {}


def p_items(rows, split, seeds, thr=None):
    out = []
    for s in seeds:
        for row in rows:
            liq, g, sc, info = paper_row(row, split, s, None if thr is None else thr[key_of(row, s)])
            out.append((key_of(row, s), liq, g, sc, info))
    return out

def stream_rows(items, split, stack, pieces=1, warm_s=0.0, kind="paper"):
    """Stream the final split (DS1 + DS2 records, states saved) or a CV split (DS1 records, states in CV_TMP) for
    the items not done yet; returns the run results (threshold sums, spike sums) of the liquids run."""
    final = split == "final"
    dss = ("DS1", "DS2") if final else ("DS1",)
    def needed(it):
        key = it[0]
        row, seed = key.split("_s")[0], int(key.split("_s")[1])
        parent = row in ("P3", "M3") and FROZEN_OF.get(row + "f") and (row + "f") in RUN_ROWS
        if parent and not os.path.exists(thr_path(key, split)) and \
                (final or not Q.is_done(row=row + "f", seed=seed, split=split)):
            return True                                   # its frozen control needs this pass's thresholds
        if final:
            return not states_done(key)
        return not Q.is_done(row=row, seed=seed, split=split) and not os.path.exists(cv_path(key, split))
    todo = [it for it in items if needed(it)]
    if not todo:
        return {}
    sinks = {}
    for it in todo:
        for ds in dss:
            p = state_path(it[0], ds) if final else cv_path(it[0], split)
            sinks[(it[0], ds)] = StateSink(p, N_BEATS[ds], it[1].N)
    res = run_liquids([it[:4] for it in todo], split, dss, stack=stack, pieces=pieces, warm_s=warm_s, sinks=sinks,
                      label=f"{split}: ")
    parents_thresholds(res, todo, split)        # everything else is written first; the states files, whose presence
    for it in todo:                             # marks the pass finished, are closed (renamed) last
        key, liq = it[0], it[1]
        r = res[key]
        if final:
            save_liquid(key, liq, it[2], it[3], it[4])
            jdump(liquid_stats(key, r, liq, {"seconds": r["seconds"], "stacked_with": r["stacked_with"],
                                             "pieces": pieces, **{k: v for k, v in it[4].items()}}),
                  wpath("liquids", f"{key}_stats.json"))
        TIMES.setdefault(f"{kind}_stream", []).append({"key": key, "split": split, "seconds": r["seconds"],
                                                        "stacked_with": r["stacked_with"], "steps": r["steps"],
                                                        "streams": r["streams"]})
    for s in sinks.values():
        s.close()
    return res

def fit_streamed(items, split, max_pending=None):
    """Readout fits of streamed rows (final: from the saved states; CV: from CV_TMP, deleted after the fit)."""
    final = split == "final"
    tr, ev, _ = split_rows(split, N_BEATS["DS1"], N_BEATS["DS2"])
    for it in items:
        key = it[0]
        row, seed = key.split("_s")[0], int(key.split("_s")[1])
        if Q.is_done(row=row, seed=seed, split=split):
            if not final:
                remover(cv_path(key, split))()
            continue
        if final:
            C1, C2 = load_states(key, "DS1"), load_states(key, "DS2")
            submit_fit(row, seed, split, rates_fn(C1), rates_fn(C2))
        else:
            if not os.path.exists(cv_path(key, split)):
                continue
            C1 = np.load(cv_path(key, split), mmap_mode="r")
            submit_fit(row, seed, split, rates_fn(C1), None, after=remover(cv_path(key, split)))
        Q.wait(max_pending=2 * FIT_THREADS if max_pending is None else max_pending)

def thr_path(key, split):
    return wpath("liquids", f"{key}_thresholds{'' if split == 'final' else '__' + split}.npz")

def parents_thresholds(res, items, split):
    """After a P3 / M3 pass: each adaptive liquid's threshold sums and frozen thresholds (DS1 records; for a CV split
    its training records), saved for the frozen control."""
    for it in items:
        key, liq = it[0], it[1]
        if key not in res or liq.adapt_c is None or not key.startswith(("P3_", "M3_")):
            continue
        if split == "final":
            thr, recs = frozen_from(res[key], liq)
        else:
            trr = set(DATA["morph"]["DS1"]["pid"][FOLD_BY[parse_split(split)]["train_idx"]])
            thr, recs = frozen_from(res[key], liq, records=trr)
            assert set(recs) == trr
        save_thresholds(key, res[key], liq, thr, recs, tag="" if split == "final" else "__" + split)

def frozen_thr(parent_key, split):
    p = thr_path(parent_key, split)
    return np.load(p)["frozen"] if os.path.exists(p) else None

def run_P(split, stack):
    """P1-P5 for every seed on a split, then P3f from P3's thresholds; readouts submitted."""
    rows = [r for r in P_ROWS if r in RUN_ROWS]
    if not rows and "P3f" not in RUN_ROWS:
        return
    if "P3f" in RUN_ROWS and "P3" not in rows:
        rows = rows + ["P3"]                              # P3f needs P3's pass
    items = p_items(rows, split, SEEDS_RUN)
    stream_rows(items, split, stack)
    cvq = None if split == "final" else CV_PENDING_FITS
    fit_streamed([it for it in items if it[0].split("_s")[0] in RUN_ROWS], split, cvq)
    if "P3f" in RUN_ROWS:
        thr = {key_of("P3f", s): frozen_thr(key_of("P3", s), split) for s in SEEDS_RUN}
        if any(v is None for v in thr.values()):
            raise RuntimeError(f"P3f on {split}: P3's thresholds are missing")
        fitems = p_items(["P3f"], split, SEEDS_RUN, thr)
        stream_rows(fitems, split, stack)
        fit_streamed(fitems, split, cvq)

def run_M(stack, pieces, warm_s):
    """M0 per beat, then M1 and M3 for every seed (final split), then M3f from M3's thresholds."""
    global M_CAL
    if not any(r in RUN_ROWS for r in ("M0", "M1", "M3", "M3f")):
        return
    M_CAL = maass_calibration()
    if "M0" in RUN_ROWS:
        X1n, X2n, _, _ = fe("final")
        for s in SEEDS_RUN:
            t0 = time.time()
            per_beat_maass(s, X1n, X2n)
            TIMES.setdefault("M0_seconds", []).append(time.time() - t0)
            submit_fit("M0", s, "final", rates_fn(load_states(key_of("M0", s), "DS1")),
                       rates_fn(load_states(key_of("M0", s), "DS2")))
    rows = [r for r in M_ROWS if r in RUN_ROWS]
    if "M3f" in RUN_ROWS and "M3" not in rows:
        rows.append("M3")
    items = []
    for s in SEEDS_RUN:
        for row in rows:
            liq, g, sc, info = maass_row(row, s)
            items.append((key_of(row, s), liq, g, sc, info))
    stream_rows(items, "final", stack, pieces=pieces, warm_s=warm_s, kind="maass")
    fit_streamed([it for it in items if it[0].split("_s")[0] in RUN_ROWS], "final")
    if "M3f" in RUN_ROWS:
        fitems = []
        for s in SEEDS_RUN:
            thr = frozen_thr(key_of("M3", s), "final")
            if thr is None:
                raise RuntimeError("M3f: M3's thresholds are missing")
            liq, g, sc, info = maass_row("M3f", s, thr)
            fitems.append((key_of("M3f", s), liq, g, sc, info))
        stream_rows(fitems, "final", stack, pieces=pieces, warm_s=warm_s, kind="maass")
        fit_streamed(fitems, "final")


DROP_SPLIT = f"final_drop{int(DROP_SECONDS)}s"

def run_drop():
    """Keep vs drop: every row with saved states refit on the DS1 beats and tested on the DS2 beats whose R peak is
    at least DROP_SECONDS into its record (the liquid starts from rest at each record's start). CPU only."""
    keep = {ds: np.where(DATA["morph"][ds]["pos"] >= int(round(DROP_SECONDS * 360)))[0] for ds in ("DS1", "DS2")}
    rows = (keep["DS1"], keep["DS2"])
    for row in RUN_ROWS:
        if row == "B2":
            if not Q.is_done(row="B2", seed=None, split=DROP_SPLIT):
                X1f, X2f, _, _ = front_end("final", cast=True)
                c1, c2 = window_counts(X1f, NWIN), window_counts(X2f, NWIN)
                submit_fit("B2", None, DROP_SPLIT, rates_fn(c1), rates_fn(c2), rows=rows)
                del X1f, X2f
            continue
        for s in SEEDS_RUN:
            key = key_of(row, s)
            if states_done(key) and not Q.is_done(row=row, seed=s, split=DROP_SPLIT):
                submit_fit(row, s, DROP_SPLIT, rates_fn(load_states(key, "DS1")), rates_fn(load_states(key, "DS2")),
                           rows=rows)
                Q.wait(max_pending=2 * FIT_THREADS)


## 15. Benchmark: timings, equality checks, projected cost, switches

In [ ]:
# BENCHMARK (and SMOKE): every equality check, the timings of each liquid type in the four switch combinations
# (plain, chunked, stacked, both) on BENCHMARK_SECONDS of every record at the FULL stream counts, the second-stage
# count, the projected FULL time and cost, and the switches chosen. Writes benchmark/benchmark.json and .md.
REAL_T = 650000 + SPAN                                               # steps of a whole record (30 min) + the tail
REAL_T_CHUNK = 650000 // CHUNK_PIECES + int(CHUNK_WARMUP_S * 360) + SPAN

def per_step(r):
    return r["seconds"] / max(r["steps"], 1)

def _timing(res, keys):
    """Seconds per step of a run (all its liquids together, or summed when they ran one after another)."""
    rs = [res[k] for k in keys]
    if rs[0]["stacked_with"] > 1:
        return per_step(rs[0])
    return sum(per_step(r) for r in rs)

def stack_structure(items, S):
    """Every number of each liquid inside the stack equals its own (synapse weights times its g, input weights
    times its input size, neuron settings, adaptation, thresholds), and no synapse joins two liquids. No simulation.
    Returns the list of mismatches (empty = exact)."""
    offs = S.meta["offsets"]
    bad = []
    bp = np.searchsorted(offs, S.pre, side="right") - 1
    bq = np.searchsorted(offs, S.post, side="right") - 1
    if np.any(bp != bq):
        bad.append(f"{int((bp != bq).sum())} synapses join two liquids")
    for j, (key, liq, g, sc) in enumerate(items):
        o, sl, m = offs[j], slice(offs[j], offs[j + 1]), bp == j
        own = np.lexsort((liq.post, liq.pre, liq.typ))
        same = lambda x, y: np.array_equal(np.asarray(x), np.asarray(y))
        for name, x, y in (("pre", S.pre[m] - o, liq.pre[own]), ("post", S.post[m] - o, liq.post[own]),
                           ("type", S.typ[m], liq.typ[own]), ("weight", S.w[m], liq.w[own] * g),
                           ("U", S.U[m], liq.U[own]), ("D", S.D_ms[m], liq.D_ms[own]), ("F", S.F_ms[m], liq.F_ms[own]),
                           ("input weights", S.Win[:, sl], liq.Win * sc), ("excitatory", S.is_exc[sl], liq.is_exc),
                           ("leak", S.beta[sl], liq.beta), ("refractory gap", S.gap[sl], liq.gap), ("tap", S.tap[sl], liq.tap),
                           ("input neurons", S.in_neurons[sl], liq.in_neurons)):
            if not same(x, y):
                bad.append(f"{key}: {name}")
        if liq.in_a is not None and not same(S.in_a[sl], liq.in_a):
            bad.append(f"{key}: input current decay")
        if liq.kappa is not None and not same(S.kappa[sl], liq.kappa):
            bad.append(f"{key}: kappa")
        ac = np.zeros(liq.N) if liq.adapt_c is None else liq.adapt_c
        if (S.adapt_c is not None or liq.adapt_c is not None) and not same(S.adapt_c[sl], ac):
            bad.append(f"{key}: adaptation strength")
        if liq.adapt_c is not None and not same(S.adapt_tau_ms[sl], liq.adapt_tau_ms):
            bad.append(f"{key}: adaptation clocks")
        tb = np.ones(liq.N) if liq.thr_base is None else liq.thr_base
        if (S.thr_base is not None or liq.thr_base is not None) and not same(S.thr_base[sl], tb):
            bad.append(f"{key}: fixed thresholds")
    return bad

def stack_short_run(items, S, steps=2000, tol=1e-4):
    """The stacked run and each liquid alone, side by side from rest on the first steps of 4 records (2 DS1, 2 DS2),
    compared after every 20 steps: until the first spike that differs, every voltage must agree within tol.
    Float32 rounding moves a voltage (of order 1) by about 1e-7 per operation, so 1e-4 is far above rounding;
    a wrong or misplaced weight moves the voltages it reaches by that weight at once."""
    recs = RECS["DS1"][:2] + RECS["DS2"][:2]
    steps = min(steps, min(len(SIG[r]) for r in recs))
    enc = StreamEncoder([norm_signal(SIG[r], "final") for r in recs], list(range(len(recs))), [0] * len(recs))
    sims = [LS.Simulator(it[1], device=DEVICE, dtype=torch.float32, g=it[2], in_scale=it[3]) for it in items]
    sm_s = LS.Simulator(S, device=DEVICE, dtype=torch.float32, g=1.0, in_scale=1.0)
    offs = S.meta["offsets"]
    m = LSS.period(sm_s)
    seg = m * max(1, -(-20 // m))
    graphs = LS.USE_CUDA_GRAPHS
    LS.USE_CUDA_GRAPHS = False                    # many short calls (graphs give the same numbers: fast_path check)
    try:
        st_s, st_a = None, [None] * len(items)
        first, max_du, matched = None, 0.0, 0
        for a in range(0, steps - steps % seg, seg):
            x = enc(a, a + seg)
            rec = {"keep_state": True, "step_counts": True}
            o_s = sm_s.run(x, np.full((len(recs), S.N), S.rest) if st_s is None else None, seg, rec=rec, state=st_s, t0=a)
            st_s = o_s["state"]
            diff = False
            du = 0.0
            for j, sm in enumerate(sims):
                o = sm.run(x, np.full((len(recs), sm.N), sm.liq.rest) if st_a[j] is None else None, seg, rec=rec,
                           state=st_a[j], t0=a)
                st_a[j] = o["state"]
                blk = slice(offs[j], offs[j + 1])
                if not torch.equal(o["step_counts"], o_s["step_counts"][:, :, blk]):
                    diff = True
                du = max(du, float((o["u_end"] - o_s["u_end"][:, blk]).abs().max()))
            if diff:
                first = a
                break
            max_du = max(max_du, du)
            matched += int(o_s["step_counts"].sum())
    finally:
        LS.USE_CUDA_GRAPHS = graphs
    ok = max_du <= tol and (first is None or matched > 0)
    return ok, {"steps_compared": first if first is not None else steps - steps % seg,
                "first_spike_difference_at_step": first, "spikes_matched": matched, "max_voltage_difference": max_du,
                "tolerance": tol}

def stack_test(kind, items):
    """Stacking is the same equations as each liquid alone; only the order of additions inside the GPU's matrix
    products can differ. Allowed if the stacked liquid's numbers equal each liquid's own exactly, and a short run
    agrees to rounding until the first spike that differs. The long-run spike comparison is only reported."""
    S = LSS.stack([it[1] for it in items], g=[it[2] for it in items], in_scale=[it[3] for it in items])
    bad = stack_structure(items, S)
    ck(f"{kind}_stack_structure", not bad, liquids=len(items), mismatches=bad[:10])
    ok, info = stack_short_run(items, S)
    ck(f"{kind}_stack_short_run", ok, **info)
    return (not bad) and ok

def bench_paper(B, secs):
    s0 = SEEDS_RUN[0]
    items = p_items(list(P_ROWS), "final", SEEDS_RUN)
    keys = [it[0] for it in items]
    k3 = key_of("P3", s0); it3 = [it for it in items if it[0] == k3]
    run = lambda its, **kw: run_liquids([it[:4] for it in its], "final", ("DS1", "DS2"), seconds=secs, **kw)
    # plain: every liquid alone (kept windows of a few records for the comparisons)
    ks_alone = keep_sinks(keys)
    r_plain = run(items, stack=False, sinks=ks_alone, label="bench paper plain: ")
    ks_stack = keep_sinks(keys)
    r_stack = run(items, stack=True, sinks=ks_stack, label="bench paper stacked: ")
    ks_chunk = keep_sinks([k3])
    r_chunk = run(it3, stack=False, pieces=CHUNK_PIECES, warm_s=CHUNK_WARMUP_S, sinks=ks_chunk, label="bench paper chunked: ")
    r_both = run(items, stack=True, pieces=CHUNK_PIECES, warm_s=CHUNK_WARMUP_S, label="bench paper stacked+chunked: ")
    # stacked vs alone, every liquid; chunked vs unbroken, P3
    stack_ok = stack_test("paper", [it[:4] for it in items])
    eq = {}
    for k in keys:
        eq[k] = compare_kept(merge_kept(ks_alone, k), merge_kept(ks_stack, k))
    nd = sum(v["beats_differ"] for v in eq.values())
    B["equality"]["paper_stacked_vs_alone"] = {"beats_differ": nd, "per_liquid": eq}
    ck("paper_stacked_vs_alone_spikes", None, report=True, beats_differ=nd, liquids=len(keys),
       beats_compared=sum(v["beats"] for v in eq.values()),
       differ_first_half=sum(v["differ_first_half"] for v in eq.values()),
       differ_second_half=sum(v["differ_second_half"] for v in eq.values()))
    c = compare_kept(merge_kept(ks_alone, k3), merge_kept(ks_chunk, k3))
    B["equality"]["paper_chunked_vs_unbroken"] = c
    ck("paper_chunked_vs_unbroken", c["beats_differ"] == 0, **{k: v for k, v in c.items() if k != "share_counts_differ"},
       note="the paper's rows run unchunked anyway")
    # adaptation on: P3's adaptive neurons above threshold 1. P1 with every threshold fixed at exactly 1 and the
    # threshold reset runs the new per-neuron threshold code (a strength of 0 would switch it off) and must give P1
    st3 = liquid_stats(k3, r_plain[k3], it3[0][1])
    k1 = key_of("P1", s0)
    l0, g0, sc0, _ = paper_row("P1", "final", s0)
    l0 = LSS.frozen_copy(l0, np.ones(l0.N)); l0.reset_to_thr = True
    ks0 = keep_sinks(["P1thr1"])
    run([("P1thr1", l0, g0, sc0)], stack=False, sinks=ks0, label="bench P1 with fixed thresholds of 1: ")
    c0 = compare_kept(merge_kept(ks_alone, k1), merge_kept(ks0, "P1thr1"))
    ck("thresholds_of_1_are_P1", c0["beats_differ"] == 0, beats_differ=c0["beats_differ"], beats_compared=c0["beats"])
    ck("adaptation_on_P3", st3.get("mean_threshold_adaptive", 1.0) > 1.0,
       mean_threshold_adaptive=st3.get("mean_threshold_adaptive"), n_adaptive=st3.get("n_adaptive"),
       n_adaptive_inhibitory=st3.get("n_adaptive_inhibitory"))
    # second stage: P3f x seeds from the plain P3 runs' thresholds; frozen-control check on the first seed
    thr = {key_of("P3f", s): frozen_from(r_plain[key_of("P3", s)], [it for it in items if it[0] == key_of("P3", s)][0][1])[0]
           for s in SEEDS_RUN}
    fitems = p_items(["P3f"], "final", SEEDS_RUN, thr)
    ksf = keep_sinks([fitems[0][0]])
    r2p = run(fitems, stack=False, sinks=ksf, label="bench P3f plain: ")
    r2s = run(fitems, stack=True, label="bench P3f stacked: ")
    frozen_check("P3f", it3[0][1], r_plain[k3], merge_kept(ks_alone, k3), r2p[fitems[0][0]], merge_kept(ksf, fitems[0][0]))
    # CV split shape: DS1 records only (one split's pass)
    rcv_s = run_liquids([it[:4] for it in items], "final", ("DS1",), seconds=secs, stack=True, label="bench paper CV-shape stacked: ")
    rcv_p = run_liquids([it[:4] for it in it3], "final", ("DS1",), seconds=secs, stack=False, label="bench paper CV-shape plain: ")
    T = {"plain": _timing(r_plain, keys), "stacked": _timing(r_stack, keys), "chunked_one": per_step(r_chunk[k3]),
         "both": _timing(r_both, keys), "second_plain": _timing(r2p, [f[0] for f in fitems]),
         "second_stacked": _timing(r2s, [f[0] for f in fitems]), "cv_stacked": _timing(rcv_s, keys),
         "cv_plain_one": per_step(rcv_p[k3])}
    B["timings"]["paper_seconds_per_step"] = T
    B["timings"]["paper_liquids"] = len(keys)
    return T, stack_ok

def frozen_check(name, parent_liq, r_par, kept_par, r_fro, kept_fro):
    """Frozen control vs its parent: per adaptive neuron, |rate ratio - 1| (median <= 2%, 90th percentile <= 10%);
    mean rate around the QRS (windows 12-16, R - 48 .. R + 52 samples) and between beats (windows 0-9) within 5%."""
    a = np.asarray(parent_liq.adapt_c) > 0
    d1 = [r for r, v in r_par["records"].items() if v["ds"] == "DS1"]
    sp = np.sum([r_par["records"][r]["spike_sum"] for r in d1], 0)
    sf = np.sum([r_fro["records"][r]["spike_sum"] for r in d1], 0)
    ok_n = a & (sp > 0)
    ratio = sf[ok_n] / sp[ok_n]
    med = float(np.median(np.abs(ratio - 1))) if ok_n.any() else float("nan")
    p90 = float(np.percentile(np.abs(ratio - 1), 90)) if ok_n.any() else float("nan")
    keys = sorted(set(kept_par) & set(kept_fro))
    def rate(kept, w):
        return float(np.mean([kept[k][w].mean() for k in keys])) if keys else float("nan")
    qrs, btw = list(range(12, 17)), list(range(0, 10))
    rq = rate(kept_fro, qrs) / max(rate(kept_par, qrs), 1e-12) - 1
    rb = rate(kept_fro, btw) / max(rate(kept_par, btw), 1e-12) - 1
    ok = med <= 0.02 and p90 <= 0.10 and abs(rq) <= 0.05 and abs(rb) <= 0.05
    extra = {} if ok or name != "M3f" else {"note": "outside the bounds: M3f is reported as approximate (the plan), no tuning"}
    return ck(f"frozen_{name}", ok, **extra, median_abs_rate_ratio=round(med, 4), p90_abs_rate_ratio=round(p90, 4),
              qrs_rate_diff=round(rq, 4), between_rate_diff=round(rb, 4), adaptive_neurons=int(ok_n.sum()),
              averaged_over="DS1 records only")

def bench_maass(B, secs):
    global M_CAL
    t0 = time.time()
    M_CAL = maass_calibration()
    B["timings"]["maass_calibration_seconds"] = M_CAL.get("seconds", time.time() - t0)
    s0 = SEEDS_RUN[0]
    items = []
    for row in ("M1", "M3"):
        liq, g, sc, info = maass_row(row, s0)
        items.append((key_of(row, s0), liq, g, sc, info))
    keys = [it[0] for it in items]
    run = lambda its, **kw: run_liquids([it[:4] for it in its], "final", ("DS1", "DS2"), seconds=secs, **kw)
    ka = keep_sinks(keys); r_plain = run(items, stack=False, sinks=ka, label="bench Maass plain: ")
    ks = keep_sinks(keys); r_stack = run(items, stack=True, sinks=ks, label="bench Maass stacked: ")
    kc = keep_sinks(keys); r_chunk = run(items, stack=False, pieces=CHUNK_PIECES, warm_s=CHUNK_WARMUP_S, sinks=kc,
                                         label="bench Maass chunked: ")
    kb = keep_sinks(keys); r_both = run(items, stack=True, pieces=CHUNK_PIECES, warm_s=CHUNK_WARMUP_S, sinks=kb,
                                        label="bench Maass stacked+chunked: ")
    stack_ok = stack_test("maass", [it[:4] for it in items])
    # chunked is proven right only by giving the unbroken run's spikes: chunked vs plain (each liquid alone), and
    # stacked+chunked vs stacked (the same additions, so only the chunking differs)
    eq = {"stacked_vs_plain": {k: compare_kept(merge_kept(ka, k), merge_kept(ks, k)) for k in keys},
          "chunked_vs_plain": {k: compare_kept(merge_kept(ka, k), merge_kept(kc, k)) for k in keys},
          "both_vs_stacked": {k: compare_kept(merge_kept(ks, k), merge_kept(kb, k)) for k in keys}}
    nd = {name: sum(v["beats_differ"] for v in e.values()) for name, e in eq.items()}
    nb = {name: sum(v["beats"] for v in e.values()) for name, e in eq.items()}
    ck("maass_stacked_vs_plain_spikes", None, report=True, beats_differ=nd["stacked_vs_plain"], beats_compared=nb["stacked_vs_plain"],
       differ_first_half=sum(v["differ_first_half"] for v in eq["stacked_vs_plain"].values()),
       differ_second_half=sum(v["differ_second_half"] for v in eq["stacked_vs_plain"].values()))
    ck("maass_chunked_vs_unbroken", nd["chunked_vs_plain"] == 0, beats_differ=nd["chunked_vs_plain"],
       beats_compared=nb["chunked_vs_plain"], note="chunks are used only if identical")
    ck("maass_stacked_chunked_vs_stacked", nd["both_vs_stacked"] == 0, beats_differ=nd["both_vs_stacked"],
       beats_compared=nb["both_vs_stacked"])
    B["equality"]["maass"] = eq
    segment_gate(B)
    # second stage: M3f from M3's plain run; frozen-control check (reported as approximate if it fails)
    k3 = key_of("M3", s0)
    thr, _ = frozen_from(r_plain[k3], items[1][1])
    liq, g, sc, info = maass_row("M3f", s0, thr)
    kf = keep_sinks([key_of("M3f", s0)])
    r2 = run([(key_of("M3f", s0), liq, g, sc, info)], stack=False, sinks=kf, label="bench M3f: ")
    frozen_check("M3f", items[1][1], r_plain[k3], merge_kept(ka, k3), r2[key_of("M3f", s0)], merge_kept(kf, key_of("M3f", s0)))
    # M0: per-beat Maass on 2000 DS1 beats
    X1n = fe("final")[0]
    sm = LS.Simulator(items[0][1], device=DEVICE, dtype=torch.float32, g=items[0][2], in_scale=items[0][3])
    n = min(2000, len(X1n))
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for s in range(0, n, NEW_SIM_BATCH):
        x = encode_t(X1n[s:min(s + NEW_SIM_BATCH, n)], "multi")
        u0 = LS.start_voltages(items[0][1], s0, np.arange(s, s + len(x)), Y_START_STREAM)
        sm.run(x, u0, LIQ["num_sim_steps"], rec={"windows": (LC.WIN, LC.NWIN)})
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    m0 = (time.time() - t0) / n
    T = {"plain": _timing(r_plain, keys), "stacked": _timing(r_stack, keys), "chunked": _timing(r_chunk, keys),
         "both": _timing(r_both, keys), "second": per_step(r2[key_of("M3f", s0)]), "M0_seconds_per_beat": m0}
    B["timings"]["maass_seconds_per_step"] = T
    allowed = {"plain": True, "stacked": stack_ok, "chunked": nd["chunked_vs_plain"] == 0,
               "both": stack_ok and nd["chunked_vs_plain"] == 0 and nd["both_vs_stacked"] == 0}
    return T, allowed

def save_json(d, p):
    jdump(d, p + ".tmp"); os.replace(p + ".tmp", p)

def bench_part(name, fn, B, secs):
    """One part of the benchmark (the paper's liquid, Maass's), saved when it finishes: a restart reuses it."""
    p = wpath("benchmark", f"part_{name}.json")
    if os.path.exists(p):
        d = json.load(open(p))
        for k in ("timings", "equality"):
            B[k].update(d[k])
        B.update(d["extra"]); CHECK.update(d["checks"])
        log(f"benchmark: {name} part reused from {p}")
        return d["T"], d["flag"]
    before_checks, before = dict(CHECK), {k: dict(B[k]) for k in ("timings", "equality")}
    keys = set(B)
    T, flag = fn(B, secs)
    save_json({"T": T, "flag": flag,
               **{k: {a: v for a, v in B[k].items() if a not in before[k]} for k in ("timings", "equality")},
               "extra": {k: B[k] for k in B if k not in keys},
               "checks": {k: v for k, v in CHECK.items() if before_checks.get(k) != v}}, p)
    return T, flag

def benchmark():
    p = wpath("benchmark", "benchmark.json")
    if os.path.exists(p):
        return json.load(open(p))
    secs = None if SMOKE_TEST else BENCHMARK_SECONDS
    B = {"mode": MODE, "seconds_per_record": secs or SMOKE_SECONDS, "timings": {}, "equality": {},
         "device": ENV.get("gpu", str(DEVICE))}
    t_start = time.time()
    Tp, paper_stack_ok = bench_part("paper", bench_paper, B, secs) if any(r in RUN_ROWS for r in P_ROWS + ("P3f",)) else (None, False)
    Tm, m_allowed = bench_part("maass", bench_maass, B, secs) if any(r in RUN_ROWS for r in ("M0", "M1", "M3", "M3f")) else (None, None)
    nS = len(SEEDS_RUN)
    proj = {}
    choice = {"paper": {"stack": False, "chunks": False}, "maass": {"stack": False, "chunks": False}}
    if Tp:
        stack_ok = paper_stack_ok and Tp["stacked"] < Tp["plain"]
        choice["paper"] = {"stack": bool(stack_ok), "chunks": False,
                           "why": "stacking test passed and faster" if stack_ok else
                                  ("stacking test failed" if not paper_stack_ok else "stacking not faster")}
        k = "stacked" if stack_ok else "plain"
        proj["P1-P5 final"] = Tp[k] * REAL_T
        proj["P3f final"] = Tp["second_stacked" if stack_ok else "second_plain"] * REAL_T
        if EXACT_CV:
            cv = Tp["cv_stacked"] if stack_ok else Tp["cv_plain_one"] * len(P_ROWS) * nS
            proj["P1-P5 CV (12 splits)"] = 12 * cv * REAL_T
            # the measured 3-liquid (P3f x 3 seeds) time per step at the final pass's 44 streams; a CV pass has 22,
            # so this errs high rather than low
            proj["P3f CV (12 splits)"] = 12 * Tp["second_stacked" if stack_ok else "second_plain"] * REAL_T
    if Tm:
        opts = {"plain": (Tm["plain"] * REAL_T, True), "stacked": (Tm["stacked"] * REAL_T, m_allowed["stacked"]),
                "chunked": (Tm["chunked"] * REAL_T_CHUNK, m_allowed["chunked"]),
                "both": (Tm["both"] * REAL_T_CHUNK, m_allowed["both"])}
        best = min((v[0], k) for k, v in opts.items() if v[1])[1]
        choice["maass"] = {"stack": best in ("stacked", "both"), "chunks": best in ("chunked", "both"), "option": best,
                           "projected_seconds": {k: v[0] * nS for k, v in opts.items()},
                           "allowed": {k: bool(v[1]) for k, v in opts.items()}}
        proj["M1, M3 final"] = opts[best][0] * nS
        # three M3f liquids, one M3f's plain cost scaled by the chosen option's speed-up over plain
        proj["M3f final"] = Tm["second"] * REAL_T * nS * (opts[best][0] / opts["plain"][0])
        proj["M0 per beat"] = Tm["M0_seconds_per_beat"] * (N_BEATS["DS1"] + N_BEATS["DS2"]) * nS
    b1 = [t["seconds"] for t in TIMES.get("B1_sim", []) if t["split"] == "final"]
    proj["machinery check at FULL's start (one B1 seed, DS1 + DS2)"] = float(np.mean(b1)) if b1 else 0.0
    done = {"Maass calibration (saved, reused by FULL)": B["timings"].get("maass_calibration_seconds", 0.0),
            "B1 and B2 on DS2 and the CV splits (kept)": float(sum(t["seconds"] for t in TIMES.get("B1_sim", [])))}
    # readout fits left for FULL: P rows (DS2 + CV), Maass rows (DS2), the keep-vs-drop refits (DS2, every row)
    n_p = sum(r in RUN_ROWS for r in P_ROWS + ("P3f",)) * nS * (1 + (len(SPLITS_CV) if EXACT_CV else 0))
    n_m = sum(r in RUN_ROWS for r in ("M0", "M1", "M3", "M3f")) * nS
    n_drop = len([r for r in RUN_ROWS if r != "B2"]) * nS + ("B2" in RUN_ROWS)
    n_fits = n_p + n_m + n_drop
    fits = [r.get("fit_seconds", 0) for r in Q.rows if r["row"] == "B1"]
    cpu_h = (np.mean(fits) if fits else 90.0) * n_fits / max(1, FIT_THREADS) / 3600
    gpu_h = sum(proj.values()) / 3600
    B.update(projection_seconds=proj, projected_gpu_hours=gpu_h, projected_cost_usd=gpu_h * GPU_PRICE_PER_H,
             projected_cpu_fit_hours=cpu_h, cpu_fits_left=n_fits, done_in_benchmark_seconds=done, choice=choice,
             benchmark_seconds=time.time() - t_start, checks={k: v for k, v in CHECK.items()})
    save_json(B, p)
    lines = ["# Benchmark", "", f"Device: {B['device']}. {B['seconds_per_record']} s of every record "
             f"({MODE}). Benchmark time {B['benchmark_seconds'] / 60:.1f} min.", "",
             f"## Projected FULL ({nS} seed{'s' if nS > 1 else ''}): what FULL still has to run", "", "| block | GPU time |", "|---|---|"]
    lines += [f"| {k} | {v / 60:.1f} min |" for k, v in proj.items()]
    lines += [f"| **total** | **{gpu_h:.2f} h (${gpu_h * GPU_PRICE_PER_H:.2f} at ${GPU_PRICE_PER_H}/h)** |", "",
              f"Readout fits left for FULL on the CPU: {n_fits}, about {cpu_h:.1f} h with {FIT_THREADS} threads "
              "(they overlap the GPU work).", "",
              "Already done here, not repeated by FULL: " + "; ".join(f"{k}: {v / 60:.1f} min" for k, v in done.items()) + ".", "",
              "## Switches chosen", "", f"* paper's liquid: {choice['paper']}", f"* Maass's liquid: {choice['maass']}", "",
              "## Checks", "", "| check | result | details |", "|---|---|---|"]
    for k, v in CHECK.items():
        res = "reported" if v.get("report") else {True: "pass", False: "FAIL", None: "not run"}[v["ok"]]
        det = ", ".join(f"{a} {b}" for a, b in v.items() if a not in ("ok", "report") and not isinstance(b, (dict, list)))
        lines.append(f"| {k} | {res} | {det} |")
    open(wpath("benchmark", "benchmark.md"), "w").write("\n".join(lines) + "\n")
    log(f"benchmark: projected FULL {gpu_h:.2f} GPU-h (${gpu_h * GPU_PRICE_PER_H:.2f}), CPU fits {cpu_h:.1f} h; "
        f"choice {choice}")
    return B


## 16. Summary

In [ ]:
# Summary: the headline table (each row against B1 per seed, the pre-declared rule), the exact-CV tests on SVEB F1,
# the memory decomposition (P3 - P1 = P3f - P1 + P3 - P3f), extras. summary.md and rows.csv, rebuilt any time.
LABELS = {"B1": "B1 config 11 (per beat)", "B2": "B2 no liquid", "P1": "P1 paper's, streaming",
          "P2": "P2 paper's, no internal connections", "P3": "P3 paper's + adaptation",
          "P3f": "P3f P3 with frozen thresholds", "P4": "P4 P3 without internal connections",
          "P5": "P5 paper's + one 2 s clock", "M0": "M0 Maass per beat", "M1": "M1 Maass streaming",
          "M3": "M3 Maass + adaptation", "M3f": "M3f M3 with frozen thresholds"}
MARGIN = 0.01                      # B1's own seed spread (plan)

def rows_df():
    R = pd.DataFrame(Q.rows)
    if len(R) == 0:
        return R
    R["seed"] = pd.to_numeric(R["seed"], errors="coerce")
    return R

def tab(R, row):
    d = R[R.row == row] if len(R) else None
    return None if d is None or len(d) == 0 else d.rename(columns={"row": "model"})

def ds2_by_seed(R, row, metric="f1_SVEB", split="final"):
    d = R[(R.row == row) & (R.split == split)]
    return {(int(s) if pd.notna(s) else None): float(v) for s, v in zip(d.seed, d[metric])} if len(d) else {}

def headline(R):
    """The pre-declared rule for P1 and P3 (Holm family), and the same numbers for the other rows."""
    b1 = ds2_by_seed(R, "B1")
    fam = [(r, LSt.compare(as_runs(tab(R, r)), as_runs(tab(R, "B1")), require_seed_sign=True, metric="f1_SVEB"))
           for r in ("P1", "P3") if tab(R, r) is not None and tab(R, "B1") is not None]
    fam_rows = {d["comparison"]: d for d in LSt.family("headline", fam, need_seed_sign=False)}
    out = []
    for row in [r for r in ROWS if r not in ("B1",)]:
        mine = ds2_by_seed(R, row)
        if not mine:
            continue
        seeds = sorted(set(mine) & set(b1)) if None not in mine else sorted(b1)
        diffs = {s: (mine[s] if None not in mine else mine[None]) - b1[s] for s in seeds}
        all_seeds = bool(diffs) and all(v >= MARGIN for v in diffs.values()) and len(diffs) == len(SEEDS_RUN)
        r = {"row": row, "label": LABELS[row], "ds2_f1_SVEB_mean": float(np.mean(list(mine.values()))),
             "B1_f1_SVEB_mean": float(np.mean(list(b1.values()))) if b1 else np.nan,
             "diff_by_seed": ", ".join(f"{s}: {v:+.4f}" for s, v in diffs.items()), "beats_B1_by_margin_all_seeds": all_seeds}
        if row in fam_rows:
            f = fam_rows[row]
            r.update(n_splits=f.get("n_splits"), cv_diff=f.get("cv_diff"), ci_low=f.get("ci_low"), ci_high=f.get("ci_high"),
                     p=f.get("p"), p_holm=f.get("p_holm"), cv_verdict=f.get("verdict"))
            r["rule"] = "BETTER (rule met)" if all_seeds and f.get("verdict") == "meaningful" else "rule not met"
        elif row in ("P2", "P4", "P5", "P3f", "B2") and tab(R, row) is not None:
            c = LSt.compare(as_runs(tab(R, row)), as_runs(tab(R, "B1")), metric="f1_SVEB")
            if c:
                c["p_holm"] = c["p"]                       # LSt.verdict reads p_holm; outside the family it is p
                r.update(n_splits=c["n_splits"], cv_diff=c["cv_diff"], ci_low=c["ci_low"], ci_high=c["ci_high"],
                         p=c["p"], p_holm=np.nan, cv_verdict=LSt.verdict(c))
            r["rule"] = "outside the family (reported)"
        else:
            r["rule"] = "DS2 only, descriptive" + (" (clears the margin on every seed)" if all_seeds else "")
        out.append(r)
    return pd.DataFrame(out)

def macro_tests(R):
    """t1-t4's test, on macro F1: every row with CV against B1 (Holm over {P1, P3}, as the headline)."""
    fam = [(r, LSt.compare(as_runs(tab(R, r)), as_runs(tab(R, "B1")), metric="macro_f1"))
           for r in ("P1", "P3") if tab(R, r) is not None and tab(R, "B1") is not None]
    fam_rows = {d["comparison"]: d for d in LSt.family("macro", fam, need_seed_sign=False)}
    out = []
    for row in [r for r in ROWS if r != "B1"]:
        if tab(R, row) is None or tab(R, "B1") is None:
            continue
        c = LSt.compare(as_runs(tab(R, row)), as_runs(tab(R, "B1")), metric="macro_f1")
        if not c:
            continue
        r = {"label": LABELS[row], "n_splits": c["n_splits"], "cv_diff": c["cv_diff"], "ci_low": c["ci_low"],
             "ci_high": c["ci_high"], "p": c["p"], "ds2_diff_mean": c.get("ds2_diff")}
        if row in fam_rows:
            r.update(p_holm=fam_rows[row].get("p_holm"), cv_verdict=fam_rows[row].get("verdict"))
        else:
            c["p_holm"] = c["p"]
            r.update(p_holm=np.nan, cv_verdict=LSt.verdict(c))
        out.append(r)
    return pd.DataFrame(out)

def figure_sveb(R):
    """figures/ds2_sveb_f1.png: DS2 SVEB F1 of every row and seed, B1's seeds as lines."""
    d = R[R.split == "final"]
    rows = [r for r in ROWS if r in set(d.row)]
    if not rows:
        return
    fig, ax = plt.subplots(figsize=(max(6, 0.7 * len(rows)), 3.6))
    for i, row in enumerate(rows):
        v = d[d.row == row]["f1_SVEB"].values
        ax.scatter(np.full(len(v), i), v, s=22, color="#4269d0" if row.startswith("P") else
                   ("#efb118" if row.startswith("M") else "#6b6b6b"), zorder=3)
    for v in d[d.row == "B1"]["f1_SVEB"].values:
        ax.axhline(v, color="#6b6b6b", lw=0.8, ls=":", zorder=1)
    ax.set_xticks(range(len(rows))); ax.set_xticklabels(rows)
    ax.set_ylabel("DS2 SVEB F1"); ax.set_title("DS2 SVEB F1 by row and seed (dotted: B1's seeds)", fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    os.makedirs(wpath("figures"), exist_ok=True)
    fig.savefig(wpath("figures", "ds2_sveb_f1.png"), dpi=150); plt.close(fig)

def decomposition(R):
    """Memory vs operating point on SVEB F1 (CV test where both rows have CV, DS2 by seed otherwise)."""
    items = [("P3 - P1 (adaptation)", "P3", "P1"), ("P3f - P1 (operating point)", "P3f", "P1"),
             ("P3 - P3f (moving thresholds: memory)", "P3", "P3f"), ("P1 - B1 (streaming alone)", "P1", "B1"),
             ("P1 - P2 (internal connections, streaming)", "P1", "P2"), ("P3 - P4 (internal connections, with adaptation)", "P3", "P4"),
             ("P3 - P5 (spread of clocks vs one 2 s clock)", "P3", "P5"),
             ("M1 - M0 (Maass: streaming)", "M1", "M0"), ("M3 - M1 (Maass: adaptation)", "M3", "M1"),
             ("M3f - M1 (Maass: operating point)", "M3f", "M1"), ("M3 - M3f (Maass: moving thresholds)", "M3", "M3f")]
    out = []
    for lab, a, b in items:
        A, Bt = tab(R, a), tab(R, b)
        if A is None or Bt is None:
            continue
        c = LSt.compare(as_runs(A), as_runs(Bt), metric="f1_SVEB")
        da, db = ds2_by_seed(R, a), ds2_by_seed(R, b)
        if None in db:
            db = {s: db[None] for s in da}
        ds2 = {s: da[s] - db[s] for s in sorted(set(da) & set(db), key=str)}
        r = {"comparison": lab, "ds2_diff_mean": float(np.mean(list(ds2.values()))) if ds2 else np.nan,
             "ds2_diff_by_seed": ", ".join(f"{s}: {v:+.4f}" for s, v in ds2.items())}
        if c:
            r.update(cv_diff=c["cv_diff"], ci_low=c["ci_low"], ci_high=c["ci_high"], p=c["p"])
        out.append(r)
    return pd.DataFrame(out)

def keep_vs_drop(R):
    """DS2 SVEB F1 with every beat (keep) and refit without each record's first DROP_SECONDS (drop), and the DS2
    part of the rule on the drop set (each row against B1's drop fit, by seed)."""
    b1 = ds2_by_seed(R, "B1", split=DROP_SPLIT)
    out = []
    for row in ROWS:
        keep, drop = ds2_by_seed(R, row), ds2_by_seed(R, row, split=DROP_SPLIT)
        if not keep and not drop:
            continue
        r = {"row": LABELS[row], "keep_f1_SVEB": float(np.mean(list(keep.values()))) if keep else np.nan,
             "drop_f1_SVEB": float(np.mean(list(drop.values()))) if drop else np.nan}
        if drop and b1 and row != "B1":
            seeds = sorted(b1) if None in drop else sorted(set(drop) & set(b1))
            d = {s: (drop[None] if None in drop else drop[s]) - b1[s] for s in seeds}
            r["drop_diff_vs_B1_by_seed"] = ", ".join(f"{s}: {v:+.4f}" for s, v in d.items())
            r["drop_clears_margin_all_seeds"] = bool(d) and len(d) == len(SEEDS_RUN) and all(v >= MARGIN for v in d.values())
        out.append(r)
    return pd.DataFrame(out)

def means_ds2(R):
    cols = ["macro_f1", "f1_SVEB", "recall_SVEB", "precision_SVEB", "auc_SVEB", "f1_VEB", "recall_VEB", "precision_VEB",
            "recall_N", "precision_N", "f1_SVEB_no232", "recall_SVEB_no232"]
    out = []
    for row in ROWS:
        d = R[(R.row == row) & (R.split == "final")] if len(R) else []
        if len(d) == 0:
            continue
        r = {"row": LABELS[row], "seeds": int(len(d))}
        r.update({c: float(d[c].mean()) for c in cols if c in d})
        cv = R[(R.row == row) & (R.split.astype(str).str.startswith("cv-"))]
        r["cv_f1_SVEB"] = float(cv["f1_SVEB"].mean()) if len(cv) else np.nan
        r["cv_macro_f1"] = float(cv["macro_f1"].mean()) if len(cv) else np.nan
        out.append(r)
    return pd.DataFrame(out)

def per_record_sveb():
    """DS2 SVEB recall per record (records with SVEB beats), mean over seeds, from the saved predictions."""
    D = DATA["morph"]["DS2"]
    rec = {}
    for row in ROWS:
        vals = []
        for s in (SEEDS_RUN if row != "B2" else (None,)):
            p = wpath("runs", f"{key_of(row, s)}__svm", "predictions.npz")
            if not os.path.exists(p):
                continue
            z = np.load(p)
            pid, yt, yp = D["pid"][z["eval_index"]], z["y_true"], z["y_pred"]
            vals.append({r: float(((yp == 1) & (yt == 1) & (pid == r)).sum() / max(((yt == 1) & (pid == r)).sum(), 1))
                         for r in np.unique(pid[yt == 1])})
        if vals:
            rec[row] = {r: float(np.mean([v[r] for v in vals])) for r in vals[0]}
    if not rec:
        return pd.DataFrame()
    recs = sorted(set().union(*[set(v) for v in rec.values()]))
    n = {r: int(((D["y"] == 1) & (D["pid"] == r)).sum()) for r in recs}
    return pd.DataFrame([{"record": r, "SVEB beats": n[r], **{row: rec[row].get(r, np.nan) for row in rec}} for r in recs])

def write_summary():
    R = rows_df()
    if len(R) == 0:
        log("summary: no rows yet"); return
    R.drop(columns=["confusion_matrix"], errors="ignore").to_csv(wpath("rows.csv"), index=False)
    H, M, Dc, PR, KD = headline(R), means_ds2(R), decomposition(R), per_record_sveb(), keep_vs_drop(R)
    HM = macro_tests(R)
    figure_sveb(R)
    chk = json.load(open(wpath("check.json"))) if os.path.exists(wpath("check.json")) else {}
    L = ["# Config 11 vs continuous liquids with memory", "",
         f"Mode {MODE}. Seeds {', '.join(map(str, SEEDS_RUN))}. DS2 = train on all DS1, test on DS2; exact CV = the "
         f"authors' {len(SPLITS_CV)} DS1 splits (each with its own normalisation and A_in).", "",
         "**Rule (pre-declared):** a paper's-liquid streaming row is better than B1 if its DS2 SVEB F1 beats B1's by at "
         f"least {MARGIN} on all three seeds and the exact-CV test on SVEB F1 (Nadeau-Bengio, Holm over {{P1, P3}}) "
         "calls it meaningful. Maass rows are DS2 only and descriptive.", "",
         "## Headline: SVEB F1 against B1", "",
         md_table(H, ["label", "ds2_f1_SVEB_mean", "B1_f1_SVEB_mean", "diff_by_seed", "beats_B1_by_margin_all_seeds",
                      "n_splits", "cv_diff", "ci_low", "ci_high", "p", "p_holm", "cv_verdict", "rule"],
                  fmt={"n_splits": "{:.0f}"}), "",
         "p: unadjusted; p_holm: Holm-adjusted within the family {P1, P3} only (empty for the other rows, whose verdict "
         "uses p). n_splits: CV splits in the test (12 when complete).", "",
         "## The same test on macro F1 (as in t1-t4; reported)", "",
         md_table(HM, ["label", "n_splits", "cv_diff", "ci_low", "ci_high", "p", "p_holm", "cv_verdict", "ds2_diff_mean"],
                  fmt={"n_splits": "{:.0f}"})
         if len(HM) else "(none yet)", "",
         "## DS2 means over seeds (and CV means)", "",
         md_table(M, ["row", "seeds", "macro_f1", "f1_SVEB", "recall_SVEB", "precision_SVEB", "auc_SVEB", "f1_VEB",
                      "recall_VEB", "precision_VEB", "recall_N", "precision_N", "f1_SVEB_no232", "recall_SVEB_no232",
                      "cv_f1_SVEB", "cv_macro_f1"]), "", "Figure: figures/ds2_sveb_f1.png", "",
         f"## Keep vs drop: without each record's first {DROP_SECONDS:g} s (refit, DS2 only)", "",
         "The streaming liquids start from rest at each record's start. Drop: every row refit on the DS1 beats and "
         f"tested on the DS2 beats at least {DROP_SECONDS:g} s into their record; the DS2 part of the rule is applied "
         "against B1's drop fit (no CV on this set).", "",
         md_table(KD, ["row", "keep_f1_SVEB", "drop_f1_SVEB", "drop_diff_vs_B1_by_seed", "drop_clears_margin_all_seeds"])
         if len(KD) else "(none yet)", "",
         "## Where a difference comes from (SVEB F1)", "",
         md_table(Dc, ["comparison", "ds2_diff_mean", "ds2_diff_by_seed", "cv_diff", "ci_low", "ci_high", "p"]), "",
         "## DS2 SVEB recall per record (mean over seeds)", "", md_table(PR, list(PR.columns)) if len(PR) else "(none yet)", "",
         "## Checks", ""]
    for k, v in chk.items():
        res = "reported" if v.get("report") else {True: "pass", False: "FAIL", None: "not run"}[v.get("ok")]
        L.append(f"* {k}: {res}" + "".join(f"; {a} {b}" for a, b in v.items()
                                           if a not in ("ok", "report") and not isinstance(b, (dict, list))))
    open(wpath("summary.md"), "w").write("\n".join(L) + "\n")
    H.to_csv(wpath("headline.csv"), index=False); Dc.to_csv(wpath("decomposition.csv"), index=False)
    KD.to_csv(wpath("keep_vs_drop.csv"), index=False); HM.to_csv(wpath("macro_f1_tests.csv"), index=False)
    log(f"summary: {wpath('summary.md')}")


## 17. Run

In [ ]:
# The run. SMOKE: checks, benchmark and every row on tiny records. BENCHMARK: checks (B1 and B2 on DS2 and every
# CV split: real results, kept) and the benchmark. FULL: the machinery check again, then every row and the summary.
log(f"stream notebook: MODE {MODE}, rows {list(RUN_ROWS)}, seeds {list(SEEDS_RUN)}, exact CV {EXACT_CV}, "
    f"work dir {WORK_DIR}")
jdump({**ENV, "authors_commit": AUTHORS_COMMIT, "notebook_build": NOTEBOOK_BUILD, "mode": MODE, "rows": list(RUN_ROWS),
       "seeds": list(SEEDS_RUN), "exact_cv": EXACT_CV, "stack": STACK, "chunks": CHUNKS, "chunk_pieces": CHUNK_PIECES,
       "chunk_warmup_s": CHUNK_WARMUP_S, "session": SESSION_TAG}, wpath("env.json"))
save_beats_info()
if os.path.exists(wpath("check.json")):
    CHECK.update(json.load(open(wpath("check.json"))))

def save_checks():
    jdump(CHECK, wpath("check.json"))

def all_B():
    for sp in SPLITS_RUN:
        run_B(sp)
    Q.wait(0)

if MODE in ("SMOKE", "BENCHMARK"):
    log("== checks")
    fast_path_check()
    if "machinery" not in CHECK or CHECK["machinery"]["ok"] is None:
        check_machinery()
    run_checks_core(); save_checks()
    log("== B1 and B2 on DS2 and the CV splits (task 1's checks)")
    all_B()
    check_t1_rows(); save_checks()
    log("== benchmark")
    BENCH = benchmark(); save_checks()

if MODE in ("SMOKE", "FULL"):
    if MODE == "FULL":
        log("== checks at the start of FULL")
        fast_path_check()
        check_machinery(); save_checks()
    bp = wpath("benchmark", "benchmark.json")
    choice = json.load(open(bp))["choice"] if os.path.exists(bp) else None
    if choice is None and "auto" in (STACK, CHUNKS):
        log("WARNING: no benchmark/benchmark.json: STACK / CHUNKS 'auto' fall back to off (run BENCHMARK first)")
    pick = lambda v, kind, k: bool(choice[kind][k]) if (v == "auto" and choice) else (False if v == "auto" else bool(v))
    stack_p, stack_m = pick(STACK, "paper", "stack"), pick(STACK, "maass", "stack")
    chunks_m = pick(CHUNKS, "maass", "chunks")
    if CHUNKS is True:
        log("CHUNKS = True: the paper's rows still run unchunked (the plan), Maass's rows chunked")
    log(f"switches: paper's rows stack {stack_p}, unchunked; Maass's rows stack {stack_m}, chunks {chunks_m}")
    jdump({"paper": {"stack": stack_p, "chunks": False}, "maass": {"stack": stack_m, "chunks": chunks_m},
           "from_benchmark": choice is not None}, wpath("switches.json"))
    log("== DS2: B1, B2, P1-P5, P3f")
    run_B("final")
    run_P("final", stack_p)
    run_drop()                                   # keep vs drop for the rows done so far (CPU, alongside the GPU)
    if EXACT_CV:
        log("== exact CV: B1, B2, P1-P5, P3f on the 12 DS1 splits")
        for sp in SPLITS_CV:
            if not budget_left():
                break
            run_B(sp)
            run_P(sp, stack_p)
    log("== Maass: calibration, M0, M1, M3, M3f")
    run_M(stack_m, CHUNK_PIECES if chunks_m else 1, CHUNK_WARMUP_S if chunks_m else 0.0)
    run_drop()                                   # keep vs drop for the Maass rows
    log("== waiting for the readout fits")
    Q.wait(0)
    check_t1_rows(); save_checks()

write_summary()
shutil.rmtree(CV_TMP, ignore_errors=True) if not os.listdir(CV_TMP) else None
log(f"done: {wpath('summary.md')}")


## 18. End of session

In [ ]:
print("results:", WORK_DIR)
print("summary:", os.path.join(WORK_DIR, "summary.md"))
print(f"session time: {(time.time() - SESSION_START) / 3600:.2f} h")
